<a href="https://colab.research.google.com/github/Andrellz27/Foundtions-models-to-histology-classification/blob/main/Fronzens_multiplos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""Fronzens_multiplos.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1dxtHA9MNch9RI-i-5GXGbJ5Rh96mhkwx

# super_codigo_11
Frozen (sonda + classificadores) e PEFT (LoRA/BitFit) de 16 foundation models em histopatologia.
"""

# -*- coding: utf-8 -*-
from __future__ import annotations
import zipfile
from pathlib import Path
from google.colab import drive

"""## CÉLULA 1: Configuração global, hiperparâmetros e seleção de testes"""

# %% [CÉLULA 1] Configuração global, hiperparâmetros e seleção de testes
import argparse
import dataclasses
import hashlib
import json
import math
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np

CLASS_NAMES = ["healthy", "mild", "moderate", "severe"]
NUM_CLASSES = len(CLASS_NAMES)
SELECTION_METRIC = "f1_macro"
SEL_NOME = {"f1_macro": "F1", "qwk": "QWK"}.get(SELECTION_METRIC, SELECTION_METRIC)
SEEDS = [0, 1]

FOUNDATION_BACKBONES = ("titan", "gigapath", "prism", "chief", "ctranspath", "lunit", "plip",
                        "virchow", "uni", "conch", "fusecpath", "beph", "pathoduet",
                        "path_foundation", "hibou_b", "athena")
BASELINES = ("imagenet_r50", "color")
BACKBONES = FOUNDATION_BACKBONES + BASELINES
FROZEN_ONLY = set(BASELINES) | {"prism", "path_foundation", "fusecpath"}
FUSECPATH_COMPONENTS = ("uni", "conch", "virchow", "gigapath")
LOSSES = ("coral", "corn", "ce")
REGIMES = ("frozen", "frozen_aug", "frozen_clf", "frozen_clf_aug", "lora", "lora_noaug",
           "bitfit", "bitfit_noaug", "partial", "partial_noaug")
TRAINED_REGIMES = ("partial", "lora", "bitfit")
CLASSIFIERS = ("knn", "simpleshot", "lda", "svm", "mlp")

PEFT_PROTOCOL = {"r": 16, "alpha": 16, "dropout": 0.1, "lr": 1e-4, "batch": 16, "epochs": 20,
                 "patience": 6, "fonte": "Ammeling et al., MELBA 2026"}
LORA_TARGETS = ("qkv", "query", "key", "value", "q_proj", "k_proj", "v_proj", "proj", "out_proj",
                "dense", "fc1", "fc2", "w12", "w3")
LR_VARIANTS = ("low", "mid", "high")

HF_IDS = {
    "titan": "hf:MahmoodLab/TITAN#return_conch",
    "gigapath": "hf_hub:prov-gigapath/prov-gigapath",
    "prism": "hf:paige-ai/Prism+hf-hub:paige-ai/Virchow",
    "chief": "local:CHIEF_CTransPath.pth",
    "ctranspath": "hf-hub:1aurent/swin_tiny_patch4_window7_224.CTransPath",
    "lunit": "hf-hub:1aurent/vit_small_patch8_224.lunit_dino",
    "plip": "hf:vinid/plip",
    "virchow": "hf-hub:paige-ai/Virchow",
    "uni": "hf-hub:MahmoodLab/uni",
    "conch": "hf_hub:MahmoodLab/conch",
    "fusecpath": "fusao:" + "+".join(FUSECPATH_COMPONENTS),
    "beph": "local:BEPH_backbone.pth",
    "pathoduet": "local:checkpoint_HE.pth",
    "path_foundation": "hf:google/path-foundation",
    "hibou_b": "hf:histai/hibou-b",
    "athena": "hf:PAICON-GmbH/Athena-0",
    "imagenet_r50": "resnet50.tv_in1k",
}

HF_REPOS = {
    "titan": "MahmoodLab/TITAN",
    "gigapath": "prov-gigapath/prov-gigapath",
    "prism": "paige-ai/Prism",
    "ctranspath": "1aurent/swin_tiny_patch4_window7_224.CTransPath",
    "lunit": "1aurent/vit_small_patch8_224.lunit_dino",
    "plip": "vinid/plip",
    "virchow": "paige-ai/Virchow",
    "uni": "MahmoodLab/uni",
    "conch": "MahmoodLab/conch",
    "path_foundation": "google/path-foundation",
    "hibou_b": "histai/hibou-b",
    "athena": "PAICON-GmbH/Athena-0",
    "imagenet_r50": "timm/resnet50.tv_in1k",
}

PESOS_DIR = "/content/drive/MyDrive/pesos_foundations"
LOCAL_WEIGHTS = {
    "chief": "CHIEF_CTransPath.pth",
    "beph": "BEPH_backbone.pth",
    "pathoduet": "checkpoint_HE.pth",
    "ctranspath": "ctranspath.pth",
}

MODEL_TRAIN_MPP = {"titan": 0.5, "gigapath": 0.5, "prism": 0.5, "chief": 1.0, "ctranspath": None,
                   "lunit": 0.5, "plip": None, "virchow": 0.5, "uni": 0.5, "conch": 0.5,
                   "fusecpath": None, "beph": 0.25, "pathoduet": None, "path_foundation": None,
                   "hibou_b": None, "athena": None, "imagenet_r50": None, "color": None}

DEFAULT_HPARAMS = {"lr_head": {"low": 1e-5, "mid": 5e-5, "high": 1e-4},
                   "lr_backbone_partial": {"low": 1e-6, "mid": 5e-6, "high": 1e-5}}
_GIANT_HPARAMS = {"lr_head": {"low": 1e-4, "mid": 5e-4, "high": 1e-3},
                  "lr_backbone_partial": {"low": 1e-6, "mid": 5e-6, "high": 1e-5}}

BACKBONE_HPARAMS: Dict[str, Dict] = {
    "gigapath": _GIANT_HPARAMS,
    "athena": _GIANT_HPARAMS,
    "virchow": {"lr_head": {"low": 5e-6, "mid": 1e-5, "high": 5e-5},
                "lr_backbone_partial": {"low": 1e-7, "mid": 5e-7, "high": 1e-6}},
    "uni": DEFAULT_HPARAMS,
    "conch": DEFAULT_HPARAMS,
    "titan": DEFAULT_HPARAMS,
}

BATCH_SIZE_T4: Dict[str, Dict[str, Optional[int]]] = {
    "imagenet_r50":    {"extract": 128, "partial": None},
    "conch":           {"extract": 32,  "partial": 8},
    "titan":           {"extract": 16,  "partial": 4},
    "uni":             {"extract": 64,  "partial": 32},
    "virchow":         {"extract": 16,  "partial": 8},
    "prism":           {"extract": 16,  "partial": None},
    "gigapath":        {"extract": 8,   "partial": 4},
    "athena":          {"extract": 8,   "partial": 4},
    "chief":           {"extract": 128, "partial": 64},
    "ctranspath":      {"extract": 128, "partial": 64},
    "lunit":           {"extract": 64,  "partial": 16},
    "plip":            {"extract": 128, "partial": 64},
    "beph":            {"extract": 64,  "partial": 32},
    "pathoduet":       {"extract": 64,  "partial": 32},
    "hibou_b":         {"extract": 64,  "partial": 32},
    "path_foundation": {"extract": 64,  "partial": None},
    "fusecpath":       {"extract": None, "partial": None},
    "color":           {"extract": None, "partial": None},
}

@dataclass
class Config:
    drive_dataset_dir: str = "/content/drive/MyDrive/Original_ROI_images (1).zip"
    local_dataset_dir: str = "/content/dataset"
    drive_results_dir: str = "/content/drive/MyDrive/resultados_histopatologia"
    embeddings_dir: str = "/content/drive/MyDrive/resultados_histopatologia/embeddings_cache"
    run_name: Optional[str] = None
    resume: bool = True
    seeds: List[int] = field(default_factory=lambda: list(SEEDS))
    partial_seeds: Optional[List[int]] = None
    backbone: List[str] = field(default_factory=lambda: ["all"])
    regime: List[str] = field(default_factory=lambda: ["all"])
    loss: List[str] = field(default_factory=lambda: ["all"])
    lr_variant: List[str] = field(default_factory=lambda: ["mid"])
    train_frac: float = 0.70
    val_frac: float = 0.20
    test_frac: float = 0.10
    split_by_group: bool = True
    subset_groups_per_class: Optional[int] = None
    scales: List[float] = field(default_factory=lambda: [1.0])
    native_mpp: Optional[float] = None
    c_grid: List[float] = field(default_factory=lambda: np.logspace(-5, 1, 13).tolist())
    c_patience: int = 3
    probe_max_iter: int = 3000
    frozen_aug_copies: int = 4
    classifiers: List[str] = field(default_factory=lambda: list(CLASSIFIERS))
    peft_lp_init: bool = True
    n_unfrozen_blocks: int = 2
    weight_decay: float = 1e-2
    epochs: int = 20
    patience: int = 6
    warmup_ratio: float = 0.05
    grad_clip: float = 1.0
    label_smoothing: float = 0.0
    use_amp: bool = True
    num_workers: int = 2
    effective_batch: int = 32
    partial_batch_configs: List[Optional[Tuple[int, int]]] = field(default_factory=lambda: [None])
    vram_scaling: bool = True
    max_oom_retries: int = 4
    n_boot: int = 1000
    ci_level: float = 0.95
    show_plots: bool = True
    plot_per_scenario: bool = True
    consolidar_sem_pendentes: bool = False
    install_deps: bool = True

def _expand(values: List[str], universe) -> List[str]:
    return list(universe) if "all" in values else list(values)

def _in_notebook() -> bool:
    import sys
    return "ipykernel" in sys.modules

def build_config(argv=None) -> Config:
    p = argparse.ArgumentParser(allow_abbrev=False)
    p.add_argument("--backbone", nargs="+", default=["all"], choices=list(BACKBONES) + ["all"])
    p.add_argument("--regime", nargs="+", default=["all"], choices=list(REGIMES) + ["all"])
    p.add_argument("--loss", nargs="+", default=["all"], choices=list(LOSSES) + ["all"])
    p.add_argument("--lr_variant", nargs="+", default=["mid"], choices=list(LR_VARIANTS) + ["all"])
    p.add_argument("--seeds", nargs="+", type=int, default=list(SEEDS))
    p.add_argument("--scales", nargs="+", type=float, default=[1.0])
    p.add_argument("--native_mpp", type=float, default=None)
    p.add_argument("--epochs", type=int, default=Config.epochs)
    p.add_argument("--effective_batch", type=int, default=Config.effective_batch)
    p.add_argument("--run_name", default=None)
    args, _ = p.parse_known_args(argv if argv is not None else [] if _in_notebook() else None)
    return Config(
        backbone=_expand(args.backbone, BACKBONES), regime=_expand(args.regime, REGIMES),
        loss=_expand(args.loss, LOSSES), lr_variant=_expand(args.lr_variant, LR_VARIANTS),
        seeds=args.seeds, scales=args.scales, native_mpp=args.native_mpp, epochs=args.epochs,
        effective_batch=args.effective_batch, run_name=args.run_name,
    )

CFG = build_config()
if _in_notebook():
    CFG = Config(backbone=list(BACKBONES), regime=["all"], loss=["all"], seeds=list(SEEDS))

drive.mount("/content/drive")
pasta_extraida = Path("/content/Original_ROI_images")

def localizar_zip_dataset(mydrive: Path, pista: str = "Original_ROI_images") -> Path:
    candidatos = sorted(mydrive.rglob(f"*{pista}*.zip"))
    if len(candidatos) == 1:
        print(f"[dataset] zip localizado: {candidatos[0]}")
        return candidatos[0]
    if not candidatos:
        todos_zips = sorted(p.name for p in mydrive.rglob("*.zip"))
        raise FileNotFoundError(f"Nenhum .zip com '{pista}' em {mydrive}. Zips: {todos_zips[:30]}")
    raise FileNotFoundError(f"Mais de um candidato: {[str(p) for p in candidatos]}. Escolha um.")

if not pasta_extraida.exists():
    with zipfile.ZipFile(localizar_zip_dataset(Path("/content/drive/MyDrive")), "r") as z:
        z.extractall(pasta_extraida)

def encontrar_raiz_classes(base: Path, classes) -> Path:
    for cand in [base] + [p for p in base.rglob("*") if p.is_dir()]:
        if all((cand / c).is_dir() for c in classes):
            return cand
    raise FileNotFoundError(f"Não encontrei as pastas {classes} dentro de {base}.")

CFG.drive_dataset_dir = str(encontrar_raiz_classes(pasta_extraida, CLASS_NAMES))
print("Raiz do dataset:", CFG.drive_dataset_dir)


Mounted at /content/drive
[dataset] zip localizado: /content/drive/MyDrive/Original_ROI_images (1).zip
Raiz do dataset: /content/Original_ROI_images/Original ROI images


In [2]:

"""## CÉLULA 2: Dependências, login no HF e registro do ambiente"""

# %% [CÉLULA 2] Dependências (com lock de versões), login no HF e registro do ambiente
import importlib.metadata as imd
import importlib.util
import os
import re
import subprocess
import sys

def pip_install(*pkgs: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

LOCK_PACKAGES = {"timm", "huggingface-hub", "scikit-learn", "scikit-image", "scipy", "pandas",
                 "tabulate", "matplotlib", "conch", "transformers", "einops", "einops-exts",
                 "environs", "sacremoses", "safetensors"}
LOCK_PATH = Path(CFG.drive_results_dir) / "requirements_lock.txt"

def _norm_pkg(name: str) -> str:
    return name.strip().lower().replace("_", "-")

def write_lock(path: Path) -> None:
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True,
                            text=True, check=True).stdout.splitlines()
    keep = [ln for ln in freeze if _norm_pkg(re.split(r"[=@ ]", ln, maxsplit=1)[0]) in LOCK_PACKAGES]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(keep) + "\n")
    print(f"[deps] lock gravado em {path}:\n  " + "\n  ".join(keep))

TRANSFORMERS_VERSAO = "4.46.0"
EXTRA_DEPS = {
    "titan": ["einops==0.6.1", "einops-exts==0.0.4"],
    "prism": ["environs", "sacremoses"],
    "athena": ["safetensors"],
}
NEEDS_TRANSFORMERS = {"titan", "prism", "plip", "hibou_b", "conch", "fusecpath"}
HUB_SPEC = "huggingface-hub>=0.23.2,<1.0"
LOCK_IGNORADOS = {"transformers", "huggingface-hub", "tokenizers"}
_MODULE_OF = {"einops-exts": "einops_exts"}

def _missing(pkgs):
    out = []
    for p in pkgs:
        base = re.split(r"[=<>]", p)[0]
        if importlib.util.find_spec(_MODULE_OF.get(base, base).replace("-", "_")) is None:
            out.append(p)
    return out

def _ver(dist: str) -> Optional[str]:
    try:
        return imd.version(dist)
    except imd.PackageNotFoundError:
        return None

def _lock_filtrado(path: Path) -> Path:
    linhas = [ln for ln in path.read_text().splitlines()
              if ln.strip() and _norm_pkg(re.split(r"[=@ <>~!]", ln.strip(), maxsplit=1)[0]) not in LOCK_IGNORADOS]
    tmp = Path("/content/_requirements_lock_filtrado.txt")
    tmp.write_text("\n".join(linhas) + "\n")
    return tmp

def _checar_modulos_desatualizados() -> None:
    obsoletos = []
    for mod, dist in (("huggingface_hub", "huggingface-hub"), ("transformers", "transformers"),
                      ("tokenizers", "tokenizers"), ("timm", "timm"), ("safetensors", "safetensors")):
        m = sys.modules.get(mod)
        em_memoria, no_disco = getattr(m, "__version__", None), _ver(dist)
        if m is not None and em_memoria and no_disco and em_memoria != no_disco:
            obsoletos.append(f"{dist}: {em_memoria} em memória x {no_disco} instalada")
    if obsoletos:
        raise RuntimeError(
            "O ambiente foi atualizado com bibliotecas já carregadas (" + "; ".join(obsoletos) + "). "
            "Reinicie la sesión y vuelva a ejecutar.")

if CFG.install_deps:
    if LOCK_PATH.exists():
        print(f"[deps] instalando versões congeladas de {LOCK_PATH}")
        pip_install("-r", str(_lock_filtrado(LOCK_PATH)))
    else:
        pip_install("timm>=1.0.3", "huggingface_hub", "scikit-learn", "scikit-image", "scipy",
                    "pandas", "tabulate", "matplotlib")
    extras = sorted({p for bb in CFG.backbone for p in EXTRA_DEPS.get(bb, [])})
    if _missing(extras):
        pip_install(*_missing(extras))
    if NEEDS_TRANSFORMERS & set(CFG.backbone):
        _hub = _ver("huggingface-hub")
        _hub_ok = _hub is not None and int(_hub.split(".")[0]) < 1
        if _ver("transformers") != TRANSFORMERS_VERSAO or not _hub_ok:
            pip_install(f"transformers=={TRANSFORMERS_VERSAO}", HUB_SPEC)
    if ("conch" in CFG.backbone or "fusecpath" in CFG.backbone) and importlib.util.find_spec("conch") is None:
        pip_install("git+https://github.com/Mahmoodlab/CONCH.git")
    if not LOCK_PATH.exists():
        write_lock(LOCK_PATH)
    _checar_modulos_desatualizados()

DINOV2_COMMIT = None
if "athena" in CFG.backbone:
    dino_dir = Path("/content/dinov2")
    if not (dino_dir / "dinov2").is_dir():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/dinov2.git", str(dino_dir)], check=True)
    if str(dino_dir) not in sys.path:
        sys.path.insert(0, str(dino_dir))
    DINOV2_COMMIT = subprocess.run(["git", "-C", str(dino_dir), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

from huggingface_hub import HfApi, get_token, login, notebook_login
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    login(token=_tok)
except Exception:
    notebook_login()
HF_TOKEN = get_token()

def _file_fingerprint(path: Path) -> Optional[str]:
    if not path.exists(): return None
    with open(path, "rb") as f:
        head = f.read(1 << 20)
    return f"{path.stat().st_size}:{hashlib.sha1(head).hexdigest()[:16]}"

def collect_env() -> Dict:
    import torch
    pkgs = {}
    for name in sorted(LOCK_PACKAGES | {"torch", "torchvision", "numpy"}):
        try: pkgs[name] = imd.version(name)
        except imd.PackageNotFoundError: pass
    hf_sha = {}
    api = HfApi()
    for bb, repo in HF_REPOS.items():
        try: hf_sha[bb] = api.model_info(repo, token=HF_TOKEN).sha
        except Exception as e: hf_sha[bb] = f"indisponível ({type(e).__name__})"
    return {
        "python": sys.version.split()[0], "packages": pkgs,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "vram_gb": (round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else None),
        "cuda": torch.version.cuda, "hf_sha": hf_sha, "dinov2_commit": DINOV2_COMMIT,
        "local_weights": {bb: _file_fingerprint(Path(PESOS_DIR) / fn) for bb, fn in LOCAL_WEIGHTS.items()},
    }

ENV_INFO = collect_env()
print(json.dumps(ENV_INFO, indent=2))


[deps] instalando versões congeladas de /content/drive/MyDrive/resultados_histopatologia/requirements_lock.txt


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{
  "python": "3.13.15",
  "packages": {
    "conch": "0.1.0",
    "einops": "0.8.2",
    "einops-exts": "0.0.4",
    "environs": "15.2.0",
    "huggingface-hub": "0.36.2",
    "matplotlib": "3.10.0",
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "sacremoses": "0.2.0",
    "safetensors": "0.8.0",
    "scikit-image": "0.25.2",
    "scikit-learn": "1.6.1",
    "scipy": "1.16.3",
    "tabulate": "0.9.0",
    "timm": "1.0.29",
    "torch": "2.11.0+cu130",
    "torchvision": "0.26.0+cu130",
    "transformers": "4.46.0"
  },
  "gpu": "Tesla T4",
  "vram_gb": 14.56,
  "cuda": "13.0",
  "hf_sha": {
    "titan": "dac6773d9961cfc75503440676ff157a2c6e8d2e",
    "gigapath": "64f9e26c15019f2d4f6d9113c6822f88bb16b01b",
    "prism": "b5ef311e78a0811c71ecf618f6f7140e9ebb5fc6",
    "ctranspath": "59c740e4ed91dda4ebe094b64e4cd2e015c31810",
    "lunit": "c23bc01cfbe022bc1f2a8a9306125a07818e4eb4",
    "plip": "67ade53ddd32195868f422585f72698ef5d15094",
    "virchow": "19eebc84ae33e79f1b2d866e6ff90ae50e

In [3]:

"""## CÉLULA 3: Dataset, grupos, split holdout 70/20/10, multi-crop e augmentações"""

# %% [CÉLULA 3] Dataset, grupos, split holdout 70/20/10, multi-crop e augmentações
import gc
import random
import shutil
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from skimage.color import hed2rgb, rgb2hed
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def _list_images(class_dir: Path) -> List[Path]:
    return sorted(p for p in class_dir.rglob("*") if p.suffix.lower() in IMG_EXTS)

def copy_dataset_to_local(src: str, dst: str) -> Path:
    src_p, dst_p = Path(src), Path(dst)
    n_src = sum(len(_list_images(src_p / c)) for c in CLASS_NAMES)
    n_dst = sum(len(_list_images(dst_p / c)) for c in CLASS_NAMES if (dst_p / c).is_dir())
    if n_src == n_dst:
        print(f"[dataset] cópia local já completa ({n_dst} imagens)")
        return dst_p
    for c in CLASS_NAMES:
        shutil.copytree(src_p / c, dst_p / c, dirs_exist_ok=True)
    return dst_p

def scan_dataset(root: Path):
    paths, labels = [], []
    for idx, c in enumerate(CLASS_NAMES):
        files = _list_images(root / c)
        paths += [str(p) for p in files]
        labels += [idx] * len(files)
    return paths, np.array(labels, dtype=np.int64)

ROI_RE = re.compile(r"(.*)-roi\d+$")

def group_id(path: str) -> str:
    p = Path(path)
    m = ROI_RE.match(p.stem)
    return f"{p.parent.name}/{m.group(1) if m else p.stem}"

def subset_by_groups(labels: np.ndarray, groups: np.ndarray, k: int, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    keep = []
    for c in range(NUM_CLASSES):
        g = np.unique(groups[labels == c])
        rng.shuffle(g)
        keep += list(g[:k])
    return np.flatnonzero(np.isin(groups, keep))

def make_holdout_split(labels: np.ndarray, groups: np.ndarray, seed: int, cfg: Config) -> Dict[str, np.ndarray]:
    fr = (cfg.train_frac, cfg.val_frac, cfg.test_frac)
    if cfg.split_by_group:
        units, first = np.unique(groups, return_index=True)
        unit_y = labels[first]
    else:
        units, unit_y = np.arange(len(labels)), labels
    u = np.arange(len(units))
    u_tv, u_te = train_test_split(u, test_size=cfg.test_frac, stratify=unit_y[u], random_state=seed)
    u_tr, u_va = train_test_split(u_tv, test_size=cfg.val_frac / (cfg.train_frac + cfg.val_frac), stratify=unit_y[u_tv], random_state=seed)

    def to_idx(uu):
        return np.flatnonzero(np.isin(groups, units[uu])) if cfg.split_by_group else np.sort(units[uu])

    sp = {"train": to_idx(u_tr), "val": to_idx(u_va), "test": to_idx(u_te)}
    return sp

def split_summary(splits: Dict[int, Dict[str, np.ndarray]], labels, groups) -> "pd.DataFrame":
    rows = []
    for seed, sp in splits.items():
        r = {"semente": seed}
        for part in ("train", "val", "test"):
            idx = sp[part]
            r[f"{part}_rois"] = len(idx)
            r[f"{part}_%"] = round(100 * len(idx) / len(labels), 1)
            r[f"{part}_imagens"] = len(set(groups[idx]))
        rows.append(r)
    import pandas as pd
    return pd.DataFrame(rows)


In [4]:

"""## CÉLULA 4: Backbones, cabeças (regime partial) e relatório de magnificação"""

# %% [CÉLULA 4] Backbones, cabeças (regime partial) e relatório de magnificação
# O código continuará as definições da Célula 4...


'## CÉLULA 4: Backbones, cabeças (regime partial) e relatório de magnificação'

In [5]:

"""## CÉLULA 5: Perdas, sonda linear L2 e métricas"""

# %% [CÉLULA 5] Perdas, sonda linear L2 (L-BFGS) e métricas
# Funções auxiliares para perda e avaliação de métricas...

"""## CÉLULA 6: Cache de embeddings e busca de C"""

# %% [CÉLULA 6] Cache de embeddings (com hash) e sonda linear com busca de C
# Extracao de embeddings de forma eficiente...

"""## CÉLULA 7: Regime partial"""

# %% [CÉLULA 7] Regime partial (backbone no grafo, AMP + acumulação + OOM fallback)
# Treinamento parcial / ajuste de adaptadores...

"""## CÉLULA 8: Bootstrap, tabelas e gráficos"""

# %% [CÉLULA 8] Bootstrap por imagem, tabelas (display + CSV) e gráficos (show + savefig)
# Métricas finais de teste e criação de figuras de análise...

"""## CÉLULA 9: Orquestrador e fusão de embeddings"""

# %% [CÉLULA 9] Orquestrador (com retomada) e consolidação dos resultados
# Orquestração de cenários e salvamento de metadados...

"""## CÉLULA 10: Rotina de testes e execução da bateria"""


'## CÉLULA 10: Rotina de testes e execução da bateria'

In [6]:

# %% [CÉLULA 10] Rotina de testes: sanidade, frozen (sonda + classificadores) e PEFT (LoRA/BitFit)
# Bloco de execução final com os testes rápidos e bateria principal...

# -*- coding: utf-8 -*-
"""super_codigo_10.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1lJeKj2RJXazdbdbNyPVunsUd8OqHuL8Y

# super_codigo_10 — Frozen (sonda + classificadores) e PEFT (LoRA/BitFit) de 16 foundation models em histopatologia (severidade ordinal)

**Ordem de execução:** rode as células 1 → 10 em sequência. A célula 10 faz, nesta ordem: sanidade, frozen de todos os modelos, pós-processamento (TTA + limiar para F1), LoRA/BitFit nos melhores, ensemble entre modelos e comparação final com teste pareado.

- **Critério principal: F1 macro** (`SELECTION_METRIC`, célula 1), sempre medido na VALIDAÇÃO; QWK só desempata.
- **Nada é refeito:** execuções já salvas em qualquer pasta de resultados (inclusive `bateria_*` do código antigo) são importadas para `frozen_<modelo>`; smoke tests de modelos já validados são pulados; fases sem nada pendente não reconsolidam.
- **Execuções antigas** foram escolhidas por QWK e ficam marcadas em `criterio_selecao`.
- **Resultados:** `frozen_<modelo>`, `ensemble/`, `comparacao_*.csv`, `comparacao_pareada_f1.csv`.
- **Se o Colab desconectar:** rode tudo de novo; a retomada pula o que já foi salvo.
- **PRISM:** sessão separada (versão do `transformers`, ver célula 2).
- **Sementes:** 2 (`SEEDS = [0, 1]` na célula 1).
- **Pré-requisitos:** GPU T4 ativa; token `HF_TOKEN` nos Secrets do Colab; acesso aprovado aos
  repositórios fechados no Hugging Face (MahmoodLab/uni, MahmoodLab/conch, MahmoodLab/TITAN,
  paige-ai/Virchow, paige-ai/Prism, prov-gigapath/prov-gigapath, histai/hibou-b,
  PAICON-GmbH/Athena-0, google/path-foundation); pesos de CHIEF, BEPH e PathoDuet em
  `PESOS_DIR` no Drive (ver `LOCAL_WEIGHTS`).
"""
from __future__ import annotations


# -*- coding: utf-8 -*-
"""super_codigo_4

Versão revisada de super_codigo_3, alinhada ao novo prompt. Mudanças principais:

  0. Split: holdout 70/20/10 com train_test_split (pedido no prompt), repetido em várias
     sementes. A divisão é feita sobre IMAGENS DE ORIGEM (grupo = pasta da classe + prefixo
     antes de '-roiN') e estratificada pela classe: ROIs da mesma imagem nunca ficam em
     partições diferentes (split_by_group=False desliga isso, mas gera vazamento).
  1. Batch size dinâmico: tabela por backbone (calibrada para T4 16 GB) separada em
     'extract' (embeddings, sem gradiente) e 'partial' (treino), escalonada pela VRAM da GPU,
     com fallback automático de OOM (divide o lote por 2 e dobra a acumulação, preservando o
     batch efetivo). O lote realmente usado é registrado em todas as tabelas.
  2. Tabelas com metadados (Modelo, Regime, Loss, LR, Época, Batch size, Accum, Batch efetivo,
     tempos e pico de VRAM), exibidas com display() e exportadas em .csv no Drive.
  3. Gráficos com plt.show() + plt.savefig(): curvas de aprendizado (loss e métricas por
     época), curvas de regularização C (frozen), matrizes de confusão do teste, resumo com IC
     e análise do impacto do batch size.
  4. Retomada (resume): cada (cenário, semente) é salvo assim que termina; rodar de novo pula
     o que já existe. Essencial no Colab gratuito, que desconecta.
  5. Rotina de testes: checagens de sanidade (gradiente da sonda, losses torch x numpy,
     métricas rápidas x sklearn, split sem vazamento), smoke test e uma bateria de fases para
     uma foundation (UNI por padrão), com escolha da melhor loss feita pela VALIDAÇÃO.
"""

# %% [CÉLULA 1] Configuração global, hiperparâmetros e seleção de testes
import argparse
import dataclasses
import hashlib
import json
import math
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np

CLASS_NAMES = ["healthy", "mild", "moderate", "severe"]  # ordem = ordem ordinal
NUM_CLASSES = len(CLASS_NAMES)
# CRITÉRIO PRINCIPAL de seleção pela VALIDAÇÃO (C da sonda, hiperparâmetro dos classificadores,
# época do LoRA/BitFit, loss, configuração e modelo): F1 macro; empate -> QWK.
# Execuções antigas (escolhidas por QWK) são reaproveitadas e marcadas em 'criterio_selecao'.
SELECTION_METRIC = "f1_macro"
SEL_NOME = {"f1_macro": "F1", "qwk": "QWK"}.get(SELECTION_METRIC, SELECTION_METRIC)
SEEDS = [0, 1]  # DUAS sementes: cada uma = um split 70/20/10 diferente (holdout repetido)

# ---------------------------------------------------------------------------------------
# FOUNDATION MODELS AVALIADOS (16). Observações importantes:
#   * TITAN, PRISM e CHIEF são modelos de LÂMINA (WSI). Em ROIs isoladas usamos o caminho
#     de nível de tile que cada um oferece:
#       - titan : encoder de tiles do TITAN (CONCH v1.5, ViT-L/16 @ 448 px) via titan.return_conch()
#       - prism : Virchow (tiles) + agregador Perceiver do PRISM sobre as janelas de cada ROI
#       - chief : encoder de tiles do CHIEF (CTransPath re-treinado, Swin-T + ConvStem)
#   * fusecpath: o FuseCPath original é um framework de fusão para WSI/MIL, sem pesos de tile.
#     Aqui ele é APROXIMADO por fusão de características: concatenação dos embeddings de
#     FUSECPATH_COMPONENTS (só regime frozen). Descreva assim no texto do estudo.
#   * chief, beph e pathoduet não estão no Hugging Face: baixe os pesos (links em LOCAL_WEIGHTS)
#     e coloque-os em PESOS_DIR no Drive.
# ---------------------------------------------------------------------------------------
FOUNDATION_BACKBONES = ("titan", "gigapath", "prism", "chief", "ctranspath", "lunit", "plip",
                        "virchow", "uni", "conch", "fusecpath", "beph", "pathoduet",
                        "path_foundation", "hibou_b", "athena")
BASELINES = ("imagenet_r50", "color")
BACKBONES = FOUNDATION_BACKBONES + BASELINES
# Sem regime partial: baselines, modelos não-PyTorch (path_foundation é TensorFlow), o
# agregador do PRISM e a fusão de embeddings (fusecpath).
FROZEN_ONLY = set(BASELINES) | {"prism", "path_foundation", "fusecpath"}
FUSECPATH_COMPONENTS = ("uni", "conch", "virchow", "gigapath")  # ordem da concatenação
LOSSES = ("coral", "corn", "ce")
# frozen         : sonda linear L2 (regressão logística, L-BFGS), C escolhido na validação.
#                  Protocolo publicado de UNI/CONCH/TITAN/PLIP: sem LR e sem lote.
# frozen_aug     : idem, com cópias aumentadas dos embeddings de treino
# frozen_clf     : OUTROS classificadores sobre os mesmos embeddings (CLASSIFIERS)
# frozen_clf_aug : idem, com as cópias aumentadas
# lora / bitfit  : adaptação eficiente (PEFT) do encoder + cabeça linear (PEFT_PROTOCOL)
# *_noaug        : idem, sem augmentation de imagem
# partial        : últimos N blocos destravados (versão anterior; fora da bateria atual)
REGIMES = ("frozen", "frozen_aug", "frozen_clf", "frozen_clf_aug", "lora", "lora_noaug",
           "bitfit", "bitfit_noaug", "partial", "partial_noaug")
TRAINED_REGIMES = ("partial", "lora", "bitfit")   # prefixos dos regimes que treinam o encoder

# Classificadores sobre embeddings congelados (hiperparâmetros escolhidos na VALIDAÇÃO pelo QWK,
# grades em ordem de mais regularizado -> menos; empate fica com o mais regularizado):
#   knn       : k vizinhos (k em 5/10/20/40), embeddings centrados + normalizados L2, peso por
#               distância. kNN e SimpleShot são avaliações usadas nos artigos do UNI/DINOv2.
#   simpleshot: centróide mais próximo com centralização + L2 (SimpleShot, "CL2N"). Sem hiperparâmetro.
#   lda       : análise discriminante linear com encolhimento automático (Ledoit-Wolf).
#   svm       : SVM RBF, C em 0,01..100, gamma='scale'.
#   mlp       : 1 camada oculta (256), regularização L2 (alpha) em 1e-1..1e-4; otimizador padrão
#               do scikit-learn (Adam, LR 1e-3), sem busca de LR.
CLASSIFIERS = ("knn", "simpleshot", "lda", "svm", "mlp")

# ---------------------------------------------------------------------------------------
# PROTOCOLO DE PEFT (LoRA e BitFit). O LoRA precisa de LR; não há valor por modelo publicado,
# então usa-se UM protocolo publicado em patologia para todos (sem busca de LR):
#   Ammeling et al., "Benchmarking Foundation Models for Mitotic Figure Classification"
#   (MELBA, 2026; UNI, Virchow, GigaPath, H-optimus, Phikon): LoRA em qkv + projeção de saída
#   da atenção + fc1/fc2 do MLP; rank 16, escala 16, dropout 0,1; Adam (betas padrão), lote 16,
#   aquecimento linear + cosseno, LR máximo 1e-4.
# Adaptações aqui: 20 épocas sobre ~316 ROIs (o artigo usa 100 pseudo-épocas de 1280 patches,
# inviável no T4), época escolhida pelo QWK de validação (o artigo usa a loss de validação) e,
# por padrão, cabeça inicializada com a sonda L-BFGS (LP-FT, Kumar et al., ICLR 2022), o que
# parte exatamente do resultado frozen.
# BitFit (só os vieses do encoder + cabeça): sem protocolo em patologia; usa o mesmo otimizador.
# ---------------------------------------------------------------------------------------
PEFT_PROTOCOL = {"r": 16, "alpha": 16, "dropout": 0.1, "lr": 1e-4, "batch": 16, "epochs": 20,
                 "patience": 6, "fonte": "Ammeling et al., MELBA 2026"}
LORA_TARGETS = ("qkv", "query", "key", "value", "q_proj", "k_proj", "v_proj", "proj", "out_proj",
                "dense", "fc1", "fc2", "w12", "w3")
LR_VARIANTS = ("low", "mid", "high")  # só o regime partial (fora da bateria atual)

# Identificador do peso (entra no hash do cache de embeddings: não altere os já existentes,
# senão o cache de UNI/Virchow/CONCH deixa de ser reaproveitado).
HF_IDS = {
    "titan": "hf:MahmoodLab/TITAN#return_conch",
    "gigapath": "hf_hub:prov-gigapath/prov-gigapath",
    "prism": "hf:paige-ai/Prism+hf-hub:paige-ai/Virchow",
    "chief": "local:CHIEF_CTransPath.pth",
    "ctranspath": "hf-hub:1aurent/swin_tiny_patch4_window7_224.CTransPath",
    "lunit": "hf-hub:1aurent/vit_small_patch8_224.lunit_dino",
    "plip": "hf:vinid/plip",
    "virchow": "hf-hub:paige-ai/Virchow",
    "uni": "hf-hub:MahmoodLab/uni",
    "conch": "hf_hub:MahmoodLab/conch",
    "fusecpath": "fusao:" + "+".join(FUSECPATH_COMPONENTS),
    "beph": "local:BEPH_backbone.pth",
    "pathoduet": "local:checkpoint_HE.pth",
    "path_foundation": "hf:google/path-foundation",
    "hibou_b": "hf:histai/hibou-b",
    "athena": "hf:PAICON-GmbH/Athena-0",
    "imagenet_r50": "resnet50.tv_in1k",  # ResNet-50 supervisionada em ImageNet (timm)
}
HF_REPOS = {  # repositório no HF de cada peso (para registrar o commit usado)
    "titan": "MahmoodLab/TITAN",
    "gigapath": "prov-gigapath/prov-gigapath",
    "prism": "paige-ai/Prism",
    "ctranspath": "1aurent/swin_tiny_patch4_window7_224.CTransPath",
    "lunit": "1aurent/vit_small_patch8_224.lunit_dino",
    "plip": "vinid/plip",
    "virchow": "paige-ai/Virchow",
    "uni": "MahmoodLab/uni",
    "conch": "MahmoodLab/conch",
    "path_foundation": "google/path-foundation",
    "hibou_b": "histai/hibou-b",
    "athena": "PAICON-GmbH/Athena-0",
    "imagenet_r50": "timm/resnet50.tv_in1k",
}
# Pesos fora do HF: baixe manualmente e salve com estes nomes em PESOS_DIR (Drive).
PESOS_DIR = "/content/drive/MyDrive/pesos_foundations"
LOCAL_WEIGHTS = {
    # github.com/hms-dbmi/CHIEF (link do Google Drive no README) -> CHIEF_CTransPath.pth
    "chief": "CHIEF_CTransPath.pth",
    # github.com/Zhcyoung/BEPH (Google Drive/Baidu). Pode ser o checkpoint completo ou só o
    # backbone extraído com o script do README; o carregador aceita os dois.
    "beph": "BEPH_backbone.pth",
    # github.com/openmedlab/PathoDuet (Google Drive) -> modelo H&E
    "pathoduet": "checkpoint_HE.pth",
    # opcional: pesos originais do CTransPath (github.com/Xiyue-Wang/TransPath). Se existir,
    # é usado no lugar da cópia do HF.
    "ctranspath": "ctranspath.pth",
}
# Resolução (µm/pixel) aproximada dos tiles de pré-treino, para o relatório de magnificação.
# None = não documentada de forma inequívoca ou multi-resolução.
MODEL_TRAIN_MPP = {"titan": 0.5, "gigapath": 0.5, "prism": 0.5, "chief": 1.0, "ctranspath": None,
                   "lunit": 0.5, "plip": None, "virchow": 0.5, "uni": 0.5, "conch": 0.5,
                   "fusecpath": None, "beph": 0.25, "pathoduet": None, "path_foundation": None,
                   "hibou_b": None, "athena": None, "imagenet_r50": None, "color": None}

# Learning rates do regime partial (o frozen não tem LR: é resolvido por L-BFGS).
# Modelos sem entrada própria usam DEFAULT_HPARAMS.
# --- Magnificação (v11) ---------------------------------------------------------------
# A magnificação da captura NÃO está nos TIFF (72 dpi padrão, sem objetiva). Se souber a
# objetiva, informe MAGNIFICACAO_CAPTURA na CÉLULA 10: a fase 4 passa a testar também a
# escala que reproduz o µm/pixel do pré-treino de cada modelo (MODEL_TRAIN_MPP).
OBJETIVA_PARA_MPP = {5: 2.0, 10: 1.0, 20: 0.5, 40: 0.25, 60: 0.17}
# Janela de entrada de cada modelo (a fase 4 converte o zoom físico em escala por janela).
# O valor real é lido do cache de embeddings; esta tabela só é usada se o cache não existir.
JANELA_PADRAO = {"titan": 448, "conch": 448}          # demais: 224
# Grade da fase 4, em ZOOM sobre o pixel da captura: 1,0 = pixel original; 0,5 = metade da
# magnificação (ex.: 40x -> 20x); 2,0 = o dobro. Igual para todos os modelos.
MAG_ZOOMS_PADRAO = [0.5, 0.75, 1.0, 1.5, 2.0]

DEFAULT_HPARAMS = {"lr_head": {"low": 1e-5, "mid": 5e-5, "high": 1e-4},
                   "lr_backbone_partial": {"low": 1e-6, "mid": 5e-6, "high": 1e-5}}
_GIANT_HPARAMS = {"lr_head": {"low": 1e-4, "mid": 5e-4, "high": 1e-3},
                  "lr_backbone_partial": {"low": 1e-6, "mid": 5e-6, "high": 1e-5}}
BACKBONE_HPARAMS: Dict[str, Dict] = {
    "gigapath": _GIANT_HPARAMS,
    "athena": _GIANT_HPARAMS,
    "virchow": {"lr_head": {"low": 5e-6, "mid": 1e-5, "high": 5e-5},
                "lr_backbone_partial": {"low": 1e-7, "mid": 5e-7, "high": 1e-6}},
    "uni": DEFAULT_HPARAMS,
    "conch": DEFAULT_HPARAMS,
    "titan": DEFAULT_HPARAMS,
}

# ---------------------------------------------------------------------------------------
# BATCH SIZE DINÂMICO. Unidade = ROIs por lote (cada ROI 450x250 vira 2 janelas no
# multi-crop, então o nº de imagens que passam pela rede é ~2x o valor abaixo).
# Calibrado para Tesla T4 (16 GB, fp16 — o T4 não tem bf16). Em GPUs maiores o valor é
# multiplicado por vram_factor() (potência de 2: 1x até ~28 GB, 2x no A100-40, 4x no A100-80).
#   extract: só inferência (sem gradiente) -> lotes grandes.
#   partial: guarda ativações dos blocos destravados -> lotes menores (None = sem partial).
# Se mesmo assim faltar memória, o fallback de OOM divide o lote por 2 automaticamente.
# ---------------------------------------------------------------------------------------
BATCH_SIZE_T4: Dict[str, Dict[str, Optional[int]]] = {
    "imagenet_r50":    {"extract": 128, "partial": None},  # ~25M params
    "conch":           {"extract": 32,  "partial": 8},     # ViT-B/16 @ 448 px
    "titan":           {"extract": 16,  "partial": 4},     # CONCH v1.5 = ViT-L/16 @ 448 px
    "uni":             {"extract": 64,  "partial": 32},    # ViT-L/16 @ 224 px (~300M)
    "virchow":         {"extract": 16,  "partial": 8},     # ViT-H/14 @ 224 px (~630M)
    "prism":           {"extract": 16,  "partial": None},  # Virchow + Perceiver
    "gigapath":        {"extract": 8,   "partial": 4},     # ViT-g/14 (~1,1B)
    "athena":          {"extract": 8,   "partial": 4},     # ViT-G/14 (~1,1B)
    "chief":           {"extract": 128, "partial": 64},    # Swin-T + ConvStem (~28M)
    "ctranspath":      {"extract": 128, "partial": 64},    # Swin-T + ConvStem (~28M)
    "lunit":           {"extract": 64,  "partial": 16},    # ViT-S/8 (784 tokens por janela)
    "plip":            {"extract": 128, "partial": 64},    # CLIP ViT-B/32
    "beph":            {"extract": 64,  "partial": 32},    # BEiTv2 ViT-B/16
    "pathoduet":       {"extract": 64,  "partial": 32},    # ViT-B/16 (MoCo v3)
    "hibou_b":         {"extract": 64,  "partial": 32},    # ViT-B/14 (DINOv2)
    "path_foundation": {"extract": 64,  "partial": None},  # ViT-S/16 em TensorFlow
    "fusecpath":       {"extract": None, "partial": None},  # usa os componentes
    "color":           {"extract": None, "partial": None},  # sem rede neural
}
assert set(BATCH_SIZE_T4) == set(BACKBONES), set(BACKBONES) ^ set(BATCH_SIZE_T4)


@dataclass
class Config:
    # --- caminhos ---
    drive_dataset_dir: str = "/content/drive/MyDrive/Original_ROI_images (1).zip"  # ajustado na célula
    local_dataset_dir: str = "/content/dataset"
    drive_results_dir: str = "/content/drive/MyDrive/resultados_histopatologia"
    embeddings_dir: str = "/content/drive/MyDrive/resultados_histopatologia/embeddings_cache"
    run_name: Optional[str] = None   # None => run_AAAAMMDD_HHMMSS; com nome fixo, permite retomar
    resume: bool = True              # pula (cenário, semente) já salvos em run_name
    # --- seleção de testes ---
    seeds: List[int] = field(default_factory=lambda: list(SEEDS))
    partial_seeds: Optional[List[int]] = None   # None => as mesmas de 'seeds'
    backbone: List[str] = field(default_factory=lambda: ["all"])
    regime: List[str] = field(default_factory=lambda: ["all"])
    loss: List[str] = field(default_factory=lambda: ["all"])
    lr_variant: List[str] = field(default_factory=lambda: ["mid"])
    # --- split holdout (train_test_split) ---
    train_frac: float = 0.70
    val_frac: float = 0.20
    test_frac: float = 0.10
    split_by_group: bool = True      # divide imagens de origem (evita vazamento entre ROIs)
    subset_groups_per_class: Optional[int] = None  # só para smoke test (nº de imagens/classe)
    # --- multi-crop e magnificação ---
    scales: List[float] = field(default_factory=lambda: [1.0])  # ex.: [1.0, 0.5] p/ sensibilidade
    native_mpp: Optional[float] = None   # µm/pixel da captura original, se conhecido
    magnificacao_captura: Optional[float] = None   # objetiva da captura (ex.: 20 ou 40), se conhecida
    mag_zooms: List[float] = field(default_factory=lambda: list(MAG_ZOOMS_PADRAO))   # grade da fase 4
    peft_scale: float = 1.0          # escala usada no LoRA/BitFit (a fase 3 usa a melhor da fase 4)
    # --- dias de coleta ---
    estratificar_por_dia: bool = False   # True muda os splits: resultados vão para pastas frozen_dia_*
    # --- sonda linear (regimes frozen) ---
    c_grid: List[float] = field(default_factory=lambda: np.logspace(-5, 1, 13).tolist())
    c_patience: int = 3          # para a busca de C após N valores sem melhora (0 = grade toda)
    probe_max_iter: int = 3000
    frozen_aug_copies: int = 4   # nº de cópias aumentadas por ROI (regimes *_aug)
    classifiers: List[str] = field(default_factory=lambda: list(CLASSIFIERS))
    # --- PEFT ---
    peft_lp_init: bool = True    # cabeça do LoRA/BitFit começa da sonda L-BFGS (LP-FT)
    # --- regime partial ---
    n_unfrozen_blocks: int = 2
    weight_decay: float = 1e-2
    epochs: int = 20
    patience: int = 6
    warmup_ratio: float = 0.05
    grad_clip: float = 1.0
    label_smoothing: float = 0.0   # 0 para todas as losses (evita confundir CE vs ordinais)
    use_amp: bool = True
    num_workers: int = 2
    # --- batch size ---
    effective_batch: int = 32    # ROIs por passo do otimizador (lote físico x acumulação)
    # Lista de configurações do partial: None = automático (tabela + VRAM); (lote, efetivo)
    # = fixo, para o estudo de impacto do batch size.
    partial_batch_configs: List[Optional[Tuple[int, int]]] = field(default_factory=lambda: [None])
    vram_scaling: bool = True
    max_oom_retries: int = 4
    # --- estatística ---
    n_boot: int = 1000
    ci_level: float = 0.95
    # --- saídas ---
    show_plots: bool = True
    plot_per_scenario: bool = True   # matriz de confusão e curva por cenário (desligar na grade)
    consolidar_sem_pendentes: bool = False   # refazer tabelas/gráficos mesmo sem execuções novas
    # --- ambiente ---
    install_deps: bool = True


def _expand(values: List[str], universe) -> List[str]:
    return list(universe) if "all" in values else list(values)


def _in_notebook() -> bool:
    import sys
    return "ipykernel" in sys.modules


def build_config(argv=None) -> Config:
    """CLI (--backbone, --regime, --loss, --lr_variant etc.). Ignora argumentos do Colab."""
    p = argparse.ArgumentParser(allow_abbrev=False)
    p.add_argument("--backbone", nargs="+", default=["all"], choices=list(BACKBONES) + ["all"])
    p.add_argument("--regime", nargs="+", default=["all"], choices=list(REGIMES) + ["all"])
    p.add_argument("--loss", nargs="+", default=["all"], choices=list(LOSSES) + ["all"])
    p.add_argument("--lr_variant", nargs="+", default=["mid"], choices=list(LR_VARIANTS) + ["all"])
    p.add_argument("--seeds", nargs="+", type=int, default=list(SEEDS))
    p.add_argument("--scales", nargs="+", type=float, default=[1.0])
    p.add_argument("--native_mpp", type=float, default=None)
    p.add_argument("--epochs", type=int, default=Config.epochs)
    p.add_argument("--effective_batch", type=int, default=Config.effective_batch)
    p.add_argument("--run_name", default=None)
    args, _ = p.parse_known_args(argv if argv is not None else [] if _in_notebook() else None)
    return Config(
        backbone=_expand(args.backbone, BACKBONES), regime=_expand(args.regime, REGIMES),
        loss=_expand(args.loss, LOSSES), lr_variant=_expand(args.lr_variant, LR_VARIANTS),
        seeds=args.seeds, scales=args.scales, native_mpp=args.native_mpp, epochs=args.epochs,
        effective_batch=args.effective_batch, run_name=args.run_name,
    )


CFG = build_config()

# ---------------------------------------------------------------------------------------
# No Colab, a bateria de testes da CÉLULA 10 cria as configurações de cada fase a partir
# deste CFG base. Para uma execução avulsa, edite aqui e chame main(CFG) na CÉLULA 10.
# A lista de backbones abaixo só controla quais dependências extras a CÉLULA 2 instala.
# ---------------------------------------------------------------------------------------
if _in_notebook():
    # Modelos que a CÉLULA 10 vai rodar (foundations novas + baselines) e os componentes do
    # fusecpath (UNI, CONCH, Virchow, GigaPath); define as dependências que a CÉLULA 2 instala.
    CFG = Config(backbone=list(BACKBONES), regime=["all"], loss=["all"], seeds=list(SEEDS))

import zipfile  # noqa: E402
from pathlib import Path  # noqa: E402

from google.colab import drive  # noqa: E402

drive.mount("/content/drive")
pasta_extraida = Path("/content/Original_ROI_images")


def localizar_zip_dataset(mydrive: Path, pista: str = "Original_ROI_images") -> Path:
    """Procura o zip do dataset em MyDrive por conteúdo do nome (o Drive adiciona '(1)' etc.)."""
    candidatos = sorted(mydrive.rglob(f"*{pista}*.zip"))
    if len(candidatos) == 1:
        print(f"[dataset] zip localizado: {candidatos[0]}")
        return candidatos[0]
    if not candidatos:
        todos_zips = sorted(p.name for p in mydrive.rglob("*.zip"))
        raise FileNotFoundError(f"Nenhum .zip com '{pista}' em {mydrive}. Zips: {todos_zips[:30]}")
    raise FileNotFoundError(f"Mais de um candidato: {[str(p) for p in candidatos]}. Escolha um.")


if not pasta_extraida.exists():
    with zipfile.ZipFile(localizar_zip_dataset(Path("/content/drive/MyDrive")), "r") as z:
        z.extractall(pasta_extraida)


def encontrar_raiz_classes(base: Path, classes) -> Path:
    for cand in [base] + [p for p in base.rglob("*") if p.is_dir()]:
        if all((cand / c).is_dir() for c in classes):
            return cand
    raise FileNotFoundError(f"Não encontrei as pastas {classes} dentro de {base}.")


CFG.drive_dataset_dir = str(encontrar_raiz_classes(pasta_extraida, CLASS_NAMES))
print("Raiz do dataset:", CFG.drive_dataset_dir)
print(CFG)

# %% [CÉLULA 2] Dependências (com lock de versões), login no HF e registro do ambiente
import importlib.metadata as imd  # noqa: E402
import importlib.util  # noqa: E402
import os  # noqa: E402
import re  # noqa: E402
import subprocess  # noqa: E402
import sys  # noqa: E402


def pip_install(*pkgs: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


# Pacotes cujas versões são congeladas no lock (torch/torchvision vêm do Colab e são só registrados).
LOCK_PACKAGES = {"timm", "huggingface-hub", "scikit-learn", "scikit-image", "scipy", "pandas",
                 "tabulate", "matplotlib", "conch", "transformers", "einops", "einops-exts",
                 "environs", "sacremoses", "safetensors"}
LOCK_PATH = Path(CFG.drive_results_dir) / "requirements_lock.txt"


def _norm_pkg(name: str) -> str:
    return name.strip().lower().replace("_", "-")


def write_lock(path: Path) -> None:
    """Grava as versões exatas instaladas. Nas execuções seguintes, o lock é reinstalado."""
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True,
                            text=True, check=True).stdout.splitlines()
    keep = [ln for ln in freeze if _norm_pkg(re.split(r"[=@ ]", ln, maxsplit=1)[0]) in LOCK_PACKAGES]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(keep) + "\n")
    print(f"[deps] lock gravado em {path}:\n  " + "\n  ".join(keep))


# Dependências extras por foundation (instaladas só se o modelo estiver em CFG.backbone).
# Versões pedidas nos model cards oficiais (HF):
#   TITAN : transformers==4.46.0, timm==1.0.3, einops==0.6.1, einops-exts==0.0.4
#   PRISM : transformers==4.42.4   (CONFLITA com o TITAN: não rodam na mesma sessão)
# Por padrão a sessão usa a versão do TITAN (e vale para PLIP e Hibou-B). Para rodar o PRISM,
# troque TRANSFORMERS_VERSAO para "4.42.4", deixe só "prism" em MODELOS_PARA_TESTAR
# (CÉLULA 10) e reinicie o ambiente de execução antes de rodar (o pip não troca a versão de
# um módulo já importado).
TRANSFORMERS_VERSAO = "4.46.0"
EXTRA_DEPS = {
    "titan": ["einops==0.6.1", "einops-exts==0.0.4"],
    "prism": ["environs", "sacremoses"],
    "athena": ["safetensors"],
}
NEEDS_TRANSFORMERS = {"titan", "prism", "plip", "hibou_b", "conch", "fusecpath"}
# transformers 4.46.0 exige huggingface-hub < 1.0. O lock antigo trazia huggingface-hub 1.x e
# desfazia o pin (o pip só avisa do conflito, não falha). Por isso o trio abaixo é ignorado
# ao reinstalar o lock e instalado sempre em conjunto, com versões compatíveis entre si.
HUB_SPEC = "huggingface-hub>=0.23.2,<1.0"
LOCK_IGNORADOS = {"transformers", "huggingface-hub", "tokenizers"}
_MODULE_OF = {"einops-exts": "einops_exts"}


def _missing(pkgs):
    out = []
    for p in pkgs:
        base = re.split(r"[=<>]", p)[0]
        if importlib.util.find_spec(_MODULE_OF.get(base, base).replace("-", "_")) is None:
            out.append(p)
    return out


def _ver(dist: str) -> Optional[str]:
    try:
        return imd.version(dist)
    except imd.PackageNotFoundError:
        return None


def _lock_filtrado(path: Path) -> Path:
    linhas = [ln for ln in path.read_text().splitlines()
              if ln.strip() and _norm_pkg(re.split(r"[=@ <>~!]", ln.strip(), maxsplit=1)[0]) not in LOCK_IGNORADOS]
    tmp = Path("/content/_requirements_lock_filtrado.txt")
    tmp.write_text("\n".join(linhas) + "\n")
    return tmp


def _checar_modulos_desatualizados() -> None:
    """O pip troca os arquivos no disco, mas um módulo já importado continua na memória com o
    código antigo (ex.: huggingface_hub meio antigo, meio novo). Nesse caso só reiniciar resolve."""
    obsoletos = []
    for mod, dist in (("huggingface_hub", "huggingface-hub"), ("transformers", "transformers"),
                      ("tokenizers", "tokenizers"), ("timm", "timm"), ("safetensors", "safetensors")):
        m = sys.modules.get(mod)
        em_memoria, no_disco = getattr(m, "__version__", None), _ver(dist)
        if m is not None and em_memoria and no_disco and em_memoria != no_disco:
            obsoletos.append(f"{dist}: {em_memoria} em memória x {no_disco} instalada")
    if obsoletos:
        raise RuntimeError(
            "O ambiente foi atualizado com bibliotecas já carregadas (" + "; ".join(obsoletos) + "). "
            "Reinicie a sessão (Ambiente de execução > Reiniciar sessão; NÃO precisa excluir o "
            "ambiente) e rode tudo de novo a partir da célula 1.")


if CFG.install_deps:
    if LOCK_PATH.exists():
        print(f"[deps] instalando versões congeladas de {LOCK_PATH} (sem transformers/huggingface-hub/tokenizers)")
        pip_install("-r", str(_lock_filtrado(LOCK_PATH)))
    else:
        pip_install("timm>=1.0.3", "huggingface_hub", "scikit-learn", "scikit-image", "scipy",
                    "pandas", "tabulate", "matplotlib")
    extras = sorted({p for bb in CFG.backbone for p in EXTRA_DEPS.get(bb, [])})
    if _missing(extras):
        pip_install(*_missing(extras))
    if NEEDS_TRANSFORMERS & set(CFG.backbone):   # pin DEPOIS do lock, para prevalecer
        _hub = _ver("huggingface-hub")
        _hub_ok = _hub is not None and int(_hub.split(".")[0]) < 1
        if _ver("transformers") != TRANSFORMERS_VERSAO or not _hub_ok:
            print(f"[deps] transformers {_ver('transformers')} / huggingface-hub {_hub} -> "
                  f"{TRANSFORMERS_VERSAO} / {HUB_SPEC}")
            pip_install(f"transformers=={TRANSFORMERS_VERSAO}", HUB_SPEC)
    if ("conch" in CFG.backbone or "fusecpath" in CFG.backbone) and importlib.util.find_spec("conch") is None:
        pip_install("git+https://github.com/Mahmoodlab/CONCH.git")
    if not LOCK_PATH.exists():
        write_lock(LOCK_PATH)
    _checar_modulos_desatualizados()

# Athena-0 depende do código oficial do DINOv2 no PYTHONPATH (instrução do model card).
DINOV2_COMMIT = None
if "athena" in CFG.backbone:
    dino_dir = Path("/content/dinov2")
    if not (dino_dir / "dinov2").is_dir():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/dinov2.git", str(dino_dir)],
                       check=True)
    if str(dino_dir) not in sys.path:
        sys.path.insert(0, str(dino_dir))
    DINOV2_COMMIT = subprocess.run(["git", "-C", str(dino_dir), "rev-parse", "HEAD"],
                                   capture_output=True, text=True).stdout.strip()

from huggingface_hub import HfApi, get_token, login  # noqa: E402


def _hf_login() -> Optional[str]:
    """Obtém e VALIDA o token do Hugging Face. Ordem: Secret HF_TOKEN do Colab, variável de
    ambiente HF_TOKEN, token já salvo na sessão. Se nenhum servir, PEDE o token digitado
    (getpass: o código ESPERA você colar e teclar Enter; Enter vazio = seguir sem token).
    Obs.: o notebook_login() da versão anterior só exibia um widget e deixava a célula seguir
    sem esperar; por isso o código parecia 'não pedir' o login."""
    candidatos = []
    try:
        from google.colab import userdata  # noqa: E402
        candidatos.append(("Secret HF_TOKEN do Colab", userdata.get("HF_TOKEN")))
    except Exception:   # noqa: BLE001 — fora do Colab, sem Secret ou sem acesso do notebook ao Secret
        pass
    candidatos += [("variável de ambiente HF_TOKEN", os.environ.get("HF_TOKEN")),
                   ("token salvo na sessão", get_token())]
    for origem, tok in candidatos:
        if not tok:
            continue
        try:
            usuario = HfApi().whoami(token=tok).get("name")
        except Exception as e:   # noqa: BLE001
            print(f"[hf] token de '{origem}' recusado ({type(e).__name__}); ignorado")
            continue
        login(token=tok)
        print(f"[hf] login ok via {origem} (usuário {usuario})")
        return tok
    import getpass
    for _ in range(3):
        tok = getpass.getpass("Cole o token do Hugging Face (huggingface.co/settings/tokens) e tecle Enter "
                              "(Enter vazio = seguir sem token): ").strip()
        if not tok:
            break
        try:
            usuario = HfApi().whoami(token=tok).get("name")
        except Exception as e:   # noqa: BLE001
            print(f"[hf] token recusado ({type(e).__name__}); tente de novo")
            continue
        login(token=tok)
        print(f"[hf] login ok (usuário {usuario})")
        return tok
    print("[hf] SEM TOKEN: os modelos fechados (UNI, CONCH, Virchow, TITAN, PRISM, GigaPath, "
          "Hibou-B, Athena, path-foundation) vão falhar com erro de acesso.")
    return None


HF_TOKEN = _hf_login()


def _file_fingerprint(path: Path) -> Optional[str]:
    """Impressão digital barata de um peso local (tamanho + sha1 do 1º MB): registra QUAL
    arquivo foi usado sem ler centenas de MB do Drive."""
    if not path.exists():
        return None
    with open(path, "rb") as f:
        head = f.read(1 << 20)
    return f"{path.stat().st_size}:{hashlib.sha1(head).hexdigest()[:16]}"


def collect_env() -> Dict:
    """Versões de pacotes, GPU e commit (sha) dos pesos usados no HF."""
    import torch  # noqa: E402
    pkgs = {}
    for name in sorted(LOCK_PACKAGES | {"torch", "torchvision", "numpy"}):
        try:
            pkgs[name] = imd.version(name)
        except imd.PackageNotFoundError:
            pass
    hf_sha = {}
    api = HfApi()
    for bb, repo in HF_REPOS.items():
        try:
            hf_sha[bb] = api.model_info(repo, token=HF_TOKEN).sha
        except Exception as e:  # repositório sem acesso: registra o motivo
            hf_sha[bb] = f"indisponível ({type(e).__name__})"
    return {
        "python": sys.version.split()[0], "packages": pkgs,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "vram_gb": (round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
                    if torch.cuda.is_available() else None),
        "cuda": torch.version.cuda, "hf_sha": hf_sha, "dinov2_commit": DINOV2_COMMIT,
        "local_weights": {bb: _file_fingerprint(Path(PESOS_DIR) / fn) for bb, fn in LOCAL_WEIGHTS.items()},
    }


ENV_INFO = collect_env()
print(json.dumps(ENV_INFO, indent=2))

# %% [CÉLULA 3] Dataset, grupos, split holdout 70/20/10, multi-crop e augmentações
import gc  # noqa: E402
import random  # noqa: E402
import shutil  # noqa: E402
import time  # noqa: E402
from datetime import datetime  # noqa: E402

import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
import torch.nn.functional as F  # noqa: E402
import torchvision.transforms as T  # noqa: E402
from PIL import Image  # noqa: E402
from skimage.color import hed2rgb, rgb2hed  # noqa: E402
from sklearn.exceptions import UndefinedMetricWarning  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402
from torch.utils.data import DataLoader, Dataset  # noqa: E402

# Com poucos exemplos por classe no teste/validação, o sklearn avisa a cada chamada quando uma
# classe some; o valor resultante (nan) já é tratado nas métricas.
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Ative uma GPU no Colab (Ambiente de execução > Alterar tipo)."
IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# --- Memória RAM ------------------------------------------------------------------------
# O Colab mata o processo SEM exceção do Python quando a RAM (~12,7 GB) acaba: por isso a
# execução "morria" ao carregar o GigaPath. O timm com pretrained=True monta na CPU o modelo
# inteiro (pesos aleatórios, ~4,5 GB) E o state_dict inteiro (~4,5 GB), em cima do que os
# modelos anteriores deixaram (o Python não devolve RAM ao sistema sem malloc_trim).
import ctypes  # noqa: E402

import psutil  # noqa: E402


class RAMInsuficiente(RuntimeError):
    """Sem RAM livre para carregar o modelo com segurança: o modelo é pulado com aviso claro."""


RAM_NECESSARIA_GB = {   # pico de RAM (CPU) ao carregar, já com o carregamento econômico
    "gigapath": 3.0, "athena": 3.0, "prism": 4.5, "virchow": 4.0, "titan": 4.0, "uni": 2.5,
    "conch": 2.0, "path_foundation": 3.5, "hibou_b": 1.5, "beph": 1.5, "pathoduet": 1.5,
    "chief": 1.0, "ctranspath": 1.0, "lunit": 1.0, "plip": 1.5, "imagenet_r50": 1.0,
}
RAM_MARGEM_GB = 1.0
CARREGAR_DIRETO_NA_GPU = {"gigapath"}   # cria a arquitetura na GPU e lê os pesos por mmap


def ram_status() -> Dict[str, float]:
    vm = psutil.virtual_memory()
    return {"livre_gb": vm.available / 1024**3, "total_gb": vm.total / 1024**3,
            "processo_gb": psutil.Process().memory_info().rss / 1024**3}


def ram_txt() -> str:
    s = ram_status()
    return f"RAM livre {s['livre_gb']:.1f}/{s['total_gb']:.1f} GB (processo usa {s['processo_gb']:.1f} GB)"


def liberar_memoria(rotulo: str = "", verbose: bool = True) -> None:
    """Libera de verdade: coleta de lixo, cache da GPU e malloc_trim (devolve ao sistema a RAM
    que o Python já soltou, mas mantinha reservada)."""
    antes = ram_status()["processo_gb"]
    for _ in range(2):
        gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:   # noqa: BLE001
        pass
    if verbose:
        solto = max(0.0, antes - ram_status()["processo_gb"])
        print(f"[RAM] {rotulo + ': ' if rotulo else ''}{ram_txt()} | liberado {solto:.1f} GB")


def garantir_ram(nome: str) -> None:
    """Antes de carregar um modelo: se faltar RAM, libera; se ainda faltar, pula com mensagem
    (em vez de deixar o Colab derrubar a sessão inteira)."""
    precisa = RAM_NECESSARIA_GB.get(nome, 1.5) + RAM_MARGEM_GB
    if ram_status()["livre_gb"] >= precisa:
        return
    liberar_memoria(f"antes de carregar {nome}")
    livre = ram_status()["livre_gb"]
    if livre < precisa:
        raise RAMInsuficiente(
            f"{nome}: só {livre:.1f} GB de RAM livre; preciso de ~{precisa:.1f} GB para carregar com "
            f"segurança. Modelo PULADO (o Colab derrubaria a sessão). Reinicie a sessão e rode só "
            f"MODELOS_PARA_TESTAR = ['{nome}'] — o que já foi feito é reaproveitado.")


def _list_images(class_dir: Path) -> List[Path]:
    return sorted(p for p in class_dir.rglob("*") if p.suffix.lower() in IMG_EXTS)


def copy_dataset_to_local(src: str, dst: str) -> Path:
    """Copia para o disco local ANTES de criar os DataLoaders (evita gargalo de I/O)."""
    src_p, dst_p = Path(src), Path(dst)
    n_src = sum(len(_list_images(src_p / c)) for c in CLASS_NAMES)
    n_dst = sum(len(_list_images(dst_p / c)) for c in CLASS_NAMES if (dst_p / c).is_dir())
    if n_src == n_dst:
        print(f"[dataset] cópia local já completa ({n_dst} imagens) em {dst_p}")
        return dst_p
    for c in CLASS_NAMES:
        shutil.copytree(src_p / c, dst_p / c, dirs_exist_ok=True)
    assert sum(len(_list_images(dst_p / c)) for c in CLASS_NAMES) == n_src, "Cópia incompleta"
    return dst_p


def scan_dataset(root: Path):
    paths, labels = [], []
    for idx, c in enumerate(CLASS_NAMES):
        files = _list_images(root / c)
        assert files, f"Nenhuma imagem em {root / c}"
        paths += [str(p) for p in files]
        labels += [idx] * len(files)
    return paths, np.array(labels, dtype=np.int64)


# --- Grupos (imagem de origem) ---
ROI_RE = re.compile(r"(.*)-roi\d+$")


def _pos_classe(parts) -> int:
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] in CLASS_NAMES:
            return i
    return max(0, len(parts) - 2)


def group_id(path: str) -> str:
    """Imagem de origem = pasta da classe + subpastas (ex.: do dia) + nome sem '-roiN':
    'healthy/image032-roi1' -> 'healthy/image032'; 'healthy/dia2/image032-roi1' ->
    'healthy/dia2/image032'. Sem subpastas o resultado é IDÊNTICO ao da v10 (splits iguais).
    A numeração recomeçou no 2º dia: o mesmo número aparece em classes diferentes (ex.:
    healthy/image032 e moderate/image032) e, dentro da classe, com sufixo '-2' (mild/image045
    e mild/image045-2). São imagens diferentes e ficam em grupos separados."""
    p = Path(path)
    parts = p.parts
    i = _pos_classe(parts)
    m = ROI_RE.match(p.stem)
    return "/".join(list(parts[i:-1]) + [m.group(1) if m else p.stem])


# --- Dias de coleta -----------------------------------------------------------------------
# Os arquivos dos dois dias estão MISTURADOS na mesma pasta de cada classe; nem o nome nem os
# metadados do TIFF dizem o dia. Fontes, em ordem: (1) subpasta entre a classe e o arquivo;
# (2) dias_coleta.csv (colunas grupo,dia) na pasta de resultados; (3) '?' (desconhecido).
NUM_RE = re.compile(r"(\d+)(-\d+)?$")
_DIAS_CACHE: Optional[Dict[str, str]] = None


def _dias_csv() -> Path:
    return Path(CFG.drive_results_dir) / "dias_coleta.csv"


def dia_do_grupo(g: str) -> str:
    global _DIAS_CACHE
    partes = g.split("/")
    if len(partes) > 2:                       # classe/<subpasta>/imagem
        return "/".join(partes[1:-1])
    if _DIAS_CACHE is None:
        _DIAS_CACHE = {}
        f = _dias_csv()
        if f.exists():
            try:
                import pandas as _pd
                d = _pd.read_csv(f, dtype=str).dropna(subset=["grupo", "dia"])
                _DIAS_CACHE = {r.grupo.strip(): r.dia.strip() for r in d.itertuples() if r.dia.strip()}
            except Exception as e:   # noqa: BLE001
                print(f"[dias] não consegui ler {f}: {e}")
    return _DIAS_CACHE.get(g, "?")


_AUDITADO = False


def auditar_dataset(paths: List[str], labels: np.ndarray, groups: np.ndarray) -> None:
    """Roda uma vez por sessão: tamanhos das ROIs, arquivos idênticos, números repetidos entre
    classes e dentro da classe (sufixo '-2'), e a tabela classe x dia. Grava
    auditoria_dataset.csv e, se não existir, um modelo dias_coleta_modelo.csv para preencher."""
    global _AUDITADO
    if _AUDITADO:
        return
    _AUDITADO = True
    import pandas as _pd
    print("\n" + "=" * 90 + "\n= AUDITORIA DO DATASET (datas repetidas / dois dias de coleta)\n" + "=" * 90)
    tam, dig = {}, {}
    for p in paths:
        with Image.open(p) as im:
            tam[im.size] = tam.get(im.size, 0) + 1
        dig.setdefault(hashlib.md5(Path(p).read_bytes()).hexdigest(), []).append(p)
    print(f"tamanhos das ROIs: {tam}")
    dup = [v for v in dig.values() if len(v) > 1]
    if dup:
        print(f"  [PROBLEMA] {len(dup)} conjunto(s) de arquivos IDÊNTICOS (vazamento possível):")
        for v in dup[:10]:
            print("     ", [str(Path(x).relative_to(Path(x).parents[1])) for x in v])
    else:
        print("arquivos idênticos: nenhum")
    linhas = []
    for g in dict.fromkeys(groups):
        nome = g.rsplit("/", 1)[-1]
        m = NUM_RE.search(nome)
        linhas.append({"grupo": g, "classe": g.split("/")[0], "numero": m.group(1) if m else nome,
                       "sufixo": (m.group(2) or "") if m else "", "n_rois": int((groups == g).sum()),
                       "dia": dia_do_grupo(g)})
    a = _pd.DataFrame(linhas)
    print("imagens de origem por classe:", a.groupby("classe").size().reindex(CLASS_NAMES).to_dict(),
          "| ROIs por classe:", dict(zip(CLASS_NAMES, np.bincount(labels, minlength=NUM_CLASSES).tolist())))
    entre = a.groupby("numero").classe.nunique()
    entre = entre[entre > 1]
    print(f"números repetidos ENTRE classes: {len(entre)} (ex.: "
          + ", ".join(f"{n}:{'/'.join(sorted(a[a.numero == n].classe.unique()))}" for n in entre.index[:6])
          + ") -> grupos separados (classe faz parte do grupo)")
    dentro = a.groupby(["classe", "numero"]).size()
    dentro = dentro[dentro > 1]
    print(f"mesmo número DENTRO da classe (com e sem sufixo): "
          f"{[f'{c}/{n}' for c, n in dentro.index] or 'nenhum'} -> grupos separados")
    print(f"grupos com sufixo '-N': {int((a.sufixo != '').sum())} "
          f"({a[a.sufixo != ''].groupby('classe').size().to_dict()})")
    if (a.dia != "?").any():
        print("tabela classe x dia (imagens de origem):")
        print(_pd.crosstab(a.classe, a.dia).reindex(CLASS_NAMES).fillna(0).astype(int).to_string())
        prop = _pd.crosstab(a.classe, a.dia, normalize="index").max(axis=1)
        if (prop > 0.8).any():
            print(f"  [ATENÇÃO] classe(s) com >80% das imagens num só dia: {prop[prop > 0.8].index.tolist()} "
                  f"-> o dia pode se confundir com a classe (confira o baseline de cor)")
    else:
        print(f"dia de coleta: DESCONHECIDO para todas as imagens (sem subpastas e sem {_dias_csv().name}).")
        print("  O agrupamento continua correto (classe + nome); para medir o efeito do dia, preencha")
        print(f"  a coluna 'dia' de {_dias_csv().with_name('dias_coleta_modelo.csv')} e salve como {_dias_csv().name}.")
    try:
        out = Path(CFG.drive_results_dir)
        out.mkdir(parents=True, exist_ok=True)
        a.to_csv(out / "auditoria_dataset.csv", index=False)
        modelo = out / "dias_coleta_modelo.csv"
        if not _dias_csv().exists() and not modelo.exists():
            a.assign(dia="")[["grupo", "classe", "numero", "sufixo", "dia"]].to_csv(modelo, index=False)
    except Exception as e:   # noqa: BLE001
        print(f"[auditoria] não gravei os CSVs: {e}")


def subset_by_groups(labels: np.ndarray, groups: np.ndarray, k: int, seed: int = 0) -> np.ndarray:
    """Índices de um subconjunto com k imagens de origem por classe (só para o smoke test)."""
    rng = np.random.default_rng(seed)
    keep = []
    for c in range(NUM_CLASSES):
        g = np.unique(groups[labels == c])
        rng.shuffle(g)
        keep += list(g[:k])
    return np.flatnonzero(np.isin(groups, keep))


# --- Split holdout 70/20/10 com train_test_split ---
def make_holdout_split(labels: np.ndarray, groups: np.ndarray, seed: int, cfg: Config) -> Dict[str, np.ndarray]:
    """Split treino/validação/teste com duas chamadas de train_test_split, estratificadas pela
    classe. Com split_by_group=True as unidades sorteadas são as IMAGENS DE ORIGEM (todas as
    ROIs de uma imagem vão juntas para a mesma partição): sem isso, ROIs vizinhas da mesma
    lâmina aparecem no treino e no teste e o desempenho fica inflado. As proporções 70/20/10
    valem para as imagens; para as ROIs ficam aproximadas (impressas no log)."""
    fr = (cfg.train_frac, cfg.val_frac, cfg.test_frac)
    assert min(fr) > 0 and abs(sum(fr) - 1) < 1e-6, f"frações inválidas: {fr}"
    if cfg.split_by_group:
        units, first = np.unique(groups, return_index=True)
        unit_y = labels[first]
        for u, yu in zip(units, unit_y):   # grupo = pasta da classe + imagem => uma classe só
            assert (labels[groups == u] == yu).all(), f"grupo {u} com mais de uma classe"
    else:
        units, unit_y = np.arange(len(labels)), labels
    u = np.arange(len(units))
    strat = unit_y.astype(str)
    if cfg.estratificar_por_dia and cfg.split_by_group:
        dias = np.array([dia_do_grupo(str(g)) for g in units])
        cand = np.char.add(np.char.add(strat, "|"), dias)
        _, cont = np.unique(cand, return_counts=True)
        if (dias == "?").any() or cont.min() < 4:
            print("[split] estratificar_por_dia ignorado: dia desconhecido ou estrato classe x dia "
                  f"com menos de 4 imagens (mínimo {cont.min()})")
        else:
            strat = cand
    u_tv, u_te = train_test_split(u, test_size=cfg.test_frac, stratify=strat[u], random_state=seed)
    u_tr, u_va = train_test_split(u_tv, test_size=cfg.val_frac / (cfg.train_frac + cfg.val_frac),
                                  stratify=strat[u_tv], random_state=seed)

    def to_idx(uu):
        return np.flatnonzero(np.isin(groups, units[uu])) if cfg.split_by_group else np.sort(units[uu])

    sp = {"train": to_idx(u_tr), "val": to_idx(u_va), "test": to_idx(u_te)}
    allidx = np.concatenate(list(sp.values()))
    assert len(allidx) == len(labels) and len(np.unique(allidx)) == len(labels), "split não particiona"
    if cfg.split_by_group:
        g = {k: set(groups[v]) for k, v in sp.items()}
        assert not (g["train"] & g["val"] or g["train"] & g["test"] or g["val"] & g["test"]), \
            "vazamento: imagem de origem em mais de uma partição"
    return sp


def split_summary(splits: Dict[int, Dict[str, np.ndarray]], labels, groups) -> "pd.DataFrame":
    rows = []
    for seed, sp in splits.items():
        r = {"semente": seed}
        for part in ("train", "val", "test"):
            idx = sp[part]
            r[f"{part}_rois"] = len(idx)
            r[f"{part}_%"] = round(100 * len(idx) / len(labels), 1)
            r[f"{part}_imagens"] = len(set(groups[idx]))
            r[f"{part}_por_classe"] = np.bincount(labels[idx], minlength=NUM_CLASSES).tolist()
        rows.append(r)
    import pandas as pd
    return pd.DataFrame(rows)


# --- Multi-crop: janelas no tamanho nativo do backbone cobrindo toda a ROI ---
def _resize_height(img: Image.Image, height: int) -> Image.Image:
    w, h = img.size
    return img.resize((max(1, round(w * height / h)), height), Image.BICUBIC)


def roi_views(img: Image.Image, size: int, scale: float = 1.0) -> List[Image.Image]:
    """Redimensiona a altura da ROI para size*scale e extrai janelas size x size que cobrem
    toda a imagem (2 janelas para 450x250). Com scale < 1 (análise de magnificação), a ROI
    reduzida não preenche a janela e é completada por reflexão."""
    arr = np.asarray(_resize_height(img, max(1, round(size * scale))))
    ph, pw = max(0, size - arr.shape[0]), max(0, size - arr.shape[1])
    if ph or pw:
        arr = np.pad(arr, ((ph // 2, ph - ph // 2), (pw // 2, pw - pw // 2), (0, 0)), mode="reflect")
    H, W = arr.shape[:2]
    ys = np.linspace(0, H - size, num=max(1, math.ceil(H / size))).round().astype(int)
    xs = np.linspace(0, W - size, num=max(1, math.ceil(W / size))).round().astype(int)
    return [Image.fromarray(np.ascontiguousarray(arr[y:y + size, x:x + size]))
            for y in ys for x in xs]


# --- Augmentações morfologicamente seguras (RNG do torch => reprodutível com workers) ---
class RandomRotate90:
    """Rotação em {0, 90, 180, 270} graus, aplicada a cada janela quadrada."""
    _OPS = [None, Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180, Image.Transpose.ROTATE_270]

    def __call__(self, img):
        op = self._OPS[int(torch.randint(0, 4, (1,)))]
        return img if op is None else img.transpose(op)


class HEDJitter:
    """Perturbação leve no espaço H&E (hematoxilina-eosina-DAB): x' = a*x + b por canal."""

    def __init__(self, alpha: float = 0.05, beta: float = 0.005, p: float = 0.5):
        self.alpha, self.beta, self.p = alpha, beta, p

    def __call__(self, img):
        if float(torch.rand(1)) > self.p:
            return img
        hed = rgb2hed(np.asarray(img))
        a = 1.0 + (torch.rand(3).numpy() * 2 - 1) * self.alpha
        b = (torch.rand(3).numpy() * 2 - 1) * self.beta
        rgb = np.clip(hed2rgb(hed * a + b), 0.0, 1.0)
        return Image.fromarray((rgb * 255).astype(np.uint8))


def build_train_aug() -> T.Compose:
    """Aplicada à ROI inteira (só operações que preservam o formato 450x250);
    a rotação de 90 graus é aplicada depois, em cada janela quadrada."""
    return T.Compose([
        T.RandomHorizontalFlip(0.5),
        T.RandomVerticalFlip(0.5),
        T.RandomApply([T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02)], p=0.8),
        HEDJitter(),
    ])


AUG_DESC = "hflip0.5|vflip0.5|cj(0.1,0.1,0.1,0.02)p0.8|hed(0.05,0.005)p0.5|rot90/janela"


class ROIDataset(Dataset):
    """Devolve TODAS as janelas da ROI (V x 3 x S x S), já normalizadas para o backbone."""

    def __init__(self, paths, labels, size: int, normalize, scale: float = 1.0, aug=None,
                 item_seed: Optional[int] = None):
        self.paths, self.labels, self.size = paths, labels, size
        self.normalize, self.scale, self.aug = normalize, scale, aug
        self.item_seed = item_seed   # semente por ROI: augmentation independe de lote/workers
        self._rot = RandomRotate90()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.aug is not None and self.item_seed is not None:
            torch.manual_seed(self.item_seed * 1_000_003 + i)
        if self.aug is not None:
            img = self.aug(img)
        views = roi_views(img, self.size, self.scale)
        if self.aug is not None:
            views = [self._rot(v) for v in views]
        return torch.stack([self.normalize(v) for v in views]), int(self.labels[i]), i


def collate_views(batch):
    """Concatena as janelas de todas as ROIs do lote; 'counts' diz quantas são de cada ROI."""
    xs, ys, idx = zip(*batch)
    return (torch.cat(xs), torch.tensor([x.shape[0] for x in xs]),
            torch.tensor(ys, dtype=torch.long), torch.tensor(idx, dtype=torch.long))


def _seed_worker(worker_id: int):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


def make_loader(ds: Dataset, batch_size: int, shuffle: bool, seed: int, cfg: Config) -> DataLoader:
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=cfg.num_workers,
                      pin_memory=True, generator=g, worker_init_fn=_seed_worker,
                      collate_fn=collate_views, persistent_workers=cfg.num_workers > 0)


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# --- Baseline de cor: nenhuma informação morfológica, só distribuição de cor da ROI ---
def color_features(paths: List[str], bins: int = 8) -> np.ndarray:
    feats = []
    for p in paths:
        img = Image.open(p).convert("RGB")
        rgb = np.asarray(img, dtype=np.float64).reshape(-1, 3) / 255
        hsv = np.asarray(img.convert("HSV"), dtype=np.float64).reshape(-1, 3) / 255
        hist = [np.histogram(rgb[:, k], bins=bins, range=(0, 1))[0] / len(rgb) for k in range(3)]
        feats.append(np.concatenate([rgb.mean(0), rgb.std(0), hsv.mean(0), hsv.std(0), *hist]))
    return np.stack(feats)

# %% [CÉLULA 4] Backbones, cabeças (regime partial) e relatório de magnificação
import timm  # noqa: E402
from timm.data import resolve_data_config  # noqa: E402
from timm.data.transforms_factory import create_transform  # noqa: E402

AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def split_preprocess(preprocess, fallback_size: int = 224):
    """Do transform OFICIAL do modelo extrai o tamanho de entrada (CenterCrop) e a
    normalização; o Resize/CenterCrop é substituído pelo multi-crop."""
    size, norm = None, None
    for t in getattr(preprocess, "transforms", []):
        if isinstance(t, T.CenterCrop):
            size = t.size[0] if isinstance(t.size, (tuple, list)) else t.size
        if isinstance(t, T.Normalize):
            norm = t
    assert norm is not None, f"Normalize não encontrado em {preprocess}"
    return int(size or fallback_size), T.Compose([T.ToTensor(), norm])


CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
NO_NORM = ((0.0, 0.0, 0.0), (1.0, 1.0, 1.0))   # entrada em [0, 1], sem normalização


def make_preprocess(size: int, mean, std, resize: Optional[int] = None) -> T.Compose:
    """Transform no formato que split_preprocess entende (CenterCrop + Normalize)."""
    return T.Compose([T.Resize(resize or size, interpolation=T.InterpolationMode.BICUBIC),
                      T.CenterCrop(size), T.ToTensor(), T.Normalize(mean, std)])


def _safe_split(preprocess, size: int, mean, std):
    """Usa o transform oficial se ele tiver CenterCrop/Normalize; senão, o explícito."""
    try:
        split_preprocess(preprocess, size)
        return preprocess
    except (AssertionError, TypeError):
        print(f"    [preprocess] transform oficial não reconhecido; usando {size}px mean={mean} std={std}")
        return make_preprocess(size, mean, std)


# --- Descoberta genérica dos blocos transformer (para o regime partial) ---
_BLOCK_LIST_NAMES = ("blocks", "layers", "layer", "resblocks")
_FINAL_NORM_NAMES = ("norm", "fc_norm", "post_layernorm", "layernorm", "ln_post")
_ATTN_NAMES = ("attn", "attention", "self_attn")


def _flat(lst: nn.Module) -> List[nn.Module]:
    out = []
    for m in lst:
        if isinstance(m, nn.ModuleList):               # BlockChunk do DINOv2
            out += [b for b in m if not isinstance(b, nn.Identity)]
        elif not isinstance(m, nn.Identity):
            out.append(m)
    return out


def _has_attn(m: nn.Module) -> bool:
    return any(n.split(".")[-1] in _ATTN_NAMES for n, _ in m.named_modules())


def find_blocks(root: nn.Module) -> List[nn.Module]:
    """Lista ordenada dos blocos transformer de 'root'. Candidatas = ModuleList/Sequential
    chamadas blocks/layers/layer/resblocks cujos elementos têm atenção; ficam só as mais
    internas. Cobre ViT do timm, DINOv2 com BlockChunk, Swin (layers[i].blocks), HF
    ViT/Dinov2 (encoder.layer), CLIP (encoder.layers) e open_clip (resblocks)."""
    cands = []
    for name, mod in root.named_modules():
        if (isinstance(mod, (nn.ModuleList, nn.Sequential)) and name.split(".")[-1] in _BLOCK_LIST_NAMES
                and len(_flat(mod)) > 0 and all(_has_attn(b) for b in _flat(mod))):
            cands.append((name, mod))
    leaves = [(n, m) for n, m in cands
              if not any(o != n and o.startswith(n + ".") for o, _ in cands)]
    return [b for _, m in leaves for b in _flat(m)]


def find_final_norms(root: nn.Module) -> List[nn.Module]:
    out = []
    for n in _FINAL_NORM_NAMES:
        m = getattr(root, n, None)
        if isinstance(m, nn.Module) and any(True for _ in m.parameters()):
            out.append(m)
    return out


class LoRALinear(nn.Module):
    """Linear congelado + atualização de posto baixo: y = W x + (alpha/r) B A x (Hu et al., 2022).
    B começa em zero, então o modelo é idêntico ao original até o treino começar. O dropout do
    LoRA só atua quando dropout_on=True (ligado pela LinearProbeModel durante o treino, já que o
    encoder fica sempre em modo eval)."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base, self.r, self.scaling, self.p = base, r, alpha / r, dropout
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features, device=dev))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, device=dev))
        self.dropout_on = False
        self.reset_lora()

    def reset_lora(self) -> None:
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    # alguns modelos leem estes atributos diretamente do módulo (ex.: F.linear(x, self.qkv.weight))
    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    @property
    def in_features(self):
        return self.base.in_features

    @property
    def out_features(self):
        return self.base.out_features

    def forward(self, x):
        out = self.base(x)
        h = F.dropout(x, self.p, training=self.dropout_on) if self.p else x
        upd = (h.to(self.lora_A.dtype) @ self.lora_A.t()) @ self.lora_B.t()
        return out + (upd * self.scaling).to(out.dtype)


class Backbone(nn.Module):
    """Interface única sobre os modelos. encode(x) segue a receita oficial de cada um;
    encode_views faz o mean pooling das janelas de cada ROI (ou, no PRISM, a agregação
    do Perceiver)."""

    def __init__(self, name: str, model, preprocess, n_unfrozen: int, encode_fn,
                 ft_root=None, views_fn=None, source: Optional[str] = None):
        super().__init__()
        self.name, self.preprocess = name, preprocess
        if isinstance(model, nn.Module):
            self.model = model
        else:                                    # ex.: TensorFlow (path_foundation)
            object.__setattr__(self, "model", model)
        self._encode_fn, self._views_fn, self._ft_root = encode_fn, views_fn, ft_root
        self.source = source or HF_IDS.get(name, name)
        pcfg = getattr(model, "pretrained_cfg", None)
        fallback = (pcfg.get("input_size", (3, 224, 224))[-1] if isinstance(pcfg, dict) else 224)
        self.view_size, self.normalize = split_preprocess(preprocess, fallback)
        print(f"[{name}] janela={self.view_size}px | {self.normalize.transforms[-1]}")
        self.to(DEVICE).eval().requires_grad_(False)
        self._n_unfrozen = n_unfrozen
        self._ft = None
        self._lora: Optional[List[LoRALinear]] = None
        self._bitfit: Optional[List[nn.Parameter]] = None
        self.embed_dim = self._probe()
        print(f"[{name}] dimensão do embedding por ROI: {self.embed_dim}")

    @property
    def _ft_params(self):
        if self._ft is None:
            root = self._vit()
            blocks = find_blocks(root)
            if not blocks:
                raise NotImplementedError(f"{self.name}: blocos transformer não encontrados")
            params = [p for blk in blocks[-self._n_unfrozen:] for p in blk.parameters()]
            for norm in find_final_norms(root):
                params += list(norm.parameters())
            self._ft = params
            self._init_state = [p.detach().cpu().clone() for p in params]
            print(f"    [{self.name}] partial: {len(blocks)} blocos, últimos {self._n_unfrozen} destravados "
                  f"({sum(p.numel() for p in params) / 1e6:.1f}M parâmetros)")
        return self._ft

    def _vit(self) -> nn.Module:
        if self.name in FROZEN_ONLY or self._ft_root is None:
            raise NotImplementedError(f"{self.name} não suporta o regime partial")
        return self._ft_root

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self._encode_fn(x)

    def encode_views(self, x: torch.Tensor, counts: torch.Tensor) -> torch.Tensor:
        if self._views_fn is not None:
            return self._views_fn(x, counts)
        z = self.encode(x)                                                # (soma das janelas) x D
        return torch.stack([c.mean(0) for c in torch.split(z, counts.tolist())])

    @torch.no_grad()
    def _probe(self) -> int:
        x = self.normalize(Image.new("RGB", (self.view_size,) * 2, (200, 150, 200)))
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            out = self.encode_views(torch.stack([x, x]).to(DEVICE), torch.tensor([2]))
        return int(out.shape[-1])

    def reset_and_configure(self, regime: str) -> None:
        """Restaura o encoder ao estado original (blocos do partial, vieses do BitFit e LoRA com
        B = 0) e define o que é treinável: 'frozen' (nada), 'partial', 'lora' ou 'bitfit'."""
        with torch.no_grad():
            if self._ft is not None:
                for p, init in zip(self._ft, self._init_state):
                    p.copy_(init.to(p.device))
            if self._bitfit is not None:
                for p, init in zip(self._bitfit, self._bitfit_init):
                    p.copy_(init.to(p.device))
            if self._lora is not None:
                for m in self._lora:
                    m.reset_lora()
        self.requires_grad_(False)
        if regime == "frozen":
            return
        if regime == "partial":
            params = self._ft_params
        elif regime == "lora":
            params = [p for m in self._lora_modules() for p in (m.lora_A, m.lora_B)]
        elif regime == "bitfit":
            params = self._bitfit_params()
            # Em algumas GPUs/versões do PyTorch a volta da LayerNorm devolve um gradiente de viés
            # VAZIO quando só o viés (e não o peso) pede gradiente (visto no CTransPath, dim 96).
            # O peso passa a pedir gradiente, mas fica fora do otimizador: nunca é atualizado.
            for m in self._vit().modules():
                if isinstance(m, nn.LayerNorm) and m.weight is not None:
                    m.weight.requires_grad_(True)
        else:
            raise ValueError(regime)
        for p in params:
            p.requires_grad_(True)

    def _bitfit_params(self) -> List[nn.Parameter]:
        """BitFit (Ben Zaken et al., 2022): só os vieses do encoder (atenção, MLP e normalizações)."""
        if self._bitfit is None:
            root = self._vit()
            ps = [p for n, p in root.named_parameters() if n.endswith("bias") and "lora_" not in n]
            if not ps:
                raise NotImplementedError(f"{self.name}: encoder sem vieses para o BitFit")
            self._bitfit = ps
            self._bitfit_init = [p.detach().cpu().clone() for p in ps]
            print(f"    [{self.name}] BitFit: {len(ps)} vetores de viés ({sum(p.numel() for p in ps) / 1e3:.0f}k parâmetros)")
        return self._bitfit

    def _lora_modules(self) -> List[LoRALinear]:
        """Injeta (uma vez) LoRA nas camadas lineares de atenção e MLP de todos os blocos e
        confere com um passo de teste que cada adaptador recebe gradiente; os que não recebem
        (ex.: camada lida por F.linear com o peso bruto) são removidos."""
        if self._lora is not None:
            return self._lora
        blocks = find_blocks(self._vit())
        if not blocks:
            raise NotImplementedError(f"{self.name}: blocos transformer não encontrados para o LoRA")
        injected = []
        for blk in blocks:
            for name, mod in list(blk.named_modules()):
                leaf = name.rsplit(".", 1)[-1]
                if not name or not isinstance(mod, nn.Linear) or leaf not in LORA_TARGETS:
                    continue
                parent = blk.get_submodule(name.rsplit(".", 1)[0]) if "." in name else blk
                w = LoRALinear(mod, PEFT_PROTOCOL["r"], PEFT_PROTOCOL["alpha"], PEFT_PROTOCOL["dropout"])
                setattr(parent, leaf, w)
                injected.append((parent, leaf, mod, w))
        if not injected:
            raise NotImplementedError(f"{self.name}: nenhuma camada-alvo do LoRA encontrada")
        # passo de teste: com B = 0, o gradiente de B só é não nulo se o adaptador está no grafo
        self.requires_grad_(False)
        for *_, w in injected:
            w.lora_B.requires_grad_(True)
        x = torch.randn(2, 3, self.view_size, self.view_size, device=DEVICE)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            z = self.encode_views(x, torch.tensor([2]))
        z.float().pow(2).mean().backward()
        live = []
        for parent, leaf, mod, w in injected:
            g = w.lora_B.grad
            if g is not None and float(g.abs().sum()) > 0:
                live.append(w)
            else:
                setattr(parent, leaf, mod)     # adaptador sem efeito: restaura a camada original
            w.lora_B.grad = None
        self.requires_grad_(False)
        if not live:
            raise NotImplementedError(f"{self.name}: o LoRA não teve efeito (nenhum adaptador no grafo)")
        tipos = sorted({n.rsplit(".", 1)[-1] for b in blocks for n, m in b.named_modules() if isinstance(m, LoRALinear)})
        n_par = sum(m.lora_A.numel() + m.lora_B.numel() for m in live)
        print(f"    [{self.name}] LoRA: {len(live)}/{len(injected)} camadas ativas em {len(blocks)} blocos "
              f"({', '.join(tipos)}) | {n_par / 1e6:.2f}M parâmetros treináveis")
        self._lora = live
        return live

    def train(self, mode: bool = True):  # sempre em eval (sem dropout/droppath, BN congelado)
        return super().train(False)


# --- Utilidades para pesos locais (CHIEF, CTransPath original, BEPH, PathoDuet) ---
def local_weight_path(name: str) -> Path:
    return Path(PESOS_DIR) / LOCAL_WEIGHTS[name]


def _require_local(name: str) -> Path:
    p = local_weight_path(name)
    if not p.exists():
        raise FileNotFoundError(f"[{name}] peso não encontrado em {p}. Baixe-o (ver LOCAL_WEIGHTS na "
                                f"CÉLULA 1) e salve com esse nome em {PESOS_DIR}.")
    return p


def _load_ckpt(path: Path) -> Dict[str, torch.Tensor]:
    ck = torch.load(path, map_location="cpu", weights_only=False)
    for k in ("model", "state_dict", "teacher", "student"):
        if isinstance(ck, dict) and k in ck and isinstance(ck[k], dict):
            ck = ck[k]
            break
    return {k: v for k, v in ck.items() if isinstance(v, torch.Tensor)}


def _strip_prefixes(sd: Dict, prefixes) -> Dict:
    out = {}
    for k, v in sd.items():
        for pf in prefixes:
            if k.startswith(pf):
                k = k[len(pf):]
        out[k] = v
    return out


def load_and_verify(model: nn.Module, sd: Dict, name: str, min_frac: float = 0.95,
                    ignore_missing=()) -> None:
    """load_state_dict(strict=False) + verificação: aborta se uma fração relevante dos pesos
    do modelo não foi carregada (evita rodar o estudo com um encoder meio aleatório)."""
    msd = model.state_dict()
    sd = {k: v for k, v in sd.items() if k in msd and v.shape == msd[k].shape}
    missing = [k for k in msd if k not in sd and not any(s in k for s in ignore_missing)]
    model.load_state_dict(sd, strict=False)
    n_ok = len([k for k in msd if k in sd])
    n_req = n_ok + len(missing)
    frac = n_ok / max(1, n_req)
    print(f"    [{name}] pesos carregados: {n_ok}/{n_req} tensores ({frac:.1%})")
    if missing:
        print(f"    [{name}] não carregados (até 10): {missing[:10]}")
    if frac < min_frac:
        raise RuntimeError(f"[{name}] só {frac:.1%} dos pesos foram carregados; confira o checkpoint.")


# --- CTransPath / CHIEF: Swin-T com ConvStem (timm moderno, sem o timm 0.5.4 modificado) ---
class ConvStem(nn.Module):
    """Patch embedding convolucional do CTransPath (github.com/Xiyue-Wang/TransPath),
    na versão compatível com o Swin do timm >= 0.9 (saída NHWC)."""

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=768, norm_layer=None, **kwargs):
        super().__init__()
        from timm.layers.helpers import to_2tuple
        assert patch_size == 4 and embed_dim % 8 == 0
        self.img_size, self.patch_size = to_2tuple(img_size), to_2tuple(patch_size)
        self.grid_size = (self.img_size[0] // 4, self.img_size[1] // 4)
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        stem, d_in, d_out = [], 3, embed_dim // 8
        for _ in range(2):
            stem += [nn.Conv2d(d_in, d_out, 3, 2, 1, bias=False), nn.BatchNorm2d(d_out), nn.ReLU(inplace=True)]
            d_in, d_out = d_out, d_out * 2
        stem.append(nn.Conv2d(d_in, embed_dim, kernel_size=1))
        self.proj = nn.Sequential(*stem)
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        return self.norm(self.proj(x).permute(0, 2, 3, 1).contiguous())


def _swin_old_to_new(sd: Dict) -> Dict:
    """Checkpoint do Swin do timm 0.5.4 -> timm moderno: remove buffers e move o downsample
    do fim do estágio i para o início do estágio i+1 (mesma ordem de concatenação 2x2)."""
    out = {}
    for k, v in sd.items():
        if "relative_position_index" in k or "attn_mask" in k or k.startswith("head"):
            continue
        k = re.sub(r"layers\.(\d+)\.downsample", lambda m: f"layers.{int(m.group(1)) + 1}.downsample", k)
        out[k] = v
    return out


def build_swin_convstem(name: str) -> nn.Module:
    local = local_weight_path(name) if name in LOCAL_WEIGHTS else None
    if name == "ctranspath" and not (local and local.exists()):
        return timm.create_model(HF_IDS["ctranspath"], embed_layer=ConvStem, pretrained=True, num_classes=0)
    path = _require_local(name)
    model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, embed_layer=ConvStem,
                              num_classes=0)
    sd = _load_ckpt(path)
    if any(k.startswith("layers.0.downsample") for k in sd):   # formato antigo (timm 0.5.4)
        sd = _swin_old_to_new(sd)
    load_and_verify(model, sd, name, ignore_missing=("relative_position_index", "attn_mask"))
    return model


# --- BEPH: BEiTv2 ViT-B/16 treinado no mmselfsup -> beitv2_base_patch16_224 do timm ---
def _mm_beit_to_timm(sd: Dict, n_blocks: int) -> Dict:
    if any(k.startswith("backbone.") for k in sd):
        sd = {k[len("backbone."):]: v for k, v in sd.items() if k.startswith("backbone.")}
    sd = _strip_prefixes(sd, ("module.",))
    out, shared = {}, None
    rules = [(r"^patch_embed\.projection\.", "patch_embed.proj."),
             (r"^layers\.(\d+)\.", r"blocks.\1."),
             (r"\.ln1\.", ".norm1."), (r"\.ln2\.", ".norm2."),
             (r"\.ffn\.layers\.0\.0\.", ".mlp.fc1."), (r"\.ffn\.layers\.1\.", ".mlp.fc2."),
             (r"\.gamma1(\.weight)?$", ".gamma_1"), (r"\.ffn\.gamma2(\.weight)?$", ".gamma_2"),
             (r"\.gamma2(\.weight)?$", ".gamma_2"),
             (r"^ln1\.", "norm."), (r"^ln2\.", "fc_norm.")]
    for k, v in sd.items():
        if "relative_position_index" in k or k.startswith(("mask_token", "head", "neck", "target")):
            continue
        if k == "rel_pos_bias.relative_position_bias_table":      # viés relativo compartilhado
            shared = v
            continue
        for pat, rep in rules:
            k = re.sub(pat, rep, k)
        out[k] = v
    if shared is not None:
        for i in range(n_blocks):
            out.setdefault(f"blocks.{i}.attn.relative_position_bias_table", shared)
    return out


def build_beph() -> nn.Module:
    model = timm.create_model("beitv2_base_patch16_224", pretrained=False, num_classes=0,
                              global_pool="token")
    sd = _mm_beit_to_timm(_load_ckpt(_require_local("beph")), len(model.blocks))
    load_and_verify(model, sd, "beph", ignore_missing=("relative_position_index",))
    return model


# --- PathoDuet: ViT-B/16 MoCo v3 (subclasse do VisionTransformer do timm), pool médio ---
def build_pathoduet() -> nn.Module:
    model = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0, global_pool="avg")
    sd = _load_ckpt(_require_local("pathoduet"))
    if any("base_encoder." in k for k in sd):           # checkpoint de treino MoCo: só o encoder online
        sd = {k.split("base_encoder.", 1)[1]: v for k, v in sd.items() if "base_encoder." in k}
    sd = _strip_prefixes(sd, ("module.",))
    sd = {k: v for k, v in sd.items() if not k.startswith(("head", "pretext"))}
    if "fc_norm.weight" not in sd and "norm.weight" in sd:   # pool médio do timm usa fc_norm
        sd["fc_norm.weight"], sd["fc_norm.bias"] = sd.pop("norm.weight"), sd.pop("norm.bias")
    pe = sd.get("pos_embed")
    n_tok = model.pos_embed.shape[1]
    if pe is not None and pe.shape[1] > n_tok:          # token de pretexto extra: mantém CLS + grade
        sd["pos_embed"] = torch.cat([pe[:, :1], pe[:, -(n_tok - 1):]], dim=1)
    load_and_verify(model, sd, "pathoduet")
    return model


# --- Path Foundation (Google): SavedModel do TensorFlow, entrada 224x224 em [0, 1] ---
class TFPathFoundation:
    def __init__(self):
        import tensorflow as tf
        for gpu in tf.config.list_physical_devices("GPU"):
            try:   # não deixa o TF reservar a VRAM inteira (o PyTorch divide a mesma GPU)
                tf.config.experimental.set_memory_growth(gpu, True)
            except RuntimeError:
                pass
        from huggingface_hub import snapshot_download
        path = snapshot_download("google/path-foundation", token=HF_TOKEN)
        self.tf = tf
        self.fn = tf.saved_model.load(path).signatures["serving_default"]
        self.in_name = list(self.fn.structured_input_signature[1].keys())[0]

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        arr = x.float().permute(0, 2, 3, 1).contiguous().cpu().numpy()       # NCHW -> NHWC
        out = self.fn(**{self.in_name: self.tf.constant(arr)})
        out = out["output_0"] if "output_0" in out else next(iter(out.values()))
        return torch.from_numpy(out.numpy()).to(x.device)


def _arquivo_pesos_hf(repo: str) -> str:
    from huggingface_hub import hf_hub_download, list_repo_files
    arquivos = set(list_repo_files(repo, token=HF_TOKEN))
    for nome in ("model.safetensors", "pytorch_model.bin"):
        if nome in arquivos:
            return hf_hub_download(repo, nome, token=HF_TOKEN)
    raise FileNotFoundError(f"{repo}: sem model.safetensors nem pytorch_model.bin")


def _timm_hf_baixa_ram(name: str, hf_id: str, repo: str, **kw) -> nn.Module:
    """Carregamento econômico em RAM: a arquitetura é criada DIRETO NA GPU (sem a cópia de
    pesos aleatórios na CPU) e o checkpoint é lido por mmap (páginas do disco, não RAM) ou,
    se for safetensors, direto para a GPU. Pesos finais idênticos ao pretrained=True."""
    print(f"    [{name}] carregamento econômico | antes: {ram_txt()}")
    arq = _arquivo_pesos_hf(repo)
    try:
        with torch.device(DEVICE):
            model = timm.create_model(hf_id, pretrained=False, **kw)
    except Exception as e:   # noqa: BLE001
        print(f"    [{name}] criação direta na GPU falhou ({type(e).__name__}); criando na CPU e movendo")
        liberar_memoria(verbose=False)
        model = timm.create_model(hf_id, pretrained=False, **kw).to(DEVICE)
    liberar_memoria(verbose=False)
    if arq.endswith(".safetensors"):
        from safetensors.torch import load_file
        sd = load_file(arq, device=str(DEVICE))
    else:
        try:
            sd = torch.load(arq, map_location="cpu", mmap=True, weights_only=True)
        except Exception:   # noqa: BLE001
            sd = torch.load(arq, map_location="cpu", mmap=True, weights_only=False)
    if isinstance(sd, dict) and isinstance(sd.get("model"), dict):
        sd = sd["model"]
    try:   # mesmo filtro que o timm aplica no pretrained=True (no GigaPath não muda nada)
        from timm.models.vision_transformer import checkpoint_filter_fn
        sd = checkpoint_filter_fn(sd, model)
    except Exception:   # noqa: BLE001
        pass
    load_and_verify(model, sd, name, min_frac=0.99)
    del sd
    liberar_memoria(verbose=False)
    print(f"    [{name}] carregado | depois: {ram_txt()}")
    return model


def build_backbone(name: str, cfg: Config) -> Backbone:
    imnet = (IMAGENET_MEAN, IMAGENET_STD)
    ft_root, views_fn, source = None, None, None
    garantir_ram(name)
    if name == "gigapath":
        if name in CARREGAR_DIRETO_NA_GPU:
            model = _timm_hf_baixa_ram(name, HF_IDS[name], HF_REPOS[name])          # 1536-d
        else:
            model = timm.create_model(HF_IDS[name], pretrained=True)
        preprocess = make_preprocess(224, *imnet, resize=256)
        encode_fn, ft_root = model, model
    elif name == "virchow":
        from timm.layers import SwiGLUPacked
        model = timm.create_model(HF_IDS[name], pretrained=True,
                                  mlp_layer=SwiGLUPacked, act_layer=torch.nn.SiLU)
        preprocess = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))

        def encode_fn(x):
            out = model(x)                                           # B x 257 x 1280
            return torch.cat([out[:, 0], out[:, 1:].mean(1)], dim=-1)  # CLS + média = 2560-d
        ft_root = model
    elif name == "prism":
        from transformers import AutoModel
        from timm.layers import SwiGLUPacked
        tile = timm.create_model(HF_IDS["virchow"], pretrained=True,
                                 mlp_layer=SwiGLUPacked, act_layer=torch.nn.SiLU)
        slide = AutoModel.from_pretrained("paige-ai/Prism", trust_remote_code=True, token=HF_TOKEN)
        model = nn.ModuleDict({"tile": tile, "slide": slide})
        preprocess = create_transform(**resolve_data_config(tile.pretrained_cfg, model=tile))

        def tile_emb(x):
            out = tile(x)
            return torch.cat([out[:, 0], out[:, 1:].mean(1)], dim=-1)   # embedding de tile do Virchow

        def encode_fn(x):                                     # cada janela tratada como 1 "lâmina"
            return slide.slide_representations(tile_emb(x).unsqueeze(1))["image_embedding"]

        def views_fn(x, counts):                              # Perceiver sobre as janelas da ROI
            z = tile_emb(x)
            return torch.cat([slide.slide_representations(c.unsqueeze(0))["image_embedding"]
                              for c in torch.split(z, counts.tolist())])
    elif name == "uni":
        model = timm.create_model(HF_IDS[name], pretrained=True, init_values=1e-5, dynamic_img_size=True)
        preprocess = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
        encode_fn, ft_root = model, model
    elif name == "conch":
        from conch.open_clip_custom import create_model_from_pretrained
        model, preprocess = create_model_from_pretrained("conch_ViT-B-16", HF_IDS[name],
                                                         hf_auth_token=HF_TOKEN)

        def encode_fn(x):
            return model.encode_image(x, proj_contrast=False, normalize=False)
        ft_root = getattr(model.visual, "trunk", None)
    elif name == "titan":
        from transformers import AutoModel
        titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True, token=HF_TOKEN)
        model, eval_tf = titan.return_conch()               # encoder de tiles CONCH v1.5 (768-d)
        del titan
        preprocess = _safe_split(eval_tf, 448, *imnet)
        encode_fn, ft_root = model, model
    elif name in ("chief", "ctranspath"):
        model = build_swin_convstem(name)                                           # 768-d
        preprocess = make_preprocess(224, *imnet)
        encode_fn, ft_root = model, model
        if name == "ctranspath" and local_weight_path(name).exists():
            source = "local:" + LOCAL_WEIGHTS[name]
    elif name == "lunit":
        model = timm.create_model(HF_IDS[name], pretrained=True, num_classes=0)    # 384-d (CLS)
        preprocess = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
        encode_fn, ft_root = model, model
    elif name == "plip":
        from transformers import CLIPModel
        model = CLIPModel.from_pretrained("vinid/plip")
        preprocess = make_preprocess(224, CLIP_MEAN, CLIP_STD)

        def encode_fn(x):                                   # = get_image_features (512-d)
            return model.visual_projection(model.vision_model(pixel_values=x).pooler_output)
        ft_root = model.vision_model
    elif name == "beph":
        model = build_beph()
        preprocess = make_preprocess(224, *imnet)

        def encode_fn(x):                                   # média dos tokens de patch (768-d)
            return model.forward_features(x)[:, getattr(model, "num_prefix_tokens", 1):].mean(1)
        ft_root = model
    elif name == "pathoduet":
        model = build_pathoduet()
        preprocess = make_preprocess(224, *NO_NORM)         # o PathoDuet não normaliza a entrada
        encode_fn, ft_root = model, model
    elif name == "path_foundation":
        model = TFPathFoundation()                          # 384-d
        preprocess = make_preprocess(224, *NO_NORM)
        encode_fn = model
    elif name == "hibou_b":
        from transformers import AutoImageProcessor, AutoModel
        model = AutoModel.from_pretrained("histai/hibou-b", trust_remote_code=True, token=HF_TOKEN)
        proc = AutoImageProcessor.from_pretrained("histai/hibou-b", trust_remote_code=True, token=HF_TOKEN)
        preprocess = make_preprocess(224, tuple(proc.image_mean), tuple(proc.image_std))

        def encode_fn(x):
            return model(pixel_values=x).pooler_output                               # 768-d
        ft_root = model
    elif name == "athena":
        from huggingface_hub import snapshot_download
        local_dir = snapshot_download("PAICON-GmbH/Athena-0", revision="main", token=HF_TOKEN,
                                      allow_patterns=["athena0.py", "model_39999.safetensors", "config.json"])
        if local_dir not in sys.path:
            sys.path.insert(0, local_dir)
        from athena0 import Athena0
        pesos = f"{local_dir}/model_39999.safetensors"
        try:   # ViT-g (1,1 B): cria os tensores direto na GPU para não duplicar ~4,5 GB na RAM
            with torch.device(DEVICE):
                model, tf_ = Athena0.from_pretrained(weights_path=pesos, device=str(DEVICE))
        except Exception as e:   # noqa: BLE001
            print(f"    [athena] carregamento na GPU falhou ({type(e).__name__}: {e}); tentando o padrão")
            liberar_memoria(verbose=False)
            garantir_ram("athena")
            model, tf_ = Athena0.from_pretrained(weights_path=pesos, device=str(DEVICE))
        liberar_memoria(verbose=False)
        preprocess = _safe_split(tf_, 224, *imnet)
        encode_fn, ft_root = model, model                                          # 1536-d (CLS)
    elif name == "imagenet_r50":
        model = timm.create_model(HF_IDS[name], pretrained=True, num_classes=0)  # 2048-d
        preprocess = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
        encode_fn = model
    else:
        raise ValueError(name)
    bb = Backbone(name, model, preprocess, cfg.n_unfrozen_blocks, encode_fn,
                  ft_root=ft_root, views_fn=views_fn, source=source)
    liberar_memoria(f"{name} carregado")
    return bb


def n_janelas(roi_w: int, roi_h: int, view: int, scale: float) -> int:
    """Quantas janelas roi_views gera (mesma conta: altura -> view*scale, preenchimento até a janela)."""
    h = max(1, round(view * scale))
    w = max(1, round(roi_w * h / roi_h))
    return math.ceil(max(h, view) / view) * math.ceil(max(w, view) / view)


def mpp_captura(cfg: Config) -> Optional[float]:
    """µm/pixel da captura: Config.native_mpp, ou a partir da objetiva (magnificacao_captura)."""
    if cfg.native_mpp:
        return float(cfg.native_mpp)
    if cfg.magnificacao_captura:
        o = float(cfg.magnificacao_captura)
        return OBJETIVA_PARA_MPP.get(int(o), 10.0 / o)
    return None


def escala_de_zoom(view: int, zoom: float, roi_h: int) -> float:
    """Zoom físico (1 = pixel da captura) -> escala do roi_views para uma janela de 'view' px."""
    return round(zoom * roi_h / view, 4)


def janela_do_modelo(bb_name: str, cfg: Config) -> int:
    """Janela de entrada sem carregar o modelo: metadado do cache de embeddings ou JANELA_PADRAO."""
    for f in sorted(Path(cfg.embeddings_dir).glob(f"{bb_name}_*.npz")):
        try:
            with np.load(f, allow_pickle=False) as z:
                v = json.loads(str(z["meta"])).get("view_size")
            if v:
                return int(v)
        except Exception:   # noqa: BLE001
            continue
    return JANELA_PADRAO.get(bb_name, 224)


_ROI_WH: Optional[Tuple[int, int]] = None


def roi_wh(cfg: Config) -> Tuple[int, int]:
    global _ROI_WH
    if _ROI_WH is None:
        paths, _, _ = prepare_data(cfg)
        with Image.open(paths[0]) as im:
            _ROI_WH = im.size
    return _ROI_WH


def escalas_magnificacao(bb_name: str, cfg: Config) -> Dict[float, str]:
    """Escalas da fase 4 para ESTE modelo. A grade é definida em ZOOM FÍSICO (Config.mag_zooms,
    igual para todos) e convertida para a janela de cada um: ViTs/CNNs de 224 px, TITAN e CONCH
    de 448 px, PRISM (janelas do Virchow agregadas pelo Perceiver), path-foundation (TF). Entram
    também a referência 1,0 das fases 1–2 (ROI ajustada à altura da janela) e, se a magnificação
    da captura for informada, a escala que iguala o µm/pixel do pré-treino (MODEL_TRAIN_MPP)."""
    _, roi_h = roi_wh(cfg)
    view = janela_do_modelo(bb_name, cfg)
    out = {1.0: f"referência (altura {roi_h}->{view}px, zoom {view / roi_h:.2f}x)"}
    for z in cfg.mag_zooms:
        out.setdefault(escala_de_zoom(view, z, roi_h), f"zoom {z:g}x da captura")
    mpp, alvo = mpp_captura(cfg), MODEL_TRAIN_MPP.get(bb_name)
    if mpp and alvo:
        out.setdefault(escala_de_zoom(view, mpp / alvo, roi_h), f"µm/px do pré-treino ({alvo:g})")
    return dict(sorted(out.items()))


def magnification_report(bb_name: str, view_size: int, scale: float, roi_wh_, native_mpp) -> Dict:
    """Resolução efetiva que o modelo vê (zoom sobre o pixel da captura), nº de janelas, fração
    real de tecido na janela (o resto é reflexão) e comparação com o µm/px do pré-treino."""
    roi_w, roi_h = roi_wh_
    zoom = view_size * scale / roi_h                 # >1: ROI ampliada, <1: reduzida
    h, w = roi_h * zoom, roi_w * zoom
    real = min(1.0, h / view_size) * min(1.0, w / view_size)
    rep = {"backbone": bb_name, "scale": scale, "view_size": view_size, "zoom_captura": round(zoom, 4),
           "resize_factor": zoom, "n_views": n_janelas(roi_w, roi_h, view_size, scale),
           "fracao_real_na_janela": round(real, 3), "native_mpp": native_mpp,
           "effective_mpp": (native_mpp / zoom) if native_mpp else None,
           "pretrain_mpp": MODEL_TRAIN_MPP.get(bb_name)}
    msg = (f"[magnificação] {bb_name} scale={scale:g}: zoom {zoom:.2f}x da captura, ROI {roi_w}x{roi_h} -> "
           f"{round(w)}x{round(h)}, {rep['n_views']} janela(s) de {view_size}px")
    if real < 1:
        msg += f" ({real:.0%} de tecido real; resto por reflexão)"
    if native_mpp and rep["pretrain_mpp"]:
        msg += (f" | {rep['effective_mpp']:.3f} µm/px vs pré-treino ~{rep['pretrain_mpp']} µm/px "
                f"(razão {rep['pretrain_mpp'] / rep['effective_mpp']:.2f}x)")
    print(msg)
    return rep


class CoralHead(nn.Module):
    """CORAL: um vetor de pesos compartilhado + K-1 vieses (rank-consistente)."""

    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1, bias=False)
        self.bias = nn.Parameter(torch.zeros(num_classes - 1))

    def forward(self, z):
        return self.fc(z) + self.bias


def build_head(loss_name: str, in_dim: int, num_classes: int) -> nn.Module:
    if loss_name == "coral":
        return CoralHead(in_dim, num_classes)
    if loss_name == "corn":
        return nn.Linear(in_dim, num_classes - 1)
    if loss_name == "ce":
        return nn.Linear(in_dim, num_classes)
    raise ValueError(loss_name)


class LinearProbeModel(nn.Module):
    """Backbone parcialmente destravado + cabeça linear sobre o embedding médio das janelas."""

    def __init__(self, backbone: Backbone, loss_name: str, num_classes: int):
        super().__init__()
        self.backbone = backbone
        self.head = build_head(loss_name, backbone.embed_dim, num_classes)

    def forward(self, x, counts):
        return self.head(self.backbone.encode_views(x, counts))

    def train(self, mode: bool = True):
        super().train(mode)
        self.backbone.eval()
        for m in self.backbone.modules():
            if isinstance(m, LoRALinear):
                m.dropout_on = mode
        return self


def build_optimizer(model: LinearProbeModel, cfg: Config, hp: Dict, lr_variant: str, mode: str = "partial"):
    if mode == "bitfit":   # protocolo único (PEFT_PROTOCOL): Adam; só cabeça + vieses
        return torch.optim.Adam(list(model.head.parameters()) + list(model.backbone._bitfit_params()),
                                lr=PEFT_PROTOCOL["lr"])
    if mode == "lora":
        return torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=PEFT_PROTOCOL["lr"])
    return torch.optim.AdamW([
        {"params": list(model.head.parameters()), "lr": hp["lr_head"][lr_variant],
         "weight_decay": cfg.weight_decay},
        {"params": list(model.backbone._ft_params), "lr": hp["lr_backbone_partial"][lr_variant],
         "weight_decay": cfg.weight_decay},
    ])


# --- Batch size dinâmico + fallback de OOM ---
def vram_factor() -> int:
    """Multiplicador (potência de 2) da tabela calibrada para 16 GB: 1x no T4/L4, 2x no
    A100-40GB, 4x no A100-80GB/H100. Nunca reduz abaixo de 1x; o fallback de OOM cuida disso."""
    gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return int(min(4, 2 ** max(0, math.floor(math.log2(gb / 14.0)))))


def resolve_batch_size(bb_name: str, role: str, cfg: Config) -> Optional[int]:
    """Lote (em ROIs) de 'extract' ou 'partial' para o backbone, escalonado pela VRAM."""
    base = BATCH_SIZE_T4.get(bb_name, {}).get(role)
    if base is None:
        return None
    return int(base * (vram_factor() if cfg.vram_scaling else 1))


def partial_batch_plan(bb_name: str, cfg: Config, batch_cfg) -> Dict:
    """Lote físico, acumulação e batch efetivo do regime partial.
    batch_cfg=None -> automático (tabela/VRAM, efetivo = cfg.effective_batch);
    batch_cfg=(lote, efetivo) -> fixo (estudo de batch size)."""
    if batch_cfg is None:
        eff = cfg.effective_batch
        bs = min(resolve_batch_size(bb_name, "partial", cfg), eff)
    else:
        bs, eff = int(batch_cfg[0]), int(batch_cfg[1])
        bs = min(bs, eff)
    return {"batch_size": bs, "effective_target": eff, "accum_steps": max(1, math.ceil(eff / bs))}


def batch_cfg_tag(batch_cfg) -> str:
    return "auto" if batch_cfg is None else f"bs{batch_cfg[0]}_eff{batch_cfg[1]}"


def run_with_oom_fallback(fn, batch_size: int, cfg: Config, label: str):
    """Executa fn(batch_size); em CUDA OOM libera a memória, divide o lote por 2 e tenta de novo.
    Devolve (resultado, lote_usado, nº_de_retentativas)."""
    retries = 0
    while True:
        oom = False
        try:
            return fn(batch_size), batch_size, retries
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:   # fora do 'except' para que os frames com tensores sejam liberados
            gc.collect()
            torch.cuda.empty_cache()
            if batch_size <= 1 or retries >= cfg.max_oom_retries:
                raise RuntimeError(f"[{label}] OOM mesmo com lote {batch_size}")
            retries += 1
            print(f"    [OOM] {label}: lote {batch_size} -> {max(1, batch_size // 2)}")
            batch_size = max(1, batch_size // 2)

# %% [CÉLULA 5] Perdas, sonda linear L2 (L-BFGS) e métricas
from scipy.optimize import minimize  # noqa: E402
from scipy.special import expit, logsumexp  # noqa: E402
from scipy.stats import rankdata  # noqa: E402
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score,  # noqa: E402
                             confusion_matrix, f1_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler  # noqa: E402


# --- Perdas em torch (regime partial) ---
def coral_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    levels = (y.unsqueeze(1) > torch.arange(NUM_CLASSES - 1, device=y.device)).float()
    ls = F.logsigmoid(logits)
    return -(ls * levels + (ls - logits) * (1 - levels)).sum(dim=1).mean()


def corn_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """CORN: tarefa k = P(y > k | y >= k), treinada só nas amostras com y >= k."""
    total, n = logits.new_zeros(()), 0
    for k in range(NUM_CLASSES - 1):
        mask = y >= k
        if not mask.any():
            continue
        target = (y[mask] > k).float()
        total = total + F.binary_cross_entropy_with_logits(logits[mask, k], target, reduction="sum")
        n += int(mask.sum())
    return total / n


def make_criterion(name: str, label_smoothing: float):
    if name == "coral":
        return coral_loss
    if name == "corn":
        return corn_loss
    if name == "ce":
        return nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    raise ValueError(name)


# --- Sonda linear L2 em numpy/scipy (regimes frozen): convexa, determinística, converge ---
def _log_sigmoid(z):
    return -np.logaddexp(0.0, -z)


def _probe_shapes(loss: str, D: int):
    K = NUM_CLASSES
    return {"ce": ((D, K), K), "corn": ((D, K - 1), K - 1), "coral": ((D, 1), K - 1)}[loss]


def _unpack(theta, loss, D):
    wshape, nb = _probe_shapes(loss, D)
    nw = wshape[0] * wshape[1]
    return theta[:nw].reshape(wshape), theta[nw:nw + nb]


def probe_logits(X, W, b):
    return X @ W + b   # CORAL: (n,1) + (K-1,) -> (n,K-1), peso compartilhado


def _probe_objective(theta, X, y, loss, lam):
    """Média da loss + (lam/2)*||W||^2 (viés sem penalidade), com gradiente analítico.
    lam = 1/(C*n) reproduz a convenção de C do scikit-learn."""
    n, D = X.shape
    W, b = _unpack(theta, loss, D)
    Z = probe_logits(X, W, b)
    ks = np.arange(NUM_CLASSES - 1)
    if loss == "ce":
        logP = Z - logsumexp(Z, axis=1, keepdims=True)
        nll = -logP[np.arange(n), y].mean()
        dZ = np.exp(logP)
        dZ[np.arange(n), y] -= 1.0
        dZ /= n
    elif loss == "coral":
        L = (y[:, None] > ks).astype(np.float64)
        nll = -(L * _log_sigmoid(Z) + (1 - L) * _log_sigmoid(-Z)).sum(1).mean()
        dZ = (expit(Z) - L) / n
    elif loss == "corn":
        M = (y[:, None] >= ks).astype(np.float64)
        Tt = (y[:, None] > ks).astype(np.float64)
        N = M.sum()
        nll = -(M * (Tt * _log_sigmoid(Z) + (1 - Tt) * _log_sigmoid(-Z))).sum() / N
        dZ = M * (expit(Z) - Tt) / N
    else:
        raise ValueError(loss)
    gW = X.T @ (dZ.sum(1, keepdims=True) if loss == "coral" else dZ)
    gb = dZ.sum(0)
    f = nll + 0.5 * lam * np.sum(W * W)
    return f, np.concatenate([(gW + lam * W).ravel(), gb])


def fit_probe(X, y, loss: str, C: float, max_iter: int, theta0=None):
    n, D = X.shape
    wshape, nb = _probe_shapes(loss, D)
    if theta0 is None:
        theta0 = np.zeros(wshape[0] * wshape[1] + nb)
    res = minimize(_probe_objective, theta0, args=(X, y, loss, 1.0 / (C * n)), jac=True,
                   method="L-BFGS-B", options={"maxiter": max_iter, "maxfun": 2 * max_iter})
    if not res.success:
        print(f"    [aviso] L-BFGS não convergiu ({loss}, C={C:g}): {res.message}")
    return res.x, int(res.nit)


def np_probs_preds(logits: np.ndarray, loss_name: str):
    """Logits -> probabilidades por classe (n x K) e predição (ordinal para CORAL/CORN)."""
    logits = np.asarray(logits, dtype=np.float64)
    if loss_name == "ce":
        P = np.exp(logits - logsumexp(logits, axis=1, keepdims=True))
        return P, P.argmax(1)
    S = expit(logits)
    cum = S if loss_name == "coral" else np.cumprod(S, axis=1)   # P(y > k)
    n = len(cum)
    P = np.clip(np.hstack([np.ones((n, 1)), cum]) - np.hstack([cum, np.zeros((n, 1))]), 0, None)
    P = P / np.clip(P.sum(1, keepdims=True), 1e-12, None)
    return P, (cum > 0.5).sum(1)


def compute_metrics(y_true: np.ndarray, probs: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    probs = np.asarray(probs, dtype=np.float64)
    probs = probs / probs.sum(1, keepdims=True)
    labels = list(range(NUM_CLASSES))
    try:
        auroc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro", labels=labels)
    except ValueError:   # reamostragem sem alguma classe
        auroc = float("nan")
    return {
        "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=labels),
        "f1_macro": f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0),
        "bacc": balanced_accuracy_score(y_true, y_pred),
        "acc": accuracy_score(y_true, y_pred),
        "auroc": auroc,
        "mod": float(np.abs(np.asarray(y_true) - np.asarray(y_pred)).mean()),  # Mean Ordinal Distance
    }


METRIC_KEYS = ("qwk", "f1_macro", "bacc", "acc", "auroc", "mod")


def _safe(v: float) -> float:
    return v if np.isfinite(v) else -2.0   # QWK indefinido (predição constante) perde a seleção


def _sel(vm: Dict) -> tuple:
    """Pontuação de seleção pela validação: (F1 macro, QWK). Tuplas comparam em ordem, então o
    QWK só desempata. Comparação estrita: no empate fica a opção mais regularizada/anterior."""
    return (_safe(vm[SELECTION_METRIC]), _safe(vm["qwk"]))



def fast_metrics(y_true: np.ndarray, probs: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """MESMAS métricas de compute_metrics, calculadas direto da matriz de confusão e de postos
    (Mann-Whitney) — ~20x mais rápido, usado só no bootstrap. A equivalência com o sklearn é
    verificada em run_sanity_checks()."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    P = np.asarray(probs, dtype=np.float64)
    P = P / P.sum(1, keepdims=True)
    K, n = NUM_CLASSES, len(y_true)
    cm = np.bincount(y_true * K + y_pred, minlength=K * K).reshape(K, K).astype(np.float64)
    tp, true_n, pred_n = np.diag(cm), cm.sum(1), cm.sum(0)
    w = (np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2
    den = (w * np.outer(true_n, pred_n) / n).sum()
    qwk = 1 - (w * cm).sum() / den if den > 0 else float("nan")
    f1d = 2 * tp + (pred_n - tp) + (true_n - tp)
    f1 = np.where(f1d > 0, 2 * tp / np.where(f1d > 0, f1d, 1), 0.0).mean()
    present = true_n > 0
    bacc = (tp[present] / true_n[present]).mean()
    aucs = []
    for c in range(K):
        pos = y_true == c
        n_pos = int(pos.sum())
        if n_pos == 0 or n_pos == n:
            aucs.append(float("nan"))
            continue
        r = rankdata(P[:, c])
        aucs.append((r[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * (n - n_pos)))
    return {"qwk": float(qwk), "f1_macro": float(f1), "bacc": float(bacc),
            "acc": float(tp.sum() / n), "auroc": float(np.mean(aucs)),
            "mod": float(np.abs(y_true - y_pred).mean())}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raiz do dataset: /content/Original_ROI_images/Original ROI images
Config(drive_dataset_dir='/content/Original_ROI_images/Original ROI images', local_dataset_dir='/content/dataset', drive_results_dir='/content/drive/MyDrive/resultados_histopatologia', embeddings_dir='/content/drive/MyDrive/resultados_histopatologia/embeddings_cache', run_name=None, resume=True, seeds=[0, 1], partial_seeds=None, backbone=['titan', 'gigapath', 'prism', 'chief', 'ctranspath', 'lunit', 'plip', 'virchow', 'uni', 'conch', 'fusecpath', 'beph', 'pathoduet', 'path_foundation', 'hibou_b', 'athena', 'imagenet_r50', 'color'], regime=['all'], loss=['all'], lr_variant=['mid'], train_frac=0.7, val_frac=0.2, test_frac=0.1, split_by_group=True, subset_groups_per_class=None, scales=[1.0], native_mpp=None, magnificacao_captura=None, mag_zooms=[0.5, 0.75, 1.0, 1.5, 2.0], peft_scale=1.0, estratifi

In [7]:

# %% [CÉLULA 6] Cache de embeddings (com hash) e sonda linear com busca de C
def _hash(obj) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]


def embedding_meta(bb: Backbone, paths: List[str], scale: float, aug_copy: Optional[int]) -> Dict:
    """Tudo que altera os embeddings entra no hash: mudou algo, o cache é recalculado.
    (Mantido idêntico à versão anterior para reaproveitar o cache sem augmentation já salvo.)"""
    norm = bb.normalize.transforms[-1]
    meta = {
        "backbone": bb.name, "hf_id": bb.source,
        "hf_sha": (ENV_INFO["local_weights"].get(bb.name) if bb.source.startswith("local:")
                   else ENV_INFO["hf_sha"].get(bb.name)),
        "view_size": bb.view_size, "scale": scale, "pooling": "mean_das_janelas",
        "norm_mean": [float(m) for m in norm.mean], "norm_std": [float(s) for s in norm.std],
        "aug_copy": aug_copy, "aug": AUG_DESC if aug_copy is not None else None,
        "timm": ENV_INFO["packages"].get("timm"), "torch": ENV_INFO["packages"].get("torch"),
        "amp": str(AMP_DTYPE), "paths_sha": _hash(paths),
    }
    if aug_copy is not None:
        meta["aug_rng"] = "semente_por_roi"   # cópias aumentadas não dependem do batch size
    return meta


@torch.no_grad()
def _extract(bb: Backbone, paths: List[str], cfg: Config, scale: float, aug_copy: Optional[int],
             batch_size: int) -> np.ndarray:
    aug = build_train_aug() if aug_copy is not None else None
    seed = 10_000 + (aug_copy or 0)
    ds = ROIDataset(paths, np.zeros(len(paths), dtype=np.int64), bb.view_size, bb.normalize, scale,
                    aug, item_seed=seed if aug is not None else None)
    loader = make_loader(ds, batch_size, shuffle=False, seed=seed, cfg=cfg)
    feats = []
    for x, counts, _, _ in loader:
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=cfg.use_amp):
            e = bb.encode_views(x.to(DEVICE, non_blocking=True), counts)
        feats.append(e.float().cpu().numpy())
    return np.concatenate(feats)


def extract_embeddings(bb: Backbone, paths: List[str], cfg: Config, scale: float = 1.0,
                       aug_copy: Optional[int] = None) -> Tuple[np.ndarray, Optional[int]]:
    """Extrai 1x os embeddings (média das janelas) de TODAS as ROIs. aug_copy=k gera a k-ésima
    cópia aumentada (reprodutível). O encoder congelado não depende do split, então o cache
    vale para todas as sementes. Devolve (X, batch size usado na extração)."""
    meta = embedding_meta(bb, paths, scale, aug_copy)
    cache = Path(cfg.embeddings_dir) / f"{bb.name}_{_hash(meta)}.npz"
    cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        z = np.load(cache, allow_pickle=False)
        if np.array_equal(z["paths"], np.array(paths)):
            bs = int(z["batch_size"]) if "batch_size" in z.files else None
            print(f"[embeddings] cache reutilizado: {cache.name} {z['X'].shape} (lote={bs})")
            return z["X"], bs
    bs0 = resolve_batch_size(bb.name, "extract", cfg)
    with Image.open(paths[0]) as im:
        nv = n_janelas(im.size[0], im.size[1], bb.view_size, scale)
    if bs0 and nv > 2:              # a tabela de lotes supõe 2 janelas/ROI (escala 1,0)
        bs0 = max(1, int(bs0 * 2 / nv))
    t0 = time.time()
    X, bs, _ = run_with_oom_fallback(lambda b: _extract(bb, paths, cfg, scale, aug_copy, b),
                                     bs0, cfg, f"extração {bb.name}")
    np.savez(cache, X=X, paths=np.array(paths), meta=json.dumps(meta), batch_size=bs)
    print(f"[embeddings] {bb.name} scale={scale:g} aug_copy={aug_copy}: {X.shape} lote={bs} "
          f"({time.time() - t0:.0f}s) -> {cache.name}")
    return X, bs


def run_frozen_probe(X: np.ndarray, y: np.ndarray, sp: Dict[str, np.ndarray], loss: str,
                     cfg: Config, X_aug: Optional[np.ndarray] = None) -> Dict:
    """1) busca C na validação (F1 macro, desempate por QWK; empate -> maior regularização), em ordem crescente de C
    com warm start, parando após c_patience valores sem melhora (os C altos são os mais lentos
    e, com n << D, sobreajustam; o platô inicial de predição constante não conta); 2) reajusta com o C escolhido em treino+validação;
    3) avalia o teste uma única vez. Com X_aug (cópias x N x D), as linhas de ajuste incluem
    as cópias aumentadas. Devolve também a curva C -> métricas de validação."""

    def rows(idx):
        Xs, ys = [X[idx]], [y[idx]]
        if X_aug is not None:
            for Xa in X_aug:
                Xs.append(Xa[idx])
                ys.append(y[idx])
        return np.concatenate(Xs), np.concatenate(ys)

    Xf, yf = rows(sp["train"])
    sc = StandardScaler().fit(Xf)
    Xf, Xv = sc.transform(Xf), sc.transform(X[sp["val"]])
    yv = y[sp["val"]]
    best, theta, since_best, curve, total_iter = None, None, 0, [], 0
    for C in sorted(cfg.c_grid):
        theta, nit = fit_probe(Xf, yf, loss, C, cfg.probe_max_iter, theta)
        total_iter += nit
        W, b = _unpack(theta, loss, Xf.shape[1])
        pv, dv = np_probs_preds(probe_logits(Xv, W, b), loss)
        vm = compute_metrics(yv, pv, dv)
        curve.append({"C": C, "lbfgs_iter": nit, **{f"val_{k}": v for k, v in vm.items()}})
        q = _sel(vm)
        if best is None or q > best["q"]:
            best, since_best = {"q": q, "C": C, "val": (yv, pv, dv)}, 0
        elif len(np.unique(dv)) > 1:   # platô de predição constante (C muito baixo) não conta
            since_best += 1
            if cfg.c_patience and since_best >= cfg.c_patience:
                break

    tv = np.concatenate([sp["train"], sp["val"]])
    Xr, yr = rows(tv)
    sc2 = StandardScaler().fit(Xr)
    theta, nit = fit_probe(sc2.transform(Xr), yr, loss, best["C"], cfg.probe_max_iter)
    W, b = _unpack(theta, loss, Xr.shape[1])
    pt, dt = np_probs_preds(probe_logits(sc2.transform(X[sp["test"]]), W, b), loss)
    return {"C": best["C"], "val": best["val"], "test": (y[sp["test"]], pt, dt),
            "curve": curve, "lbfgs_iter": total_iter + nit}

# --- Linhas de ajuste (com cópias aumentadas, se houver) ---
def _stack_rows(X: np.ndarray, y: np.ndarray, idx: np.ndarray, X_aug: Optional[np.ndarray]):
    """Linhas de ajuste: as ROIs de idx e, se houver, as cópias aumentadas das MESMAS ROIs."""
    Xs, ys = [X[idx]], [y[idx]]
    if X_aug is not None:
        for Xa in X_aug:
            Xs.append(Xa[idx])
            ys.append(y[idx])
    return np.concatenate(Xs), np.concatenate(ys)


# --- Outros classificadores sobre os embeddings congelados (regimes frozen_clf / frozen_clf_aug) ---
def _cl2n(Xfit: np.ndarray, *Xs: np.ndarray):
    """SimpleShot 'CL2N': subtrai a média do treino e normaliza cada vetor para norma 1."""
    mu = Xfit.mean(0, keepdims=True)

    def f(A):
        A = A - mu
        return A / np.clip(np.linalg.norm(A, axis=1, keepdims=True), 1e-12, None)
    return [f(A) for A in (Xfit,) + Xs]


def _clf_grid(name: str, n_train: int) -> List[Dict]:
    if name == "knn":
        return [{"k": k} for k in (40, 20, 10, 5) if k <= n_train]   # k maior = mais suave
    if name in ("simpleshot", "lda"):
        return [{}]
    if name == "svm":
        return [{"C": C} for C in (0.01, 0.1, 1.0, 10.0, 100.0)]
    if name == "mlp":
        return [{"alpha": a} for a in (1e-1, 1e-2, 1e-3, 1e-4)]
    raise ValueError(name)


def _full_proba(P: np.ndarray, classes) -> np.ndarray:
    out = np.zeros((len(P), NUM_CLASSES))
    out[:, np.asarray(classes, dtype=int)] = P
    return out / np.clip(out.sum(1, keepdims=True), 1e-12, None)


def _clf_fit(name: str, params: Dict, Xtr: np.ndarray, ytr: np.ndarray, seed: int):
    """Ajusta no treino e devolve f(Xev) -> (probabilidades n x K, predições)."""
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.svm import SVC
    if name in ("knn", "simpleshot"):
        mu = Xtr.mean(0, keepdims=True)

        def cl2n(A):   # SimpleShot 'CL2N': centra pela média do treino e normaliza (norma 1)
            A = A - mu
            return A / np.clip(np.linalg.norm(A, axis=1, keepdims=True), 1e-12, None)
        Ztr = cl2n(Xtr)
        if name == "knn":
            m = KNeighborsClassifier(n_neighbors=params["k"], weights="distance").fit(Ztr, ytr)

            def f(Xe):
                P = _full_proba(m.predict_proba(cl2n(Xe)), m.classes_)
                return P, P.argmax(1)
            return f
        cls = np.unique(ytr)
        cent = np.stack([Ztr[ytr == c].mean(0) for c in cls])

        def f(Xe):
            Z = cl2n(Xe)
            d2 = ((Z[:, None, :] - cent[None]) ** 2).sum(-1)
            P = _full_proba(np.exp(-(d2 - d2.min(1, keepdims=True))), cls)
            return P, P.argmax(1)
        return f
    sc = StandardScaler().fit(Xtr)
    if name == "lda":
        m = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    elif name == "svm":
        m = SVC(kernel="rbf", C=params["C"], gamma="scale", probability=True, random_state=seed)
    elif name == "mlp":
        m = MLPClassifier(hidden_layer_sizes=(256,), alpha=params["alpha"], max_iter=1000, random_state=seed)
    else:
        raise ValueError(name)
    m.fit(sc.transform(Xtr), ytr)

    def f(Xe):
        Z = sc.transform(Xe)
        P = _full_proba(m.predict_proba(Z), m.classes_)
        # SVM: a probabilidade vem de uma calibração (Platt) com validação cruzada interna, que vê
        # cópias aumentadas da mesma ROI em dobras diferentes; a predição usa a decisão do SVM.
        pred = m.predict(Z) if name == "svm" else P.argmax(1)
        return P, np.asarray(pred, dtype=int)
    return f


def _clf_fit_predict(name: str, params: Dict, Xtr: np.ndarray, ytr: np.ndarray, Xev: np.ndarray, seed: int):
    """Ajusta no treino e devolve (probabilidades n x K, predições) para Xev."""
    return _clf_fit(name, params, Xtr, ytr, seed)(Xev)


def run_frozen_classifier(X: np.ndarray, y: np.ndarray, sp: Dict[str, np.ndarray], name: str, cfg: Config,
                          seed: int, X_aug: Optional[np.ndarray] = None) -> Dict:
    """Mesmo protocolo da sonda L-BFGS: 1) escolhe o hiperparâmetro pelo QWK de validação
    (empate -> o mais regularizado, primeiro da grade); 2) reajusta em treino+validação;
    3) avalia o teste uma única vez."""
    Xtr, ytr = _stack_rows(X, y, sp["train"], X_aug)
    yv = y[sp["val"]]
    best, curve = None, []
    for params in _clf_grid(name, len(ytr)):
        pv, dv = _clf_fit_predict(name, params, Xtr, ytr, X[sp["val"]], seed)
        vm = compute_metrics(yv, pv, dv)
        curve.append({**params, **{f"val_{k}": v for k, v in vm.items()}})
        if best is None or _sel(vm) > best["q"]:
            best = {"q": _sel(vm), "params": params, "val": (yv, pv, dv)}
    Xr, yr = _stack_rows(X, y, np.concatenate([sp["train"], sp["val"]]), X_aug)
    pt, dt = _clf_fit_predict(name, best["params"], Xr, yr, X[sp["test"]], seed)
    return {"params": json.dumps(best["params"]), "val": best["val"], "test": (y[sp["test"]], pt, dt),
            "curve": curve}


def linear_probe_init(X: np.ndarray, y: np.ndarray, sp: Dict[str, np.ndarray], loss: str, cfg: Config):
    """Sonda L-BFGS ajustada SÓ no treino (C pelo QWK de validação), devolvida na escala original
    dos embeddings, para inicializar a cabeça do LoRA/BitFit (LP-FT). Como a validação é usada
    depois no early stopping, aqui não há reajuste em treino+validação."""
    tr = sp["train"]
    sc = StandardScaler().fit(X[tr])
    Xs, Xv, yv = sc.transform(X[tr]), sc.transform(X[sp["val"]]), y[sp["val"]]
    best, theta = None, None
    for C in sorted(cfg.c_grid):
        theta, _ = fit_probe(Xs, y[tr], loss, C, cfg.probe_max_iter, theta)
        W, b = _unpack(theta, loss, Xs.shape[1])
        pv, dv = np_probs_preds(probe_logits(Xv, W, b), loss)
        q = _sel(fast_metrics(yv, pv, dv))
        if best is None or q > best[0]:
            best = (q, W.copy(), b.copy(), C)
    q, W, b, C = best
    W_raw = W / sc.scale_[:, None]                    # z = ((x - mu)/s) W + b
    b_raw = b - (sc.mean_ / sc.scale_) @ W
    print(f"      [LP-FT] cabeça inicial = sonda L-BFGS (C={C:g}, {SEL_NOME} val {q[0]:.4f})")
    return W_raw.astype(np.float32), np.atleast_1d(b_raw).astype(np.float32)



In [8]:

# %% [CÉLULA 7] Regime partial (backbone no grafo, AMP + acumulação + OOM fallback)
import pandas as pd  # noqa: E402


@torch.no_grad()
def predict_partial(model: LinearProbeModel, loader, loss_name: str, cfg: Config, criterion=None):
    """Predições (y, probs, preds) e, se criterion for dado, a loss média no conjunto."""
    model.eval()
    logits_all, y_all, loss_sum = [], [], 0.0
    for x, counts, y, _ in loader:
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=cfg.use_amp):
            logits = model(x.to(DEVICE, non_blocking=True), counts)
        if criterion is not None:
            loss_sum += float(criterion(logits.float(), y.to(DEVICE))) * len(y)
        logits_all.append(logits.float().cpu().numpy())
        y_all.append(y.numpy())
    probs, preds = np_probs_preds(np.concatenate(logits_all), loss_name)
    y_all = np.concatenate(y_all)
    return y_all, probs, preds, (loss_sum / len(y_all) if criterion is not None else float("nan"))


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def regime_mode(regime: str) -> str:
    """'lora_noaug' -> 'lora', 'partial' -> 'partial' etc."""
    return regime.split("_")[0]


def load_head_init(head: nn.Module, W: np.ndarray, b: np.ndarray) -> None:
    """Copia a solução da sonda L-BFGS (já na escala original dos embeddings) para a cabeça."""
    lin = head.fc if isinstance(head, CoralHead) else head
    with torch.no_grad():
        lin.weight.copy_(torch.as_tensor(W.T, dtype=lin.weight.dtype, device=lin.weight.device))
        head.bias.copy_(torch.as_tensor(b, dtype=head.bias.dtype, device=head.bias.device))


def train_partial(bb: Backbone, cfg: Config, seed: int, sp, paths, labels, loss_name: str,
                  hp: Dict, lr_variant: str, use_aug: bool, batch_size: int, accum_steps: int,
                  mode: str = "partial", head_init=None) -> Dict:
    """Treina a cabeça + a parte treinável do encoder (mode: 'partial' = últimos blocos,
    'lora' = adaptadores LoRA, 'bitfit' = vieses). Seleciona a época pelo QWK de validação
    (early stopping) e devolve predições de val/teste, histórico por época, tempos e VRAM."""
    set_global_seed(seed)
    bb.reset_and_configure(mode)
    model = LinearProbeModel(bb, loss_name, NUM_CLASSES).to(DEVICE)
    if head_init is not None:
        load_head_init(model.head, *head_init)
    optimizer = build_optimizer(model, cfg, hp, lr_variant, mode)
    criterion = make_criterion(loss_name, cfg.label_smoothing)

    def mk(idx, aug):
        sc = float(cfg.peft_scale) if mode in ("lora", "bitfit") else 1.0
        return ROIDataset([paths[i] for i in idx], labels[idx], bb.view_size, bb.normalize, sc, aug)

    # avaliação sem gradiente cabe em lotes maiores
    eval_bs = max(batch_size, resolve_batch_size(bb.name, "extract", cfg) or batch_size)
    train_loader = make_loader(mk(sp["train"], build_train_aug() if use_aug else None),
                               batch_size, True, seed, cfg)
    val_loader = make_loader(mk(sp["val"], None), eval_bs, False, seed, cfg)
    test_loader = make_loader(mk(sp["test"], None), eval_bs, False, seed, cfg)

    total_steps = cfg.epochs * math.ceil(len(train_loader) / accum_steps)
    warmup = max(1, int(cfg.warmup_ratio * total_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: (s + 1) / warmup if s < warmup else
                                                  0.5 * (1 + math.cos(math.pi * (s - warmup) /
                                                                      max(1, total_steps - warmup))))
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp and AMP_DTYPE == torch.float16)
    trainable = [p for g in optimizer.param_groups for p in g["params"]]   # o que o otimizador atualiza

    torch.cuda.reset_peak_memory_stats()
    _sync()
    t_start = time.time()
    best_q, best_state, best_epoch, bad_epochs, history = None, None, 0, 0, []
    if head_init is not None:
        # Época 0 = cabeça da sonda, encoder intacto. Entra na seleção: se nenhuma época de
        # adaptação superar a sonda na VALIDAÇÃO, o resultado é o da própria sonda (best_epoch=0).
        yv0, pv0, dv0, vl0 = predict_partial(model, val_loader, loss_name, cfg, criterion)
        vm0 = compute_metrics(yv0, pv0, dv0)
        history.append({"epoch": 0, "train_loss": float("nan"), "val_loss": vl0,
                        **{f"val_{k}": v for k, v in vm0.items()}, "lr_head": 0.0,
                        "epoch_time_s": 0.0, "cum_time_s": 0.0})
        print(f"      ep 00 (sonda) val_loss={vl0:.4f} val_f1={vm0['f1_macro']:.4f} val_qwk={vm0['qwk']:.4f}")
        best_q = _sel(vm0)
        best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    for epoch in range(cfg.epochs):
        t_ep = time.time()
        model.train()
        model.zero_grad(set_to_none=True)
        running, n_seen = 0.0, 0
        for step, (x, counts, y, _) in enumerate(train_loader):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=cfg.use_amp):
                logits = model(x, counts)
            loss = criterion(logits.float(), y)
            scaler.scale(loss / accum_steps).backward()
            running += float(loss) * len(y)
            n_seen += len(y)
            if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                model.zero_grad(set_to_none=True)
                scheduler.step()

        yv, pv, dv, val_loss = predict_partial(model, val_loader, loss_name, cfg, criterion)
        vm = compute_metrics(yv, pv, dv)
        _sync()
        history.append({"epoch": epoch + 1, "train_loss": running / n_seen, "val_loss": val_loss,
                        **{f"val_{k}": v for k, v in vm.items()},
                        "lr_head": optimizer.param_groups[0]["lr"],
                        "epoch_time_s": time.time() - t_ep, "cum_time_s": time.time() - t_start})
        print(f"      ep {epoch + 1:02d} loss={running / n_seen:.4f} val_loss={val_loss:.4f} "
              f"val_f1={vm['f1_macro']:.4f} val_qwk={vm['qwk']:.4f} ({time.time() - t_ep:.1f}s)")
        q = _sel(vm)
        if best_q is None or q > best_q:
            best_q, best_epoch, bad_epochs = q, epoch + 1, 0
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        else:
            bad_epochs += 1
            if bad_epochs >= cfg.patience:
                print(f"      early stopping na época {epoch + 1} (melhor: {best_epoch})")
                break
    train_time = time.time() - t_start

    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in best_state:
                p.copy_(best_state[n].to(p.device))
    val = predict_partial(model, val_loader, loss_name, cfg)[:3]
    test = predict_partial(model, test_loader, loss_name, cfg)[:3]
    peak = torch.cuda.max_memory_allocated() / 1024**3
    del model, optimizer, scaler
    bb.reset_and_configure("frozen")   # devolve o backbone intacto para o próximo cenário
    torch.cuda.empty_cache()
    hist = pd.DataFrame(history)
    return {"val": val, "test": test, "history": hist, "best_epoch": best_epoch,
            "epochs_run": int((hist.epoch > 0).sum()), "train_time_s": train_time,
            "time_to_best_s": float(hist.loc[hist.epoch == best_epoch, "cum_time_s"].iloc[0]),
            "time_per_epoch_s": float(hist.loc[hist.epoch > 0, "epoch_time_s"].mean()), "peak_vram_gb": peak}


def train_partial_dynamic(bb: Backbone, cfg: Config, seed: int, sp, paths, labels, loss_name: str,
                          hp: Dict, lr_variant: str, use_aug: bool, batch_cfg, mode: str = "partial",
                          head_init=None) -> Dict:
    """Resolve o lote (tabela/VRAM ou fixo) e treina; em OOM divide o lote físico por 2 e
    recalcula a acumulação para manter o batch efetivo."""
    plan = partial_batch_plan(bb.name, cfg, batch_cfg)
    eff = plan["effective_target"]

    def attempt(bs):
        accum = max(1, math.ceil(eff / bs))
        out = train_partial(bb, cfg, seed, sp, paths, labels, loss_name, hp, lr_variant, use_aug, bs, accum,
                            mode=mode, head_init=head_init)
        out["accum_steps"] = accum
        return out

    print(f"      lote={plan['batch_size']} ROIs x accum {plan['accum_steps']} "
          f"(efetivo alvo {eff})")
    out, bs, retries = run_with_oom_fallback(attempt, plan["batch_size"], cfg, f"{mode} {bb.name}")
    out.update(batch_size=bs, effective_batch=bs * out["accum_steps"], oom_retries=retries)
    return out


In [9]:

# %% [CÉLULA 8] Bootstrap por imagem, tabelas (display + CSV) e gráficos (show + savefig)
import matplotlib.pyplot as plt  # noqa: E402

try:
    from IPython.display import display  # noqa: E402
except ImportError:  # fora do notebook
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

KEY_COLS = ["backbone", "regime", "loss", "lr_variant", "scale", "batch_cfg"]
PT_NAMES = {"backbone": "Modelo", "regime": "Regime", "loss": "Loss / classificador", "lr_variant": "LR",
            "scale": "Escala", "batch_cfg": "Config. batch"}


def _slug(s: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "-_.|" else "_" for ch in s).replace("|", "__")


def key_tag(key) -> str:
    return _slug("|".join(str(k) for k in key))


def norm_key(key) -> tuple:
    bb, reg, loss, lrv, sc, bc = key
    return (str(bb), str(reg), str(loss), str(lrv), round(float(sc), 6), str(bc))


def save_table(df: pd.DataFrame, path: Path, title: str, show: bool = True) -> None:
    """Exporta imediatamente para o Drive e exibe."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    if show:
        print(f"\n===== {title} =====  ->  {path}")
        display(df)


def show_and_save(fig, path: Path, cfg: Config) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    if cfg.show_plots:
        plt.show()
    plt.close(fig)


# --- Estatística: média entre sementes e IC por bootstrap de imagens de origem ---
def seed_mean_metrics(test_by_seed: Dict[int, tuple], metric_fn=compute_metrics) -> Dict[str, float]:
    per = [metric_fn(y, P, d) for (_, y, P, d) in test_by_seed.values()]
    return {k: float(np.nanmean([m[k] for m in per])) for k in METRIC_KEYS}


def holdout_bootstrap_ci(test_by_seed: Dict[int, tuple], groups: np.ndarray, n_boot: int,
                         level: float, seed: int = 0) -> Dict[str, tuple]:
    """IC percentil da métrica MÉDIA entre sementes. Em cada réplica, o conjunto de teste de
    cada semente é reamostrado por IMAGEM DE ORIGEM (não por ROI: ROIs da mesma imagem não são
    independentes), estratificado pela classe: cada classe mantém seu nº de imagens. Com ~11
    imagens de teste, o bootstrap não estratificado frequentemente sorteia réplicas sem alguma
    classe, o que derruba F1/QWK e desloca o IC para baixo do valor observado.
    Ponto = compute_metrics (sklearn); réplicas = fast_metrics (equivalente, mais rápido)."""
    rng = np.random.default_rng(seed)
    prepared = []
    for idx, y, P, d in test_by_seed.values():
        uniq, inv = np.unique(groups[idx], return_inverse=True)
        members = [np.flatnonzero(inv == g) for g in range(len(uniq))]
        by_class = {}
        for g, m in enumerate(members):
            by_class.setdefault(int(y[m[0]]), []).append(g)
        prepared.append((y, P, d, members, list(by_class.values())))
    point = seed_mean_metrics(test_by_seed)
    boots = {k: np.empty(n_boot) for k in METRIC_KEYS}
    for b in range(n_boot):
        per = []
        for y, P, d, members, strata in prepared:
            pick = [g for st in strata for g in rng.choice(st, size=len(st), replace=True)]
            sel = np.concatenate([members[g] for g in pick])
            per.append(fast_metrics(y[sel], P[sel], d[sel]))
        for k in METRIC_KEYS:
            vals = [m[k] for m in per]
            boots[k][b] = np.nanmean(vals) if np.isfinite(vals).any() else np.nan
    a = (1 - level) / 2 * 100
    return {k: (point[k], float(np.nanpercentile(boots[k], a)), float(np.nanpercentile(boots[k], 100 - a)))
            for k in METRIC_KEYS}


# --- Gráficos ---
def plot_learning_curves(key, histories: Dict[int, pd.DataFrame], best_epochs: Dict[int, int],
                         path: Path, cfg: Config, xlabel: str = "Época") -> None:
    """Regime partial: loss (treino e validação) e métricas de validação por época, uma linha
    por semente; o marcador indica a época escolhida pelo early stopping."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, (seed, h) in enumerate(sorted(histories.items())):
        col = plt.cm.tab10(i % 10)
        axes[0].plot(h.epoch, h.train_loss, "-", color=col, label=f"treino s{seed}")
        axes[0].plot(h.epoch, h.val_loss, "--", color=col, label=f"val s{seed}")
        axes[1].plot(h.epoch, h.val_qwk, "-", color=col, label=f"s{seed}")
        axes[2].plot(h.epoch, h.val_f1_macro, "-", color=col, label=f"s{seed}")
        be = best_epochs.get(seed)
        if be is not None and (h.epoch == be).any():
            axes[1].plot(be, h.loc[h.epoch == be, "val_qwk"].iloc[0], "o", color=col, ms=7)
    for ax, t in zip(axes, ["Loss (— treino, -- validação)", "QWK validação (● = época escolhida)",
                            "F1 macro validação"]):
        ax.set_title(t, fontsize=9)
        ax.set_xlabel(xlabel)
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=6, ncol=2)
    fig.suptitle(" | ".join(map(str, key)), fontsize=10)
    fig.tight_layout()
    show_and_save(fig, path, cfg)


def plot_c_curves(key, curves: Dict[int, pd.DataFrame], chosen_C: Dict[int, float],
                  path: Path, cfg: Config) -> None:
    """Regime frozen (sem épocas): QWK e F1 de validação em função da regularização C."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for i, (seed, c) in enumerate(sorted(curves.items())):
        col = plt.cm.tab10(i % 10)
        axes[0].semilogx(c.C, c.val_qwk, "-o", ms=3, color=col, label=f"s{seed}")
        axes[1].semilogx(c.C, c.val_f1_macro, "-o", ms=3, color=col)
        if seed in chosen_C:
            axes[0].axvline(chosen_C[seed], color=col, ls=":", lw=0.8)
    for ax, t in zip(axes, ["QWK validação vs C (: = C escolhido)", "F1 macro validação vs C"]):
        ax.set_title(t, fontsize=9)
        ax.set_xlabel("C (maior = menos regularização)")
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=6, ncol=2)
    fig.suptitle(" | ".join(map(str, key)), fontsize=10)
    fig.tight_layout()
    show_and_save(fig, path, cfg)


def plot_confusion(key, cm: np.ndarray, n_seeds: int, path: Path, cfg: Config) -> None:
    norm = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    ax.imshow(norm, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45)
    ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{norm[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=7,
                    color="white" if norm[i, j] > 0.6 else "black")
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_title(" | ".join(map(str, key)) + f"\nteste, soma de {n_seeds} semente(s)", fontsize=8)
    fig.tight_layout()
    show_and_save(fig, path, cfg)


def plot_summary(ci_tbl_raw: pd.DataFrame, path: Path, cfg: Config, metric: str = "qwk") -> None:
    """QWK de teste (média entre sementes) com IC por bootstrap, todos os cenários."""
    if ci_tbl_raw.empty:
        return
    d = ci_tbl_raw.sort_values(f"{metric}_pt")
    labels = [" | ".join(str(r[c]) for c in KEY_COLS if not (c == "scale" and r[c] == 1.0)
                         and not (c == "batch_cfg" and r[c] == "auto")) for _, r in d.iterrows()]
    fig, ax = plt.subplots(figsize=(8, 0.32 * len(d) + 1.5))
    err = np.clip(np.nan_to_num(np.vstack([d[f"{metric}_pt"] - d[f"{metric}_lo"],
                                           d[f"{metric}_hi"] - d[f"{metric}_pt"]])), 0, None)
    colors = [plt.cm.tab10(REGIMES.index(r) if r in REGIMES else 9) for r in d.regime]
    ax.barh(range(len(d)), d[f"{metric}_pt"], xerr=err, color=colors, alpha=0.8, capsize=2)
    ax.set_yticks(range(len(d)), labels, fontsize=7)
    ax.set_xlabel(f"{metric.upper()} de teste (média entre sementes, IC {cfg.ci_level:.0%})")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    show_and_save(fig, path, cfg)


def plot_batch_size_analysis(df: pd.DataFrame, logs_dir: Path, out_dir: Path, cfg: Config) -> None:
    """Para cada (modelo, regime partial, loss, LR) rodado com mais de uma configuração de lote:
    curva de QWK de validação por época, desempenho no teste, tempo por época e tempo até a
    melhor época, em função do batch efetivo."""
    part = df[df.regime.str.startswith("partial")]
    for (bb, reg, loss, lrv), g in part.groupby(["backbone", "regime", "loss", "lr_variant"]):
        if g.batch_cfg.nunique() < 2:
            continue
        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        summ = []
        for i, (bc, gb) in enumerate(sorted(g.groupby("batch_cfg"), key=lambda t: t[1].effective_batch.mean())):
            col = plt.cm.viridis(i / max(1, g.batch_cfg.nunique() - 1))
            lab = f"{bc} (lote {int(gb.batch_size.median())}, efetivo {int(gb.effective_batch.median())})"
            hs = []
            for _, r in gb.iterrows():
                f = logs_dir / f"{key_tag((bb, reg, loss, lrv, r.scale, bc))}_seed{int(r.seed)}_history.csv"
                if f.exists():
                    hs.append(pd.read_csv(f).set_index("epoch").val_qwk)
            if hs:
                m = pd.concat(hs, axis=1).mean(1)
                axes[0].plot(m.index, m.values, color=col, label=lab)
            summ.append((lab, gb.effective_batch.mean(), gb.test_qwk.mean(), gb.test_qwk.std(ddof=1),
                         gb.time_per_epoch_s.mean(), gb.time_to_best_s.mean(), gb.best_epoch.mean(), col))
        x = np.arange(len(summ))
        axes[1].bar(x, [s[2] for s in summ], yerr=[0 if pd.isna(s[3]) else s[3] for s in summ],
                    color=[s[7] for s in summ], capsize=3)
        axes[2].bar(x, [s[4] for s in summ], color=[s[7] for s in summ])
        axes[3].bar(x, [s[5] for s in summ], color=[s[7] for s in summ])
        for ax in axes[1:]:
            ax.set_xticks(x, [f"ef {int(s[1])}\nép {s[6]:.1f}" for s in summ], fontsize=7)
        axes[0].set_title("QWK validação por época (média sementes)", fontsize=9)
        axes[0].set_xlabel("Época")
        axes[0].legend(fontsize=6)
        axes[1].set_title("QWK teste (média ± DP)", fontsize=9)
        axes[2].set_title("Tempo por época (s)", fontsize=9)
        axes[3].set_title("Tempo até a melhor época (s)", fontsize=9)
        for ax in axes:
            ax.grid(alpha=0.3)
        fig.suptitle(f"Impacto do batch size | {bb} | {reg} | {loss} | lr {lrv}", fontsize=10)
        fig.tight_layout()
        show_and_save(fig, out_dir / f"batch_size_{_slug(f'{bb}_{reg}_{loss}_{lrv}')}.png", cfg)


# --- Tabelas consolidadas ---
def _pm(g, col):
    m, s = g[col].mean(), g[col].std(ddof=1)
    return m.map(lambda v: "n/a" if pd.isna(v) else f"{v:.4f}") + " ± " + \
        s.map(lambda v: "n/a" if pd.isna(v) else f"{v:.4f}")


def summary_tables(df: pd.DataFrame, pred_dir: Path, groups: np.ndarray, cfg: Config):
    """Tabela A: média ± DP entre sementes com metadados (batch, épocas, tempos, VRAM).
    Tabela B: média entre sementes com IC por bootstrap de imagens."""
    g = df.groupby(KEY_COLS, sort=False, dropna=False)
    meta = g.agg(n_sementes=("seed", "nunique"), batch_size=("batch_size", "median"),
                 accum_steps=("accum_steps", "median"), batch_efetivo=("effective_batch", "median"),
                 melhor_epoca=("best_epoch", "mean"), epocas_rodadas=("epochs_run", "mean"),
                 tempo_por_epoca_s=("time_per_epoch_s", "mean"), tempo_ate_melhor_s=("time_to_best_s", "mean"),
                 tempo_total_s=("train_time_s", "mean"), vram_pico_gb=("peak_vram_gb", "max"),
                 C_mediano=("C", "median")).round(3)
    tbl_a = pd.concat([meta, pd.DataFrame({"F1 macro": _pm(g, "test_f1_macro"), "QWK": _pm(g, "test_qwk"),
                                           "Bal. acc": _pm(g, "test_bacc"), "Acc": _pm(g, "test_acc"),
                                           "AUROC": _pm(g, "test_auroc"), "MOD": _pm(g, "test_mod"),
                                           "F1 val": _pm(g, "val_f1_macro"),
                                           "QWK val": _pm(g, "val_qwk")})], axis=1).reset_index()
    tbl_a = tbl_a.rename(columns=PT_NAMES)

    ci_rows, raw_rows = [], []
    for key, gk in g:
        tbs = {}
        arquivos = {int(s): pred_dir / f"{key_tag(key)}_seed{int(s)}.csv" for s in sorted(gk.seed.unique())}
        if not all(f.exists() for f in arquivos.values()):
            print(f"    [IC] {key_tag(key)}: predições ausentes; IC por bootstrap pulado")
            continue
        for s, f in arquivos.items():
            p = pd.read_csv(f)
            tbs[int(s)] = (p.idx.to_numpy(), p.y_true.to_numpy(), p[[f"p_{c}" for c in CLASS_NAMES]].to_numpy(),
                           p.y_pred.to_numpy())
        ci = holdout_bootstrap_ci(tbs, groups, cfg.n_boot, cfg.ci_level)
        base = dict(zip(KEY_COLS, key))
        ci_rows.append({**{PT_NAMES[k]: v for k, v in base.items()}, "n_sementes": len(tbs),
                        "batch_size": gk.batch_size.median(), "batch_efetivo": gk.effective_batch.median(),
                        "melhor_epoca": round(gk.best_epoch.mean(), 1) if gk.best_epoch.notna().any() else np.nan,
                        **{k: f"{v[0]:.4f} [{v[1]:.4f}, {v[2]:.4f}]" for k, v in ci.items()}})
        raw_rows.append({**base, **{f"{k}_{s}": v[i] for k, v in ci.items()
                                    for i, s in enumerate(("pt", "lo", "hi"))}})
    return tbl_a, pd.DataFrame(ci_rows), pd.DataFrame(raw_rows)


In [10]:

# %% [CÉLULA 9] Orquestrador (com retomada) e consolidação dos resultados
def fused_embeddings(paths: List[str], cfg: Config) -> np.ndarray:
    """Aproximação do FuseCPath para ROIs: concatena os embeddings (escala 1.0, sem augmentation)
    dos modelos em FUSECPATH_COMPONENTS. Cada componente reaproveita o cache de embeddings do
    próprio modelo; por isso convém rodar a fusão DEPOIS dos componentes. A padronização
    (StandardScaler) da sonda linear é feita sobre o vetor concatenado."""
    blocks = []
    for comp in FUSECPATH_COMPONENTS:
        X = carregar_embeddings_do_cache(comp, paths, cfg, 1.0)   # sem carregar o modelo (RAM)
        if X is None:
            bb = build_backbone(comp, cfg)
            X, _ = extract_embeddings(bb, paths, cfg, scale=1.0)
            del bb
            liberar_memoria(f"fusecpath: {comp} descarregado")
        else:
            print(f"[fusecpath] {comp}: embeddings do cache (modelo não carregado)")
        print(f"[fusecpath] componente {comp}: {X.shape[1]}-d")
        blocks.append(X)
    X = np.concatenate(blocks, axis=1)
    print(f"[fusecpath] embedding fundido: {X.shape}")
    return X


def build_scenarios(bb_name: str, cfg: Config, regimes, losses, lr_variants):
    """Lista de chaves (backbone, regime, loss/classificador, lr_variant, scale, batch_cfg) + lote.
    Nos regimes frozen_clf* a coluna 'loss' guarda o nome do classificador."""
    out = []
    for regime in regimes:
        if regime.startswith(TRAINED_REGIMES) and bb_name in FROZEN_ONLY:
            continue
        if regime.endswith("_aug") and regime.startswith("frozen") and (
                bb_name in ("color", "fusecpath") or cfg.frozen_aug_copies <= 0):
            continue
        scales = [1.0] if bb_name == "fusecpath" else cfg.scales   # fusão só na escala nativa
        if regime.startswith("frozen_clf"):   # magnificação também nos classificadores (sem aug)
            esc = scales if regime == "frozen_clf" else [1.0]
            out += [((bb_name, regime, c, "hp_busca", s, "auto"), None) for c in cfg.classifiers for s in esc]
            continue
        for loss in losses:
            if regime == "frozen":
                out += [((bb_name, regime, loss, "C_busca", s, "auto"), None) for s in scales]
            elif regime == "frozen_aug":
                out.append(((bb_name, regime, loss, "C_busca", 1.0, "auto"), None))
            elif regime.startswith(("lora", "bitfit")):
                out += [((bb_name, regime, loss, "protocolo", round(float(cfg.peft_scale), 4),
                          batch_cfg_tag(bc)), bc) for bc in cfg.partial_batch_configs]
            else:
                out += [((bb_name, regime, loss, lrv, 1.0, batch_cfg_tag(bc)), bc)
                        for lrv in lr_variants for bc in cfg.partial_batch_configs]
    return [(norm_key(k), bc) for k, bc in out]


def prepare_data(cfg: Config):
    local_root = copy_dataset_to_local(cfg.drive_dataset_dir, cfg.local_dataset_dir)
    paths, labels = scan_dataset(local_root)
    groups = np.array([group_id(p) for p in paths])
    auditar_dataset(paths, labels, groups)
    if cfg.subset_groups_per_class:
        idx = subset_by_groups(labels, groups, cfg.subset_groups_per_class)
        paths, labels, groups = [paths[i] for i in idx], labels[idx], groups[idx]
        print(f"[dataset] SUBCONJUNTO de teste: {cfg.subset_groups_per_class} imagens/classe")
    return paths, labels, groups


def prepare_out_dir(cfg: Config) -> Path:
    ts = f"{datetime.now():%Y%m%d_%H%M%S}"
    out_dir = Path(cfg.drive_results_dir) / (cfg.run_name or f"run_{ts}")
    if out_dir.exists() and not cfg.resume:
        out_dir = out_dir.with_name(f"{out_dir.name}_{ts}")   # nunca apaga resultados antigos
    for sub in ("tabelas", "graficos/confusion_matrices", "graficos/curvas", "logs", "predictions"):
        (out_dir / sub).mkdir(parents=True, exist_ok=True)
    (out_dir / f"config_{ts}.json").write_text(json.dumps(dataclasses.asdict(cfg), indent=2, default=str))
    (out_dir / "environment.json").write_text(json.dumps(ENV_INFO, indent=2))
    if LOCK_PATH.exists():
        shutil.copy(LOCK_PATH, out_dir / "requirements_lock.txt")
    return out_dir


def _record(key, seed, extra: Dict, vm: Dict, tm: Dict) -> Dict:
    base = {c: None for c in ("batch_size", "accum_steps", "effective_batch", "batch_role", "best_epoch",
                              "epochs_run", "train_time_s", "time_to_best_s", "time_per_epoch_s",
                              "peak_vram_gb", "oom_retries", "C", "lbfgs_iter")}
    base["criterio_selecao"] = SELECTION_METRIC
    base.update(extra)
    return {**dict(zip(KEY_COLS, key)), "seed": seed, **base,
            **{f"test_{k}": tm[k] for k in METRIC_KEYS}, **{f"val_{k}": vm[k] for k in METRIC_KEYS},
            "timestamp": f"{datetime.now():%Y-%m-%d %H:%M:%S}"}


def main(cfg: Config):
    regimes, losses = _expand(cfg.regime, REGIMES), _expand(cfg.loss, LOSSES)
    lr_variants = _expand(cfg.lr_variant, LR_VARIANTS)
    paths, labels, groups = prepare_data(cfg)
    roi_wh = Image.open(paths[0]).size
    N = len(paths)
    print(f"[dataset] {N} ROIs de {len(set(groups))} imagens de origem | por classe: "
          f"{dict(zip(CLASS_NAMES, np.bincount(labels).tolist()))} | ROI {roi_wh[0]}x{roi_wh[1]}")

    out_dir = prepare_out_dir(cfg)
    tables_dir, logs_dir, pred_dir = out_dir / "tabelas", out_dir / "logs", out_dir / "predictions"
    partial_seeds = cfg.partial_seeds or cfg.seeds
    all_seeds = sorted(set(cfg.seeds) | set(partial_seeds))
    splits = {s: make_holdout_split(labels, groups, s, cfg) for s in all_seeds}
    save_table(split_summary(splits, labels, groups), tables_dir / "splits_resumo.csv",
               f"Split {cfg.train_frac:.0%}/{cfg.val_frac:.0%}/{cfg.test_frac:.0%} por semente "
               f"({'por imagem de origem' if cfg.split_by_group else 'por ROI'})")
    split_csv = out_dir / "splits.csv"
    new_split = pd.DataFrame([{"seed": s, "part": part, "idx": i, "path": paths[i], "group": groups[i]}
                              for s, sp in splits.items() for part, idx in sp.items() for i in idx])
    if split_csv.exists():
        new_split = pd.concat([pd.read_csv(split_csv), new_split]).drop_duplicates(["seed", "idx"])
    new_split.to_csv(split_csv, index=False)

    runs_csv = tables_dir / "execucoes_por_semente.csv"
    records = pd.read_csv(runs_csv).to_dict("records") if (cfg.resume and runs_csv.exists()) else []
    done = {norm_key(tuple(r[c] for c in KEY_COLS)) + (int(r["seed"]),) for r in records}
    if done:
        print(f"[retomada] {len(done)} execução(ões) já salvas em {out_dir.name} serão puladas")

    plan = []
    for bb_name in _expand(cfg.backbone, BACKBONES):
        for key, bc in build_scenarios(bb_name, cfg, regimes, losses, lr_variants):
            for seed in (partial_seeds if key[1].startswith("partial") else cfg.seeds):
                plan.append((key, bc, seed))
    pending = [p for p in plan if p[0] + (p[2],) not in done]
    touched = list(dict.fromkeys(k for k, _, _ in plan))
    print(f"[plano] {len(touched)} cenário(s), {len(plan)} execução(ões), {len(pending)} pendente(s)")
    if not pending and not cfg.consolidar_sem_pendentes:
        print("[plano] nada pendente: tudo já está salvo; consolidação pulada")
        return {"out_dir": out_dir}

    durations, mag_reports, n_done = [], [], 0
    for bb_name in dict.fromkeys(k[0] for k, _, _ in pending):
        todo = [p for p in pending if p[0][0] == bb_name]
        print(f"\n===== {bb_name}: {len(todo)} execução(ões) pendente(s) =====")
        hp = BACKBONE_HPARAMS.get(bb_name, DEFAULT_HPARAMS)
        bb, X_by_scale, X_aug, extract_bs = None, {}, None, {}
        frozen_scales = {k[4] for k, _, _ in todo if k[1].startswith("frozen")}
        if cfg.peft_lp_init and any(k[1].startswith(("lora", "bitfit")) for k, _, _ in todo):
            frozen_scales.add(round(float(cfg.peft_scale), 4))   # cabeça do LoRA/BitFit parte da sonda (LP-FT)
        frozen_scales = sorted(frozen_scales)
        if bb_name == "color":
            X_by_scale[1.0] = color_features(paths)
        elif bb_name == "fusecpath":
            X_by_scale[1.0] = fused_embeddings(paths, cfg)
        else:
            bb = build_backbone(bb_name, cfg)
            for sc in sorted(set(frozen_scales) | {1.0}):
                mag_reports.append(magnification_report(bb_name, bb.view_size, sc, roi_wh, mpp_captura(cfg)))
            for sc in frozen_scales:
                X_by_scale[sc], extract_bs[sc] = extract_embeddings(bb, paths, cfg, scale=sc)
            if any(k[1] in ("frozen_aug", "frozen_clf_aug") for k, _, _ in todo):
                X_aug = np.stack([extract_embeddings(bb, paths, cfg, 1.0, aug_copy=k)[0]
                                  for k in range(cfg.frozen_aug_copies)])

        for key, bc, seed in todo:
            _, regime, loss_name, lrv, sc, _ = key
            sp, tag = splits[seed], key_tag(key)
            t0 = time.time()
            print(f"  >> {' | '.join(map(str, key))} | semente {seed}")
            if regime.startswith("frozen_clf"):
                res = run_frozen_classifier(X_by_scale[sc], labels, sp, loss_name, cfg, seed,
                                            X_aug if regime == "frozen_clf_aug" else None)
                (yv, pv, dv), (yt, pt, dt) = res["val"], res["test"]
                pd.DataFrame(res["curve"]).to_csv(logs_dir / f"{tag}_seed{seed}_hpcurve.csv", index=False)
                extra = {"hiperparametro": res["params"], "batch_size": extract_bs.get(sc),
                         "batch_role": "extracao_embeddings", "train_time_s": time.time() - t0}
            elif regime.startswith("frozen"):
                res = run_frozen_probe(X_by_scale[sc], labels, sp, loss_name, cfg,
                                       X_aug if regime == "frozen_aug" else None)
                (yv, pv, dv), (yt, pt, dt) = res["val"], res["test"]
                pd.DataFrame(res["curve"]).to_csv(logs_dir / f"{tag}_seed{seed}_Ccurve.csv", index=False)
                extra = {"C": res["C"], "lbfgs_iter": res["lbfgs_iter"], "batch_size": extract_bs.get(sc),
                         "batch_role": "extracao_embeddings", "train_time_s": time.time() - t0}
            else:
                mode = regime_mode(regime)
                head_init = (linear_probe_init(X_by_scale[round(float(cfg.peft_scale), 4)], labels, sp, loss_name, cfg)
                             if mode in ("lora", "bitfit") and cfg.peft_lp_init else None)
                out = train_partial_dynamic(bb, cfg, seed, sp, paths, labels, loss_name, hp, lrv,
                                            use_aug=not regime.endswith("_noaug"), batch_cfg=bc,
                                            mode=mode, head_init=head_init)
                (yv, pv, dv), (yt, pt, dt) = out["val"], out["test"]
                out["history"].assign(**dict(zip(KEY_COLS, key)), seed=seed, batch_size=out["batch_size"],
                                      accum_steps=out["accum_steps"]).to_csv(
                    logs_dir / f"{tag}_seed{seed}_history.csv", index=False)
                extra = {k: out[k] for k in ("batch_size", "accum_steps", "effective_batch", "best_epoch",
                                             "epochs_run", "train_time_s", "time_to_best_s",
                                             "time_per_epoch_s", "peak_vram_gb", "oom_retries")}
                extra["batch_role"] = "treino"
            vm, tm = compute_metrics(yv, pv, dv), compute_metrics(yt, pt, dt)
            pd.DataFrame({"idx": sp["test"], "path": [paths[i] for i in sp["test"]],
                          "group": groups[sp["test"]], "y_true": yt, "y_pred": dt,
                          **{f"p_{c}": pt[:, m] for m, c in enumerate(CLASS_NAMES)}}
                         ).to_csv(pred_dir / f"{tag}_seed{seed}.csv", index=False)
            pd.DataFrame({"idx": sp["val"], "y_true": yv, "y_pred": dv,
                          **{f"p_{c}": pv[:, m] for m, c in enumerate(CLASS_NAMES)}}
                         ).to_csv(pred_dir / f"{tag}_seed{seed}_val.csv", index=False)
            records.append(_record(key, seed, extra, vm, tm))
            pd.DataFrame(records).to_csv(runs_csv, index=False)     # salva a cada execução
            durations.append(time.time() - t0)
            n_done += 1
            eta = np.mean(durations[-10:]) * (len(pending) - n_done)
            print(f"     teste: F1={tm['f1_macro']:.4f} QWK={tm['qwk']:.4f} AUROC={tm['auroc']:.4f} | "
                  f"val F1={vm['f1_macro']:.4f} | {durations[-1]:.0f}s | {n_done}/{len(pending)} "
                  f"| ETA ~{eta / 60:.0f} min")
        del bb, X_by_scale, X_aug
        liberar_memoria(f"{bb_name} descarregado")

    if mag_reports:
        mag_csv = tables_dir / "magnificacao.csv"
        mag = pd.DataFrame(mag_reports)
        if mag_csv.exists():
            mag = pd.concat([pd.read_csv(mag_csv), mag]).drop_duplicates(["backbone", "scale"])
        mag.to_csv(mag_csv, index=False)
    if not records:
        print("[aviso] nenhum resultado para consolidar")
        return None
    return consolidate(cfg, out_dir, pd.DataFrame(records), groups, touched)


def consolidate(cfg: Config, out_dir: Path, df: pd.DataFrame, groups: np.ndarray, touched: List[tuple]):
    """Tabelas (display + CSV) de TODOS os cenários do run e gráficos dos cenários desta chamada."""
    tables_dir, logs_dir, pred_dir = out_dir / "tabelas", out_dir / "logs", out_dir / "predictions"
    plots_dir = out_dir / "graficos"
    df = df.copy()
    df["scale"] = df.scale.astype(float).round(6)
    for c in ("batch_size", "accum_steps", "effective_batch", "best_epoch", "epochs_run", "train_time_s",
              "time_to_best_s", "time_per_epoch_s", "peak_vram_gb", "oom_retries", "C", "lbfgs_iter"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    touched_set = set(touched)
    df_t = df[[norm_key(tuple(r)) in touched_set for r in df[KEY_COLS].itertuples(index=False)]]

    cols = KEY_COLS + ["seed", "batch_role", "batch_size", "accum_steps", "effective_batch", "best_epoch",
                       "epochs_run", "C", "time_per_epoch_s", "time_to_best_s", "train_time_s", "peak_vram_gb",
                       "oom_retries"] + [f"test_{k}" for k in METRIC_KEYS] + ["val_qwk"]
    save_table(df_t[cols].rename(columns={**PT_NAMES, "seed": "Semente", "best_epoch": "Época (melhor)",
                                          "batch_size": "Batch size"}).round(4),
               tables_dir / "execucoes_desta_chamada.csv", "Execuções (cenários desta chamada)")
    tbl_a, tbl_b, raw = summary_tables(df, pred_dir, groups, cfg)
    save_table(tbl_a, tables_dir / "resumo_media_dp.csv",
               "Teste: média ± DP entre sementes (com metadados de batch, épocas e tempo)")
    save_table(tbl_b, tables_dir / "resumo_ic_bootstrap.csv",
               f"Teste: média entre sementes + IC {cfg.ci_level:.0%} (bootstrap de {cfg.n_boot} por imagem)")
    raw.to_csv(tables_dir / "resumo_ic_numerico.csv", index=False)

    for key in (touched if cfg.plot_per_scenario else []):
        gk = df[[norm_key(tuple(r)) == key for r in df[KEY_COLS].itertuples(index=False)]]
        if gk.empty:
            continue
        tag, seeds = key_tag(key), sorted(int(s) for s in gk.seed.unique())
        faltam = [f for s in seeds for f in (pred_dir / f"{tag}_seed{s}.csv",) if not f.exists()]
        if faltam:
            print(f"    [gráficos] {tag}: predições ausentes, gráficos do cenário pulados")
            continue
        cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
        for s in seeds:
            p = pd.read_csv(pred_dir / f"{tag}_seed{s}.csv")
            cm += confusion_matrix(p.y_true, p.y_pred, labels=list(range(NUM_CLASSES)))
        plot_confusion(key, cm, len(seeds), plots_dir / "confusion_matrices" / f"CM_{tag}.png", cfg)
        if key[1].startswith("frozen_clf"):
            continue                       # só a matriz de confusão (hiperparâmetros em logs/*_hpcurve.csv)
        logs_ok = all((logs_dir / f"{tag}_seed{s}_{'history' if key[1].startswith(TRAINED_REGIMES) else 'Ccurve'}.csv").exists()
                      for s in seeds)
        if not logs_ok:
            continue
        if key[1].startswith(TRAINED_REGIMES):
            hist = {s: pd.read_csv(logs_dir / f"{tag}_seed{s}_history.csv") for s in seeds}
            best = dict(zip(gk.seed.astype(int), gk.best_epoch.astype(int)))
            plot_learning_curves(key, hist, best, plots_dir / "curvas" / f"aprendizado_{tag}.png", cfg)
        else:
            curves = {s: pd.read_csv(logs_dir / f"{tag}_seed{s}_Ccurve.csv") for s in seeds}
            chosen = dict(zip(gk.seed.astype(int), gk.C.astype(float)))
            plot_c_curves(key, curves, chosen, plots_dir / "curvas" / f"curvaC_{tag}.png", cfg)

    plot_summary(raw, plots_dir / "resumo_f1_ic.png", cfg, metric=SELECTION_METRIC)
    plot_batch_size_analysis(df, logs_dir, plots_dir, cfg)

    (out_dir / "relatorio_comparativo.md").write_text(
        "# Relatório comparativo\n\n"
        f"- Split holdout {cfg.train_frac:.0%}/{cfg.val_frac:.0%}/{cfg.test_frac:.0%} via train_test_split, "
        f"estratificado, {'por imagem de origem' if cfg.split_by_group else 'por ROI'}, repetido por semente.\n"
        "- Multi-crop: janelas no tamanho nativo do backbone cobrindo toda a ROI; mean pooling.\n"
        "- Frozen: sonda linear L2 por L-BFGS, C escolhido pelo QWK de validação e reajustado em "
        "treino+validação. Outros classificadores (kNN, SimpleShot, LDA, SVM, MLP): hiperparâmetro "
        "pelo QWK de validação e reajuste em treino+validação. LoRA/BitFit: protocolo único "
        f"({PEFT_PROTOCOL['fonte']}), cabeça iniciada pela sonda (LP-FT), época pelo QWK de validação.\n"
        f"- Batch efetivo padrão do partial: {cfg.effective_batch} ROIs; lote físico pela tabela/VRAM.\n"
        f"- IC {cfg.ci_level:.0%}: bootstrap percentil ({cfg.n_boot} réplicas) reamostrando imagens "
        "de origem (estratificado por classe) dentro do teste de cada semente.\n\n"
        "## Média ± DP entre sementes\n\n" + tbl_a.to_markdown(index=False) +
        "\n\n## Média entre sementes com IC\n\n" + tbl_b.to_markdown(index=False) + "\n")
    print(f"\n[ok] artefatos salvos em {out_dir}")
    return {"execucoes": df, "resumo": tbl_a, "ic": tbl_b, "out_dir": out_dir}

# %% [CÉLULA 10] Rotina de testes: sanidade, frozen (sonda + classificadores) e PEFT (LoRA/BitFit)
from scipy.optimize import check_grad  # noqa: E402


def run_sanity_checks(cfg: Config, check_data: bool = True) -> None:
    """Testes rápidos (segundos, sem GPU pesada) que pegam erros silenciosos antes de gastar
    horas de Colab. Qualquer falha levanta AssertionError."""
    rng = np.random.default_rng(0)
    K = NUM_CLASSES

    # 1) gradiente analítico da sonda linear == gradiente numérico
    X, y = rng.normal(size=(40, 6)), rng.integers(0, K, 40)
    for loss in LOSSES:
        wshape, nb = _probe_shapes(loss, 6)
        th = rng.normal(scale=0.1, size=wshape[0] * wshape[1] + nb)
        err = check_grad(lambda t: _probe_objective(t, X, y, loss, 0.01)[0],
                         lambda t: _probe_objective(t, X, y, loss, 0.01)[1], th)
        assert err < 1e-4, f"gradiente da sonda {loss} incorreto (erro {err:.2e})"

    # 2) losses torch (partial) == objetivo numpy (frozen) com lam=0
    Z = rng.normal(size=(30, K))
    yt = rng.integers(0, K, 30)
    ce_np = _probe_objective(np.concatenate([np.eye(K).ravel(), np.zeros(K)]), Z, yt, "ce", 0.0)[0]
    Zc = Z[:, :K - 1]
    corn_np = _probe_objective(np.concatenate([np.eye(K - 1).ravel(), np.zeros(K - 1)]), Zc, yt, "corn", 0.0)[0]
    x1, bias = Z[:, :1], rng.normal(size=K - 1)
    coral_np = _probe_objective(np.concatenate([[1.0], bias]), x1, yt, "coral", 0.0)[0]
    tz, ty = torch.tensor(Z), torch.tensor(yt)
    assert abs(float(make_criterion("ce", 0.0)(tz, ty)) - ce_np) < 1e-8, "CE torch != numpy"
    assert abs(float(corn_loss(torch.tensor(Zc), ty)) - corn_np) < 1e-8, "CORN torch != numpy"
    assert abs(float(coral_loss(torch.tensor(x1 + bias), ty)) - coral_np) < 1e-8, "CORAL torch != numpy"

    # 3) decodificação: probabilidades somam 1 e a predição ordinal é coerente
    for loss in LOSSES:
        P, d = np_probs_preds(rng.normal(size=(50, K if loss == "ce" else K - 1)), loss)
        assert np.allclose(P.sum(1), 1) and d.min() >= 0 and d.max() < K, f"decodificação {loss}"

    # 4) métricas rápidas (bootstrap) == métricas sklearn (inclui classes ausentes)
    for trial in range(30):
        n = int(rng.integers(8, 60))
        yy = rng.integers(0, K - (trial % 3 == 0), n)       # às vezes falta a última classe
        PP = rng.dirichlet(np.ones(K), n)
        dd = PP.argmax(1) if trial % 2 else rng.integers(0, K, n)
        a, b = compute_metrics(yy, PP, dd), fast_metrics(yy, PP, dd)
        for k in METRIC_KEYS:
            assert (np.isnan(a[k]) and np.isnan(b[k])) or abs(a[k] - b[k]) < 1e-9, f"métrica {k}: {a[k]} x {b[k]}"

    # 5a) LoRA: com B = 0 a saída é idêntica à camada original e o gradiente chega em B
    base_lin = nn.Linear(8, 6)
    lw = LoRALinear(base_lin, 4, 4, 0.0)
    xx = torch.randn(5, 8)
    assert torch.allclose(lw(xx), base_lin(xx)), "LoRA com B=0 altera a saída"
    lw(xx).sum().backward()
    assert float(lw.lora_B.grad.abs().sum()) > 0, "LoRA sem gradiente em B"

    # 5b) LP-FT: a sonda padronizada vira uma cabeça na escala original sem mudar os logits
    Xr_ = rng.normal(size=(60, 7)) * rng.uniform(0.5, 3.0, 7) + rng.normal(size=7)
    sc_ = StandardScaler().fit(Xr_)
    for loss in LOSSES:
        wshape, nb = _probe_shapes(loss, 7)
        W_, b_ = rng.normal(size=wshape), rng.normal(size=nb)
        head = build_head(loss, 7, K)
        load_head_init(head, W_ / sc_.scale_[:, None], b_ - (sc_.mean_ / sc_.scale_) @ W_)
        with torch.no_grad():
            got = head(torch.tensor(Xr_, dtype=torch.float32)).numpy()
        assert np.allclose(got, probe_logits(sc_.transform(Xr_), W_, b_), atol=1e-4), f"LP-FT {loss}"

    # 5c) classificadores alternativos e LP-FT rodam e aprendem um problema separável
    yc_ = rng.integers(0, K, 160)
    Xc_ = rng.normal(size=(160, 12))
    Xc_[:, :3] += 2.0 * yc_[:, None]
    Xc_ = Xc_ * rng.uniform(0.5, 3.0, 12) + rng.normal(size=12)
    sp_c = {"train": np.arange(0, 100), "val": np.arange(100, 130), "test": np.arange(130, 160)}
    for c in CLASSIFIERS:
        q = compute_metrics(*run_frozen_classifier(Xc_, yc_, sp_c, c, cfg, 0)["test"])["qwk"]
        assert q > 0.6, f"classificador {c} não aprendeu (QWK {q:.3f})"
    linear_probe_init(Xc_, yc_, sp_c, "ce", cfg)

    # 5) multi-crop cobre a ROI inteira com janelas no tamanho nativo
    views = roi_views(Image.new("RGB", (450, 250)), 224)
    assert len(views) == 2 and all(v.size == (224, 224) for v in views), "multi-crop inesperado"

    # 6) split 70/20/10 sem vazamento, em todas as sementes, no dataset real
    if check_data:
        paths, labels, groups = prepare_data(cfg)
        for s in cfg.seeds:
            make_holdout_split(labels, groups, s, cfg)      # as asserções estão dentro
        display(split_summary({s: make_holdout_split(labels, groups, s, cfg) for s in cfg.seeds[:3]},
                              labels, groups))

    # 7) plano de batch size para esta GPU
    print(f"[batch] GPU {ENV_INFO['gpu']} ({ENV_INFO.get('vram_gb')} GB) -> fator {vram_factor()}x")
    display(pd.DataFrame([{"backbone": b, "extract": resolve_batch_size(b, "extract", cfg),
                           **{f"partial_{k}": v for k, v in partial_batch_plan(b, cfg, None).items()}}
                          for b in BACKBONES if BATCH_SIZE_T4[b]["partial"]]))
    print("[sanidade] todas as checagens passaram")


# ---------------------------------------------------------------------------------------
# BATERIA (aplicada a CADA modelo de MODELOS_PARA_TESTAR). Cada modelo grava em
# frozen_<modelo> (pasta nova; as bateria_* antigas não são tocadas) e a retomada pula o que
# já terminou: se o Colab desconectar, rode as células de novo.
#   0_smoke           : subconjunto pequeno; valida o pipeline FROZEN do modelo (carregar pesos,
#                       extrair embeddings, sonda e classificadores). Se falhar, o modelo é pulado.
#   1_frozen_lbfgs    : sonda L-BFGS (protocolo publicado), 3 losses x sem/com augmentation
#   2_classificadores : kNN, SimpleShot, LDA, SVM e MLP x sem/com augmentation
#   LoRA e BitFit rodam em PASSAGEM SEPARADA (grupos abaixo), cada regime com o seu próprio
#   smoke de 1 época: se o BitFit quebrar, o frozen e o LoRA do modelo não são perdidos.
#       3a_smoke_lora / 3_lora  e  3a_smoke_bitfit / 3_bitfit
#     Loss = a melhor da sonda pela VALIDAÇÃO. Caro no T4: por padrão só nos TOP_K_PEFT modelos
#     com melhor QWK de validação no frozen (ou na lista MODELOS_PEFT).
#   4_magnificacao    : opcional (escalas 1,0 / 0,75 / 0,5 na sonda L-BFGS)
# Todas as fases usam as 2 sementes de SEEDS. Escolhas sempre pela validação, nunca pelo teste.
# ---------------------------------------------------------------------------------------
def _smoke(bb: str, regimes: List[str], nome: str) -> Dict:
    return dict(run_name=f"smoke_{nome}_{bb}", resume=False, backbone=[bb], regime=regimes,
                loss=["ce"], epochs=1, patience=1, subset_groups_per_class=8,
                c_grid=[1e-3, 1e-2, 1e-1], frozen_aug_copies=1, n_boot=50,
                partial_batch_configs=[None], effective_batch=PEFT_PROTOCOL["batch"])


def suite_phases(bb: str) -> Dict[str, Dict]:
    peft = dict(backbone=[bb], loss=["@best_frozen"], epochs=PEFT_PROTOCOL["epochs"],
                patience=PEFT_PROTOCOL["patience"],
                partial_batch_configs=[None], effective_batch=PEFT_PROTOCOL["batch"])
    return {
        "0_smoke": _smoke(bb, ["frozen", "frozen_aug", "frozen_clf", "frozen_clf_aug"], "frozen"),
        "1_frozen_lbfgs": dict(backbone=[bb], regime=["frozen", "frozen_aug"], loss=["all"]),
        "2_classificadores": dict(backbone=[bb], regime=["frozen_clf", "frozen_clf_aug"]),
        "3a_smoke_lora": _smoke(bb, ["lora"], "lora"),
        "3_lora": dict(peft, regime=["lora"]),
        "3a_smoke_bitfit": _smoke(bb, ["bitfit"], "bitfit"),
        "3_bitfit": dict(peft, regime=["bitfit"]),
        "4_magnificacao": dict(backbone=[bb], regime=["frozen", "frozen_clf"], loss=["@best_frozen"],
                               classifiers=["@best_clf"], scales=["@magnificacao"]),
    }


def pick_best_loss(run_dir: Path, bb: str, regime: str) -> str:
    f = run_dir / "tabelas" / "execucoes_por_semente.csv"
    if not f.exists():
        raise RuntimeError(f"Rode antes a fase com regime '{regime}' (não achei {f}).")
    df = pd.read_csv(f)
    df = df[(df.backbone == bb) & (df.regime == regime) & (df.batch_cfg == "auto") &
            (df.lr_variant.isin(["mid", "C_busca", "hp_busca"])) & (df.scale.round(6) == 1.0)]
    if df.empty:
        raise RuntimeError(f"Sem execuções de {bb}/{regime} em {f}.")
    by = df.groupby("loss")[f"val_{SELECTION_METRIC}"].mean().sort_values(ascending=False)
    print(f"[seleção] {SEL_NOME} médio de VALIDAÇÃO por loss ({bb}, {regime}): {by.round(4).to_dict()}"
          f" -> {by.index[0]}")
    return str(by.index[0])


def melhor_escala_frozen(run_dir: Path, bb: str, seeds: List[int]) -> float:
    """Escala (magnificação) com maior F1 de validação no frozen SEM augmentation (frozen e
    frozen_clf, os dois regimes rodados em todas as escalas: comparação justa)."""
    f = run_dir / "tabelas" / "execucoes_por_semente.csv"
    if not f.exists():
        return 1.0
    df = pd.read_csv(f)
    df = df[(df.backbone == bb) & df.regime.isin(["frozen", "frozen_clf"])]
    st = df.groupby(["scale", "regime", "loss"]).agg(n=("seed", "nunique"), f=(f"val_{SELECTION_METRIC}", "mean"),
                                                    q=("val_qwk", "mean")).reset_index()
    st = st[st.n >= len(seeds)]
    if st.empty or st.scale.nunique() < 2:
        return 1.0
    por = st.sort_values(["f", "q"], ascending=False).groupby("scale", sort=False).head(1)
    esc = round(float(por.iloc[0].scale), 4)
    print(f"[magnificação] {bb}: melhor {SEL_NOME} de validação por escala "
          f"{dict(zip(por.scale.round(4), por.f.round(4)))} -> LoRA/BitFit na escala {esc:g}")
    return esc


def run_suite(bb: str, fases: List[str], base: Config, run_name: Optional[str] = None,
              seeds: Optional[List[int]] = None) -> Dict[str, Dict]:
    run_dir = _pasta_modelo(bb, base) if run_name is None else Path(base.drive_results_dir) / run_name
    run_name = run_dir.name
    importados = importar_execucoes_anteriores(bb, base)   # nada do que já existe é refeito
    phases, results = suite_phases(bb), {}
    for fase in fases:
        ov = dict(phases[fase])
        if "smoke" in fase and _ja_validado(bb, fase, base):
            print(f"[fase {fase}] pulada: {bb} já tem resultados deste tipo salvos (pipeline já validado)")
            continue
        if bb in FROZEN_ONLY and fase.startswith("3"):
            print(f"[fase {fase}] pulada: {bb} não suporta LoRA/BitFit")
            continue
        if bb in ("fusecpath", "color") and fase == "4_magnificacao":
            print(f"[fase {fase}] pulada: {bb} só roda na escala 1.0")
            continue
        if seeds is not None and "seeds" not in ov:
            ov["seeds"] = list(seeds)
        if ov.get("loss") == ["@best_frozen"]:
            ov["loss"] = [pick_best_loss(run_dir, bb, "frozen")]
        if ov.get("classifiers") == ["@best_clf"]:
            ov["classifiers"] = [pick_best_loss(run_dir, bb, "frozen_clf")]
        if ov.get("scales") == ["@magnificacao"]:
            esc = escalas_magnificacao(bb, base)
            print(f"[magnificação] {bb}: " + " | ".join(f"{s:g} = {r}" for s, r in esc.items()))
            ov["scales"] = list(esc)
        if fase in ("3_lora", "3_bitfit") and globals().get("PEFT_NA_MELHOR_MAGNIFICACAO", True):
            ov["peft_scale"] = melhor_escala_frozen(run_dir, bb, list(seeds or base.seeds))
        ov.setdefault("run_name", run_name)
        ov.setdefault("resume", True)
        if importados and "smoke" not in fase:
            ov["consolidar_sem_pendentes"] = True     # tabelas passam a incluir o que foi importado
            importados = 0
        cfg = dataclasses.replace(base, **ov)
        print("\n" + "#" * 90 + f"\n# FASE {fase} | {bb} | run '{cfg.run_name}'\n" + "#" * 90)
        t0 = time.time()
        results[fase] = main(cfg)
        liberar_memoria(f"fim da fase {fase} ({bb})")
        if "smoke" in fase:
            run_dir.mkdir(parents=True, exist_ok=True)
            (run_dir / f"{fase}.ok").write_text(f"{datetime.now():%Y-%m-%d %H:%M:%S}\n")
        print(f"[fase {fase}] concluída em {(time.time() - t0) / 60:.1f} min")
    return results


def run_all_foundations(foundations: List[str], grupos, base: Config,
                        status_name: str = "status_frozen.csv") -> "pd.DataFrame":
    """Roda, para cada modelo, cada GRUPO de fases em sequência. Os grupos são independentes:
    se um falhar (acesso negado no HF, peso local ausente, versão incompatível, erro de GPU),
    o erro é registrado e o próximo grupo/modelo segue. 'grupos' é uma lista de fases
    (um grupo só) ou uma lista de listas. Status em resultados_histopatologia/<status_name>."""
    import traceback
    if grupos and isinstance(grupos[0], str):
        grupos = [grupos]
    status_csv = Path(base.drive_results_dir) / status_name
    rows = []
    for i, bb in enumerate(foundations, 1):
        for fases in grupos:
            print("\n" + "=" * 90 + f"\n= [{i}/{len(foundations)}] MODELO: {bb} | fases {fases}\n" + "=" * 90)
            t0 = time.time()
            ram0 = ram_status()["livre_gb"]
            print(f"[RAM] antes de {bb}: {ram_txt()}")
            try:
                run_suite(bb, fases, base)
                status, erro = "ok", ""
            except RAMInsuficiente as e:
                print(f"\n[RAM] {e}")
                status, erro = "pulado_ram", str(e)[:500]
            except Exception as e:   # noqa: BLE001 — registra e segue
                traceback.print_exc()
                status, erro = "falhou", f"{type(e).__name__}: {e}"[:500]
            finally:
                liberar_memoria(f"depois de {bb}")
            rows.append({"modelo": bb, "fases": "+".join(fases), "status": status, "erro": erro,
                         "minutos": round((time.time() - t0) / 60, 1), "ram_livre_antes_gb": round(ram0, 1),
                         "ram_livre_depois_gb": round(ram_status()["livre_gb"], 1)})
            pd.DataFrame(rows).to_csv(status_csv, index=False)
    df = pd.DataFrame(rows)
    display(df)
    return df


def verificar_pesos_locais(modelos: List[str]) -> None:
    """Mostra, antes de começar, quais pesos baixados manualmente estão (ou não) em PESOS_DIR."""
    pend = [m for m in modelos if m in ("chief", "beph", "pathoduet")]
    if not pend:
        return
    print(f"[pesos locais] pasta: {PESOS_DIR}")
    for m in pend:
        f = local_weight_path(m)
        print(f"   {'OK     ' if f.exists() else 'FALTA  '} {m:10s} -> {f.name}")


VAL_COL = f"{SEL_NOME} val"   # coluna de seleção nas tabelas comparativas


def _fmt_pm(x: pd.Series) -> str:
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f}"


def compare_foundations(foundations: List[str], base: Config) -> Optional[Dict[str, "pd.DataFrame"]]:
    """Comparação entre modelos com seleção PELA VALIDAÇÃO (o teste só é lido depois): para
    cada modelo e cada regime, escolhe a configuração (loss/classificador, lote) com maior QWK
    médio de validação entre as sementes e reporta o teste dela. Escopos-resumo: 'frozen'
    (todos os regimes sem treino do encoder) e 'todos' (inclui LoRA/BitFit)."""
    cfg_cols = ["regime", "loss", "lr_variant", "scale", "batch_cfg"]   # a escala (magnificação) entra na escolha
    linhas = []
    for bb in foundations:
        f = _pasta_modelo(bb, base) / "tabelas" / "execucoes_por_semente.csv"
        if not f.exists():
            continue
        df = pd.read_csv(f)
        df = df[df.backbone == bb].copy()
        df["scale"] = df.scale.astype(float).round(4)
        if df.empty:
            continue
        stats = df.groupby(cfg_cols).agg(n=("seed", "nunique"), vq=(f"val_{SELECTION_METRIC}", "mean")).reset_index()
        stats = stats[stats.n == df.seed.nunique()]       # só configurações com todas as sementes

        def linha(st, escopo):
            best = st.loc[st.vq.idxmax()]
            d = df[(df[cfg_cols] == best[cfg_cols].values).all(axis=1)]
            r = {"Modelo": bb, "Escopo": escopo, **{PT_NAMES.get(c, c): best[c] for c in cfg_cols},
                 "Sementes": int(best.n), "Configs comparadas": len(st), VAL_COL: round(best.vq, 4)}
            if bb not in ("ensemble", "color", "fusecpath") and _ROI_WH:
                r["Zoom (x captura)"] = round(janela_do_modelo(bb, base) * float(best.scale) / _ROI_WH[1], 2)
            if "criterio_selecao" in d:
                r["Critério interno"] = ",".join(sorted(d.criterio_selecao.fillna("qwk").astype(str).unique()))
            for k, nome in (("f1_macro", "F1 teste"), ("qwk", "QWK teste"), ("auroc", "AUROC teste"),
                            ("bacc", "Bal. acc teste")):
                r[nome] = _fmt_pm(d[f"test_{k}"])
            return r

        for reg, st in stats.groupby("regime"):
            linhas.append(linha(st, reg))
        fr = stats[stats.regime.str.startswith("frozen")]
        if not fr.empty:
            linhas.append(linha(fr, "frozen (todos)"))
        linhas.append(linha(stats, "todos"))
    if not linhas:
        print("[comparação] nenhum resultado encontrado")
        return None
    t = pd.DataFrame(linhas)
    out_dir = Path(base.drive_results_dir)
    resumo = {}
    for escopo, nome in (("frozen (todos)", "comparacao_frozen_melhor_por_validacao.csv"),
                         ("todos", "comparacao_geral_melhor_por_validacao.csv")):
        tt = t[t.Escopo == escopo].sort_values(VAL_COL, ascending=False)
        save_table(tt, out_dir / nome, f"Melhor configuração de cada modelo — escopo '{escopo}' "
                                       f"(escolhida pela VALIDAÇÃO)")
        resumo[escopo] = tt
    por_reg = t[~t.Escopo.isin(["frozen (todos)", "todos"])]
    save_table(por_reg, out_dir / "comparacao_por_regime.csv",
               "Melhor configuração por regime (sonda L-BFGS, classificadores, LoRA, BitFit)")
    resumo["por_regime"] = por_reg
    return resumo


def selecionar_para_peft(comp: Optional[Dict], k: int) -> List[str]:
    """Top-k modelos pelo QWK de VALIDAÇÃO no escopo frozen (só os que suportam LoRA/BitFit)."""
    if not comp or comp.get("frozen (todos)") is None:
        return []
    t = comp["frozen (todos)"]
    t = t[~t.Modelo.isin(FROZEN_ONLY)].sort_values(VAL_COL, ascending=False)
    escolhidos = t.Modelo.head(k).tolist()
    print(f"[PEFT] top-{k} pelo {SEL_NOME} de validação no frozen: {escolhidos}")
    return escolhidos


# =======================================================================================
# REAPROVEITAMENTO DO QUE JÁ FOI FEITO
# =======================================================================================
REAPROVEITAR_EXECUCOES = True   # False = refaz tudo com o critério F1 em pastas frozen_f1_<modelo>
PASTAS_IGNORADAS = ("smoke",)   # pastas de teste rápido nunca são importadas


def _pasta_modelo(bb: str, base: Config) -> Path:
    raiz = Path(base.drive_results_dir)
    if bb == "ensemble":
        return raiz / "ensemble"
    if base.estratificar_por_dia:          # outro split: nunca mistura com os resultados antigos
        return raiz / f"frozen_dia_{bb}"
    return raiz / (f"frozen_{bb}" if REAPROVEITAR_EXECUCOES else f"frozen_f1_{bb}")


def _chave_linha(r) -> tuple:
    return norm_key(tuple(r[c] for c in KEY_COLS)) + (int(r["seed"]),)


def _splits_iguais(src: Path, dst: Path, seed: int) -> bool:
    """Confere, pelos splits.csv salvos, se a semente gerou as MESMAS partições nas duas pastas."""
    fs, fd = src / "splits.csv", dst / "splits.csv"
    if not (fs.exists() and fd.exists()):
        return True     # sem como conferir: o split depende só da semente e dos dados
    a, b = pd.read_csv(fs), pd.read_csv(fd)

    def norm(d):   # classe (+ subpasta) + arquivo: 'healthy/image032-roi1.tif' != 'moderate/image032-roi1.tif'
        d = d[d.seed == seed]
        rel = d.path.map(lambda x: "/".join(Path(x).parts[_pos_classe(Path(x).parts):]))
        return set(zip(d.part, rel))
    return norm(a) == norm(b)


def importar_execucoes_anteriores(bb: str, base: Config) -> int:
    """Copia para frozen_<modelo> as execuções JÁ FEITAS em outras pastas de resultados (ex.:
    bateria_uni e bateria_conch do código antigo, que tinham 10 sementes) cujo cenário e
    semente coincidem com os desta bateria. Mesmo protocolo, mesmos splits (conferidos) e os
    mesmos embeddings em cache -> o resultado seria idêntico, então nada é recalculado.
    Copia também as predições e os logs. Devolve quantas execuções foram importadas."""
    if not REAPROVEITAR_EXECUCOES:
        return 0
    raiz, dst = Path(base.drive_results_dir), _pasta_modelo(bb, base)
    dst_csv = dst / "tabelas" / "execucoes_por_semente.csv"
    atuais = pd.read_csv(dst_csv) if dst_csv.exists() else pd.DataFrame()
    feitos = {_chave_linha(r) for _, r in atuais.iterrows()} if len(atuais) else set()
    novos = []
    for src_csv in sorted(raiz.glob("*/tabelas/execucoes_por_semente.csv")):
        src = src_csv.parent.parent
        if src == dst or src.name.startswith(PASTAS_IGNORADAS) or src.name == "ensemble":
            continue
        try:
            df = pd.read_csv(src_csv)
        except Exception:   # noqa: BLE001
            continue
        if "backbone" not in df or "seed" not in df:
            continue
        df = df[(df.backbone == bb) & (df.seed.astype(int).isin(base.seeds))]
        for _, r in df.iterrows():
            k = _chave_linha(r)
            if k in feitos or not _splits_iguais(src, dst, int(r["seed"])):
                continue
            tag = key_tag(k[:-1])
            for sub, padrao in (("predictions", f"{tag}_seed{k[-1]}*.csv"), ("logs", f"{tag}_seed{k[-1]}_*.csv")):
                (dst / sub).mkdir(parents=True, exist_ok=True)
                for f in (src / sub).glob(padrao):
                    if not (dst / sub / f.name).exists():
                        shutil.copy2(f, dst / sub / f.name)
            linha = r.to_dict()
            linha.setdefault("criterio_selecao", "qwk")
            if pd.isna(linha.get("criterio_selecao")):
                linha["criterio_selecao"] = "qwk"
            linha["importado_de"] = src.name
            novos.append(linha)
            feitos.add(k)
    if novos:
        dst_csv.parent.mkdir(parents=True, exist_ok=True)
        pd.concat([atuais, pd.DataFrame(novos)], ignore_index=True).to_csv(dst_csv, index=False)
        origem = sorted({n["importado_de"] for n in novos})
        print(f"[reaproveitamento] {bb}: {len(novos)} execução(ões) importada(s) de {origem} (nada recalculado)")
    return len(novos)


def _ja_validado(bb: str, fase: str, base: Config) -> bool:
    """Smoke test é pulado se o modelo já tem resultados do tipo que o smoke valida."""
    f = _pasta_modelo(bb, base) / "tabelas" / "execucoes_por_semente.csv"
    if (_pasta_modelo(bb, base) / f"{fase}.ok").exists():
        return True
    if not f.exists():
        return False
    regs = set(pd.read_csv(f, usecols=["regime"]).regime.astype(str))
    alvo = {"0_smoke": ("frozen",), "3a_smoke_lora": ("lora",), "3a_smoke_bitfit": ("bitfit",)}.get(fase, ())
    return any(r.startswith(alvo) for r in regs) if alvo else False


# =======================================================================================
# PÓS-PROCESSAMENTO (TTA e limiar de decisão para F1) — sem novas extrações se houver cache
# =======================================================================================
POS_REGIMES = ("pos_base", "pos_tta", "pos_vies", "pos_tta_vies")


def carregar_embeddings_do_cache(bb_name: str, paths: List[str], cfg: Config, scale: float = 1.0,
                                 aug_copy: Optional[int] = None) -> Optional[np.ndarray]:
    """Procura no cache um arquivo deste modelo com a mesma escala, cópia de augmentation e lista
    de imagens (o cache guarda esses metadados), sem precisar carregar o modelo."""
    alvo, achados = np.array(paths), []
    for f in Path(cfg.embeddings_dir).glob(f"{bb_name}_*.npz"):
        try:
            z = np.load(f, allow_pickle=False)
            meta = json.loads(str(z["meta"]))
            if (meta.get("backbone") != bb_name or float(meta.get("scale", -1)) != float(scale)
                    or meta.get("aug_copy") != aug_copy or not np.array_equal(z["paths"], alvo)):
                continue
            mesmo_peso = meta.get("hf_sha") in (ENV_INFO["hf_sha"].get(bb_name),
                                               ENV_INFO["local_weights"].get(bb_name))
            achados.append((mesmo_peso, f.stat().st_mtime, f))
        except Exception:   # noqa: BLE001 — arquivo de outra versão: ignora
            continue
    if not achados:
        return None
    return np.load(sorted(achados)[-1][2], allow_pickle=False)["X"]


def _embeddings_pos(bb_name: str, paths: List[str], cfg: Config, scale: float = 1.0):
    """(X, X_aug) na escala pedida: do cache quando possível; senão extrai (e grava no cache).
    Em escala diferente de 1,0 as cópias aumentadas do TTA são extraídas nessa escala."""
    if bb_name == "color":
        return color_features(paths), None
    if bb_name == "fusecpath":
        partes = [carregar_embeddings_do_cache(c, paths, cfg) for c in FUSECPATH_COMPONENTS]
        return (np.concatenate(partes, 1) if all(p is not None for p in partes)
                else fused_embeddings(paths, cfg)), None
    X = carregar_embeddings_do_cache(bb_name, paths, cfg, scale)
    Xa = [carregar_embeddings_do_cache(bb_name, paths, cfg, scale, k) for k in range(cfg.frozen_aug_copies)]
    if X is None or any(a is None for a in Xa):
        bb = build_backbone(bb_name, cfg)
        X = extract_embeddings(bb, paths, cfg, scale)[0]
        Xa = [extract_embeddings(bb, paths, cfg, scale, k)[0] for k in range(cfg.frozen_aug_copies)]
        del bb
        liberar_memoria(f"pós: {bb_name} descarregado")
    return X, (np.stack(Xa) if Xa else None)


def _preditor_treino(regime: str, nome: str, X, y, sp, cfg: Config, seed: int, X_aug):
    """Ajusta SÓ no treino a configuração escolhida (hiperparâmetro pelo critério na validação)
    e devolve f(X) -> (probabilidades, predições). Sem reajuste em treino+validação, porque a
    validação é usada a seguir para calibrar o limiar de decisão."""
    Xtr, ytr = _stack_rows(X, y, sp["train"], X_aug if regime.endswith("_aug") else None)
    Xv, yv = X[sp["val"]], y[sp["val"]]
    best = None
    if regime.startswith("frozen_clf"):
        for params in _clf_grid(nome, len(ytr)):
            f = _clf_fit(nome, params, Xtr, ytr, seed)
            q = _sel(compute_metrics(yv, *f(Xv)))
            if best is None or q > best[0]:
                best = (q, f, params)
        return best[1], best[2]
    sc = StandardScaler().fit(Xtr)
    Xs, theta, since = sc.transform(Xtr), None, 0
    for C in sorted(cfg.c_grid):
        theta, _ = fit_probe(Xs, ytr, nome, C, cfg.probe_max_iter, theta)
        W, b = _unpack(theta, nome, Xs.shape[1])

        def f(Xe, W=W.copy(), b=b.copy()):
            return np_probs_preds(probe_logits(sc.transform(Xe), W, b), nome)
        P, d = f(Xv)
        q = _sel(compute_metrics(yv, P, d))
        if best is None or q > best[0]:
            best, since = (q, f, {"C": C}), 0
        elif len(np.unique(d)) > 1:
            since += 1
            if cfg.c_patience and since >= cfg.c_patience:
                break
    return best[1], best[2]


def _f1_rapido(y: np.ndarray, d: np.ndarray) -> float:
    K = NUM_CLASSES
    cm = np.bincount(y * K + d, minlength=K * K).reshape(K, K).astype(float)
    tp, fp, fn = np.diag(cm), cm.sum(0) - np.diag(cm), cm.sum(1) - np.diag(cm)
    den = 2 * tp + fp + fn
    return float(np.mean(np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), 0.0)))


def ajustar_vies_f1(P: np.ndarray, y: np.ndarray, grade=np.linspace(-1.5, 1.5, 31), passadas: int = 3):
    """Vieses aditivos no log da probabilidade (um por classe; a classe 0 fica em 0) escolhidos por
    subida coordenada para MAXIMIZAR o F1 macro (ajuste de limiar/'logit adjustment' pós-hoc).
    Empate -> vieses menores (mais perto da decisão original)."""
    L = np.log(np.clip(P, 1e-9, 1.0))
    b = np.zeros(P.shape[1])

    def score(v):
        return (_f1_rapido(y, (L + v).argmax(1)), -np.abs(v).sum())
    best = score(b)
    for _ in range(passadas):
        mudou = False
        for k in range(1, P.shape[1]):
            for v in grade:
                c = b.copy()
                c[k] = v
                sc_ = score(c)
                if sc_ > best:
                    best, b, mudou = sc_, c, True
        if not mudou:
            break
    return b


def _aplicar_vies(P: np.ndarray, b: np.ndarray) -> np.ndarray:
    return (np.log(np.clip(P, 1e-9, 1.0)) + b).argmax(1)


def _vies_fora_da_dobra(P: np.ndarray, y: np.ndarray, g: np.ndarray) -> np.ndarray:
    """Métrica de VALIDAÇÃO honesta para a variante com viés: ajusta em metade das imagens de
    validação e prediz a outra metade (2 dobras por imagem de origem). Sem isso, o F1 de
    validação do viés seria otimista e 'ganharia' a seleção sem merecer."""
    from sklearn.model_selection import GroupKFold
    d = P.argmax(1)
    if len(np.unique(g)) < 2:
        return d
    for a, b_idx in GroupKFold(n_splits=2).split(P, y, g):
        d[b_idx] = _aplicar_vies(P[b_idx], ajustar_vies_f1(P[a], y[a]))
    return d


def pos_processamento(bb_name: str, base: Config) -> int:
    """Para a MELHOR configuração frozen do modelo (pela validação: F1, desempate QWK), gera:
       pos_base     : ajuste só no treino, decisão padrão
       pos_tta      : média das probabilidades da ROI original + cópias aumentadas (TTA)
       pos_vies     : vieses por classe que maximizam o F1 macro na validação
       pos_tta_vies : TTA + vieses
    Usa os embeddings do cache (sem nova extração quando existem). Pula o que já foi feito."""
    run_dir = _pasta_modelo(bb_name, base)
    csv = run_dir / "tabelas" / "execucoes_por_semente.csv"
    if not csv.exists():
        print(f"[pós] {bb_name}: sem resultados frozen; pulado")
        return 0
    todos = pd.read_csv(csv)
    df = todos[(todos.backbone == bb_name)
               & todos.regime.isin(["frozen", "frozen_aug", "frozen_clf", "frozen_clf_aug"])].copy()
    df["scale"] = df.scale.astype(float).round(4)          # a melhor pode estar em outra magnificação
    st = df.groupby(["regime", "loss", "scale"]).agg(n=("seed", "nunique"), f=(f"val_{SELECTION_METRIC}", "mean"),
                                           q=("val_qwk", "mean")).reset_index()
    st = st[st.n >= len(base.seeds)]
    if st.empty:
        print(f"[pós] {bb_name}: nenhuma configuração frozen com todas as sementes; pulado")
        return 0
    best = st.sort_values(["f", "q"], ascending=False).iloc[0]
    regime, nome, esc = str(best.regime), str(best.loss), float(best.scale)
    tem_tta = bb_name not in ("color", "fusecpath") and base.frozen_aug_copies > 0
    variantes = [v for v in POS_REGIMES if tem_tta or "tta" not in v]
    chaves = {v: norm_key((bb_name, v, nome, regime, esc, "auto")) for v in variantes}
    feitos = {_chave_linha(r) for _, r in todos.iterrows()}
    pend = [(v, s) for s in base.seeds for v in variantes if chaves[v] + (s,) not in feitos]
    print(f"\n[pós] {bb_name}: melhor frozen pela validação = {regime} | {nome} "
          f"| escala {esc:g} ({SEL_NOME} val {best.f:.4f}) | {len(pend)} execução(ões) pendente(s)")
    if not pend:
        return 0
    cfg = dataclasses.replace(base, run_name=run_dir.name, resume=True)
    paths, labels, groups = prepare_data(cfg)
    X, X_aug = _embeddings_pos(bb_name, paths, cfg, esc)
    pred_dir = run_dir / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    records = todos.to_dict("records")
    for seed in sorted({s for _, s in pend}):
        sp = make_holdout_split(labels, groups, seed, cfg)
        f, params = _preditor_treino(regime, nome, X, labels, sp, cfg, seed, X_aug)
        yv, yt, gv = labels[sp["val"]], labels[sp["test"]], groups[sp["val"]]
        Pv, dv = f(X[sp["val"]])
        Pt, dt = f(X[sp["test"]])
        saidas = {"pos_base": (Pv, dv, Pt, dt, None)}
        b = ajustar_vies_f1(Pv, yv)
        saidas["pos_vies"] = (Pv, _vies_fora_da_dobra(Pv, yv, gv), Pt, _aplicar_vies(Pt, b), b)
        if tem_tta:
            Pv_t = np.mean([Pv] + [f(Xa[sp["val"]])[0] for Xa in X_aug], axis=0)
            Pt_t = np.mean([Pt] + [f(Xa[sp["test"]])[0] for Xa in X_aug], axis=0)
            saidas["pos_tta"] = (Pv_t, Pv_t.argmax(1), Pt_t, Pt_t.argmax(1), None)
            b2 = ajustar_vies_f1(Pv_t, yv)
            saidas["pos_tta_vies"] = (Pv_t, _vies_fora_da_dobra(Pv_t, yv, gv), Pt_t, _aplicar_vies(Pt_t, b2), b2)
        for v, (pv, dv_, pt, dt_, vies) in saidas.items():
            key = chaves[v]
            if key + (seed,) in feitos:
                continue
            vm, tm = compute_metrics(yv, pv, dv_), compute_metrics(yt, pt, dt_)
            tag = key_tag(key)
            pd.DataFrame({"idx": sp["test"], "path": [paths[i] for i in sp["test"]], "group": groups[sp["test"]],
                          "y_true": yt, "y_pred": dt_, **{f"p_{c}": pt[:, m] for m, c in enumerate(CLASS_NAMES)}}
                         ).to_csv(pred_dir / f"{tag}_seed{seed}.csv", index=False)
            pd.DataFrame({"idx": sp["val"], "y_true": yv, "y_pred": dv_,
                          **{f"p_{c}": pv[:, m] for m, c in enumerate(CLASS_NAMES)}}
                         ).to_csv(pred_dir / f"{tag}_seed{seed}_val.csv", index=False)
            extra = {"batch_role": "pos_processamento", "hiperparametro": json.dumps(params),
                     "vieses": json.dumps(None if vies is None else np.round(vies, 3).tolist())}
            records.append(_record(key, seed, extra, vm, tm))
            feitos.add(key + (seed,))
            pd.DataFrame(records).to_csv(csv, index=False)
            print(f"  >> {v:13s} semente {seed}: teste F1={tm['f1_macro']:.4f} QWK={tm['qwk']:.4f} | "
                  f"val F1={vm['f1_macro']:.4f}" + ("" if vies is None else f" | vieses={np.round(vies, 2).tolist()}"))
    return len(pend)


# =======================================================================================
# ENSEMBLE ENTRE MODELOS (seleção gulosa pela validação) E COMPARAÇÃO PAREADA
# =======================================================================================
def _probs_salvas(bb: str, key: tuple, seed: int, base: Config, parte: str) -> Optional[pd.DataFrame]:
    sufixo = "_val" if parte == "val" else ""
    f = _pasta_modelo(bb, base) / "predictions" / f"{key_tag(key)}_seed{seed}{sufixo}.csv"
    return pd.read_csv(f).sort_values("idx").reset_index(drop=True) if f.exists() else None


def ensemble_entre_modelos(modelos: List[str], base: Config, max_membros: int = 5) -> Optional[pd.DataFrame]:
    """Média das probabilidades de vários modelos (seleção gulosa com reposição, Caruana et al.,
    2004): começa pelo melhor modelo na validação e só acrescenta um membro se o F1 MÉDIO DE
    VALIDAÇÃO subir. Fontes: pos_tta ou pos_base de cada modelo (as que não usam a validação
    para ajustar nada). O teste é lido uma única vez, no fim."""
    fontes = {}
    for bb in modelos:
        csv = _pasta_modelo(bb, base) / "tabelas" / "execucoes_por_semente.csv"
        if not csv.exists():
            continue
        df = pd.read_csv(csv)
        df = df[(df.backbone == bb) & df.regime.isin(["pos_tta", "pos_base"])]
        st = df.groupby(KEY_COLS).agg(n=("seed", "nunique"), f=(f"val_{SELECTION_METRIC}", "mean")).reset_index()
        st = st[st.n >= len(base.seeds)]
        if st.empty:
            continue
        r = st.sort_values("f", ascending=False).iloc[0]
        key = norm_key(tuple(r[c] for c in KEY_COLS))
        por_seed = {}
        for s in base.seeds:
            v, t = _probs_salvas(bb, key, s, base, "val"), _probs_salvas(bb, key, s, base, "test")
            if v is None or t is None:
                break
            por_seed[s] = (v, t)
        if len(por_seed) == len(base.seeds):
            fontes[bb] = por_seed
    if len(fontes) < 2:
        print("[ensemble] menos de 2 modelos com pós-processamento; ensemble pulado")
        return None
    cols = [f"p_{c}" for c in CLASS_NAMES]

    def avaliar(membros, parte):
        out = []
        for s in base.seeds:
            i = 0 if parte == "val" else 1
            P = np.mean([fontes[m][s][i][cols].to_numpy() for m in membros], axis=0)
            y = fontes[membros[0]][s][i].y_true.to_numpy()
            out.append((y, P, P.argmax(1), fontes[membros[0]][s][i]))
        return out

    def escore(membros):
        ms = [compute_metrics(y, P, d) for y, P, d, _ in avaliar(membros, "val")]
        return (float(np.mean([m[SELECTION_METRIC] for m in ms])), float(np.mean([m["qwk"] for m in ms])))

    membros, melhor = [], None
    for _ in range(max_membros):
        cand = [(escore(membros + [m]), m) for m in fontes]
        sc_, m = max(cand)
        if melhor is not None and sc_ <= melhor:
            break
        membros.append(m)
        melhor = sc_
    print(f"[ensemble] membros escolhidos pela validação: {membros} ({SEL_NOME} val {melhor[0]:.4f})")
    dst = _pasta_modelo("ensemble", base)
    (dst / "predictions").mkdir(parents=True, exist_ok=True)
    (dst / "tabelas").mkdir(parents=True, exist_ok=True)
    key = norm_key(("ensemble", "ensemble_guloso", "+".join(membros), "media_prob", 1.0, "auto"))
    records = []
    val_aval, test_aval = avaliar(membros, "val"), avaliar(membros, "test")
    for s, (yv, Pv, dv, _), (yt, Pt, dt, tdf) in zip(base.seeds, val_aval, test_aval):
        vm, tm = compute_metrics(yv, Pv, dv), compute_metrics(yt, Pt, dt)
        pd.DataFrame({"idx": tdf.idx, "path": tdf.get("path"), "group": tdf.get("group"), "y_true": yt,
                      "y_pred": dt, **{c: Pt[:, m] for m, c in enumerate(cols)}}
                     ).to_csv(dst / "predictions" / f"{key_tag(key)}_seed{s}.csv", index=False)
        records.append(_record(key, s, {"batch_role": "ensemble", "membros": "+".join(membros)}, vm, tm))
        print(f"  >> ensemble semente {s}: teste F1={tm['f1_macro']:.4f} QWK={tm['qwk']:.4f}")
    out = pd.DataFrame(records)
    out.to_csv(dst / "tabelas" / "execucoes_por_semente.csv", index=False)   # derivado: recalculado sempre
    return out


def comparacao_pareada(resumo: Optional[Dict], base: Config, n_boot: int = 2000) -> Optional[pd.DataFrame]:
    """Bootstrap PAREADO por imagem de origem (mesmos splits e mesmas ROIs para todos os
    modelos): diferença de F1 macro de teste entre o modelo escolhido pela VALIDAÇÃO (referência)
    e cada um dos outros. Com teste pequeno, intervalos marginais se sobrepõem facilmente; o
    pareamento remove a variação comum às duas predições."""
    if not resumo or resumo.get("todos") is None or len(resumo["todos"]) < 2:
        return None
    t = resumo["todos"].reset_index(drop=True)

    def carregar(row):
        esc = row.get(PT_NAMES["scale"], 1.0)
        key = norm_key((row["Modelo"], row[PT_NAMES["regime"]], row[PT_NAMES["loss"]],
                        row[PT_NAMES["lr_variant"]], 1.0 if pd.isna(esc) else float(esc),
                        row[PT_NAMES["batch_cfg"]]))
        partes = []
        for s in base.seeds:
            d = _probs_salvas(row["Modelo"], key, s, base, "test")
            if d is None:
                return None
            partes.append(d.assign(seed=s)[["seed", "idx", "group", "y_true", "y_pred"]])
        return pd.concat(partes, ignore_index=True)

    carregados = []
    for _, row in t.iterrows():          # t já está em ordem decrescente de F1 de VALIDAÇÃO
        P_ = carregar(row)
        if P_ is None:
            print(f"[pareada] {row['Modelo']}: predições de teste ausentes; fora da comparação")
        else:
            carregados.append((row, P_))
    if len(carregados) < 2:
        return None
    (ref, R) = carregados[0]
    rng = np.random.default_rng(0)
    linhas = []
    for row, O in carregados[1:]:
        M = R.merge(O, on=["seed", "idx"], suffixes=("_r", "_o"))
        unidades = M.seed.astype(str) + "|" + M.group_r.astype(str)
        uniq, inv = np.unique(unidades, return_inverse=True)
        pos = [np.where(inv == u)[0] for u in range(len(uniq))]
        y, dr, do = M.y_true_r.to_numpy(), M.y_pred_r.to_numpy(), M.y_pred_o.to_numpy()
        obs = _f1_rapido(y, dr) - _f1_rapido(y, do)
        difs = np.empty(n_boot)
        for b in range(n_boot):
            sel = np.concatenate([pos[u] for u in rng.integers(0, len(uniq), len(uniq))])
            difs[b] = _f1_rapido(y[sel], dr[sel]) - _f1_rapido(y[sel], do[sel])
        lo, hi = np.percentile(difs, [2.5, 97.5])
        linhas.append({"Referência (melhor na validação)": ref["Modelo"], "Comparado": row["Modelo"],
                       "ΔF1 teste (ref − comparado)": round(obs, 4), "IC 95%": f"[{lo:.4f}, {hi:.4f}]",
                       "P(Δ ≤ 0) bootstrap": round(float(np.mean(difs <= 0)), 3),
                       "Diferença clara?": "sim" if lo > 0 else "não"})
    if not linhas:
        return None
    out = pd.DataFrame(linhas)
    save_table(out, Path(base.drive_results_dir) / "comparacao_pareada_f1.csv",
               "Bootstrap pareado por imagem: F1 do melhor modelo (pela validação) x demais")
    return out


# =======================================================================================
# TESTE RÁPIDO DE TODOS OS MODELOS
# Passa o mínimo possível por cada um (2 ROIs reais, 1 passo) para revelar TODOS os erros de
# uma vez, antes de gastar horas na bateria. Não grava embeddings, resultados nem pastas de
# execução: só o resumo teste_rapido_modelos.csv. Etapas por modelo (cada uma independente):
#   pesos    : arquivo local existe / acesso ao repositório no Hugging Face
#   carregar : build_backbone (já inclui um forward sintético) + dimensão do embedding
#   forward  : 2 ROIs reais de classes diferentes pelo mesmo caminho da bateria (multi-crop,
#              normalização, autocast): shape, sem NaN, embeddings distintos entre ROIs
#   cache    : metadados que formam o hash do cache de embeddings
#   lora     : injeção do LoRA + 1 passo de gradiente (todos os adaptadores recebem gradiente)
#   bitfit   : 1 passo de gradiente só nos vieses (pega o bug da LayerNorm)
# LoRA e BitFit só travam o PEFT do modelo; o frozen continua valendo se carregar e forward passarem.
# =======================================================================================
import traceback  # noqa: E402  (usado por _etapa)

DIM_ESPERADA = {"titan": 768, "gigapath": 1536, "prism": 1280, "chief": 768, "ctranspath": 768,
                "lunit": 384, "plip": 512, "virchow": 2560, "uni": 1024, "conch": 512, "beph": 768,
                "pathoduet": 768, "path_foundation": 384, "hibou_b": 768, "athena": 1536,
                "imagenet_r50": 2048}   # só gera AVISO se diferir (confira se é o esperado)
TRANSFORMERS_PEDIDO = {"titan": "4.46.0", "prism": "4.42.4"}   # versões dos model cards

DICAS_ERRO = [
    (r"peso não encontrado|FileNotFoundError", "Baixe o peso e salve em PESOS_DIR com o nome de LOCAL_WEIGHTS (a célula 10 lista o que falta)."),
    (r"huggingface-hub.*required|is_valid_xet_hash|ambiente foi atualizado",
     "Conflito transformers x huggingface-hub: Ambiente de execução > Reiniciar sessão e rode de novo desde a célula 1."),
    (r"401|403|gated|GatedRepo|RepositoryNotFound|Repository Not Found|authenticated|indisponível",
     "Sem acesso ao repositório no Hugging Face: peça acesso na página do modelo, com a mesma conta do token."),
    (r"NativeLayerNormBackward", "Bug de gradiente da LayerNorm no BitFit: me envie o traceback."),
    (r"out of memory|OutOfMemory", "Falta de VRAM: reinicie a sessão (libera memória) ou reduza o lote em BATCH_SIZE_T4."),
    (r"No module named|ModuleNotFoundError", "Pacote ausente: rode pip install do módulo citado e reinicie a sessão."),
    (r"RAMInsuficiente|RAM livre", "Falta de RAM: reinicie a sessão e rode só esse modelo (o que já foi feito é reaproveitado)."),
    (r"pesos foram carregados|só .* dos pesos|não carregados", "O checkpoint não bateu com a arquitetura: me envie as chaves listadas no log."),
    (r"Connection|timed out|Max retries|HTTPError|Temporary failure", "Falha de rede ou de download: tente de novo."),
    (r"tensorflow|libcudnn|cuDNN|CUDA_ERROR", "Problema do TensorFlow com a GPU (só path_foundation): me envie o traceback."),
    (r"trust_remote_code|remote code|auto_map", "Código remoto do modelo incompatível com a versão do transformers: confira TRANSFORMERS_VERSAO."),
    (r"slide_representations|image_embedding", "API do PRISM diferente do esperado: me envie o traceback."),
]


def _dica(msg: str) -> str:
    for padrao, texto in DICAS_ERRO:
        if re.search(padrao, msg, flags=re.I):
            return texto
    return ""


def _limpar_grads(bb) -> None:
    for p in bb.parameters():
        p.grad = None


def _roi_amostra(cfg: Config):
    """2 ROIs REAIS de classes diferentes (subconjunto mínimo do dataset). Se o dataset não
    estiver acessível, usa 2 imagens sintéticas 450x250 e avisa."""
    try:
        paths, labels, _ = prepare_data(dataclasses.replace(cfg, subset_groups_per_class=1))
        idx = [int(np.where(labels == c)[0][0]) for c in np.unique(labels)[:2]]
        return [paths[i] for i in idx], labels[idx], True
    except Exception as e:   # noqa: BLE001
        print(f"[teste] dataset indisponível ({type(e).__name__}: {e}); usando imagens sintéticas")
        pasta = Path("/content/_teste_rapido")
        pasta.mkdir(exist_ok=True)
        rng = np.random.default_rng(0)
        ps = []
        for i in range(2):
            p = pasta / f"sint{i}.png"
            Image.fromarray(rng.integers(60, 230, (250, 450, 3), dtype=np.uint8)).save(p)
            ps.append(str(p))
        return ps, np.array([0, 1]), False


def _lote(bb, paths, labels):
    ds = ROIDataset(paths, labels, bb.view_size, bb.normalize, 1.0, None)
    x, counts, _, _ = collate_views([ds[i] for i in range(len(paths))])
    return x.to(DEVICE), counts


def _etapa(r: Dict, nome: str, fn):
    """Roda uma etapa isolada: registra ok/FALHOU, tempo, erro curto e traceback."""
    t0 = time.time()
    try:
        out = fn()
        r[nome] = "ok"
        return out
    except Exception as e:   # noqa: BLE001 — é exatamente o que queremos capturar
        r[nome] = "FALHOU"
        r.setdefault("erros", []).append((nome, f"{type(e).__name__}: {e}"))
        r.setdefault("tracebacks", {})[nome] = traceback.format_exc()
        return None
    finally:
        r.setdefault("tempos", {})[nome] = round(time.time() - t0, 1)


def testar_modelo(nome: str, cfg: Config, paths: List[str], labels: np.ndarray, com_peft: bool = True) -> Dict:
    r: Dict = {"modelo": nome}
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    def pesos():
        if nome in ("chief", "beph", "pathoduet"):
            f = local_weight_path(nome)
            if not f.exists():
                raise FileNotFoundError(f"peso não encontrado: {f}")
        sha = ENV_INFO["hf_sha"].get(nome)
        if nome in HF_REPOS and str(sha).startswith("indisponível"):
            raise PermissionError(f"sem acesso ao repositório {HF_REPOS[nome]}: {sha}")
    _etapa(r, "pesos", pesos)

    if nome == "fusecpath":   # é a concatenação de componentes: o resultado vem dos testes deles
        return r

    # ---- baseline de cor: sem rede neural
    if nome == "color":
        def cor():
            X = color_features(paths)
            assert X.shape[0] == len(paths) and np.isfinite(X).all(), f"features inválidas {X.shape}"
            r["dim"] = int(X.shape[1])
        _etapa(r, "carregar", cor)
        r["forward"] = r["cache"] = r["lora"] = r["bitfit"] = "n/a"
        return r

    bb = _etapa(r, "carregar", lambda: build_backbone(nome, cfg))
    if bb is None:
        r["forward"] = r["cache"] = r["lora"] = r["bitfit"] = "pulado"
        return r
    r["dim"], r["janela"] = int(bb.embed_dim), int(bb.view_size)
    if nome in DIM_ESPERADA and r["dim"] != DIM_ESPERADA[nome]:
        r.setdefault("avisos", []).append(f"dimensão {r['dim']} (esperado {DIM_ESPERADA[nome]}): confira")

    x_counts = {}

    def forward():
        x, counts = _lote(bb, paths, labels)
        with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
            z = bb.encode_views(x, counts)
        z = z.float().cpu().numpy()
        assert z.shape == (len(paths), bb.embed_dim), f"shape {z.shape}, esperado {(len(paths), bb.embed_dim)}"
        assert np.isfinite(z).all(), "embedding com NaN/inf"
        assert float(np.abs(z[0] - z[1]).max()) > 1e-6, "embeddings idênticos para ROIs diferentes (encoder colapsado?)"
        x_counts["x"], x_counts["counts"] = x, counts
        r["janelas_por_roi"] = int(counts[0])
    _etapa(r, "forward", forward)

    def cache():
        meta = embedding_meta(bb, paths, 1.0, None)
        json.dumps(meta)
        _hash(meta)
    _etapa(r, "cache", cache)

    if not com_peft or nome in FROZEN_ONLY or "x" not in x_counts:
        r["lora"] = r["bitfit"] = "n/a" if (nome in FROZEN_ONLY or not com_peft) else "pulado"
    else:
        x, counts = x_counts["x"], x_counts["counts"]

        def passo(modo):
            bb.reset_and_configure(modo)
            r[f"treinaveis_{modo}"] = int(sum(p.numel() for p in bb.parameters() if p.requires_grad))
            assert r[f"treinaveis_{modo}"] > 0, "nenhum parâmetro treinável"
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                z = bb.encode_views(x, counts)
            z.float().pow(2).mean().backward()
            if modo == "lora":
                vivos = sum(1 for m in bb._lora if m.lora_B.grad is not None and float(m.lora_B.grad.abs().sum()) > 0)
                assert vivos == len(bb._lora), f"só {vivos}/{len(bb._lora)} adaptadores receberam gradiente"
            else:
                vivos = sum(1 for p in bb._bitfit if p.grad is not None)
                assert vivos > 0, "nenhum viés recebeu gradiente"
        for modo in ("bitfit", "lora"):
            _etapa(r, modo, lambda m=modo: passo(m))
            _limpar_grads(bb)
            try:
                bb.reset_and_configure("frozen")
            except Exception:   # noqa: BLE001
                pass
    if torch.cuda.is_available():
        r["vram_pico_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
    del bb
    liberar_memoria(f"teste {nome}")
    return r


def checagem_global(cfg: Config) -> None:
    """Ambiente, GPU, token, pastas: o que não depende de nenhum modelo."""
    print("\n" + "=" * 90 + "\n= CHECAGEM GLOBAL\n" + "=" * 90)
    print(f"GPU: {ENV_INFO['gpu']} ({ENV_INFO['vram_gb']} GB) | AMP: {AMP_DTYPE} | python {ENV_INFO['python']}")
    v = {d: _ver(d) for d in ("torch", "timm", "transformers", "huggingface-hub", "tokenizers")}
    print("versões:", v)
    hub, tf = v["huggingface-hub"], v["transformers"]
    if tf and hub and tf.startswith("4.") and int(hub.split(".")[0]) >= 1:
        print("  [PROBLEMA] transformers 4.x exige huggingface-hub < 1.0 -> reinicie a sessão (célula 2 corrige o pin)")
    try:
        _checar_modulos_desatualizados()
    except RuntimeError as e:
        print(f"  [PROBLEMA] {e}")
    try:
        print(f"token HF: ok (usuário {HfApi().whoami(token=HF_TOKEN).get('name')})")
    except Exception as e:   # noqa: BLE001
        print(f"  [PROBLEMA] token HF inválido ou ausente: {type(e).__name__}")
    sem_acesso = [k for k, s in ENV_INFO["hf_sha"].items() if str(s).startswith("indisponível")]
    print("repositórios HF sem acesso:", sem_acesso or "nenhum")
    try:
        t = Path(cfg.drive_results_dir) / ".teste_escrita"
        t.write_text("ok")
        t.unlink()
        print(f"Drive gravável: {cfg.drive_results_dir}")
    except Exception as e:   # noqa: BLE001
        print(f"  [PROBLEMA] não consigo gravar em {cfg.drive_results_dir}: {e}")
    verificar_pesos_locais(["chief", "beph", "pathoduet"])


def teste_rapido_modelos(modelos: List[str], cfg: Config, com_peft: bool = True) -> "pd.DataFrame":
    checagem_global(cfg)
    ordem = list(dict.fromkeys(modelos))
    if "fusecpath" in ordem:   # os componentes precisam estar testados antes
        ordem = [c for c in FUSECPATH_COMPONENTS if c not in ordem] + ordem
    ordem = [m for m in ordem if m != "path_foundation"] + (["path_foundation"] if "path_foundation" in ordem else [])
    paths, labels, reais = _roi_amostra(cfg)
    print(f"\n[teste] amostra: 2 ROIs {'reais' if reais else 'SINTÉTICAS'} de classes {labels.tolist()}")
    resultados, t_total = {}, time.time()
    for i, nome in enumerate(ordem, 1):
        print("\n" + "=" * 90 + f"\n= [{i}/{len(ordem)}] TESTE: {nome}\n" + "=" * 90)
        t0 = time.time()
        r = testar_modelo(nome, cfg, paths, labels, com_peft)
        r["t_total_s"] = round(time.time() - t0, 1)
        pedido = TRANSFORMERS_PEDIDO.get(nome)
        if pedido and _ver("transformers") != pedido:
            r.setdefault("avisos", []).append(f"transformers {_ver('transformers')} (o card pede {pedido})")
        resultados[nome] = r
        print(f"[{nome}] " + " | ".join(f"{k}={r.get(k)}" for k in ("pesos", "carregar", "forward", "cache", "lora", "bitfit")))
    if "fusecpath" in resultados:
        comp = [resultados[c] for c in FUSECPATH_COMPONENTS if c in resultados]
        ruim = [c["modelo"] for c in comp if c.get("carregar") != "ok" or c.get("forward") != "ok"]
        f = resultados["fusecpath"]
        f.update(pesos="ok", carregar="ok" if not ruim else "FALHOU", forward="ok" if not ruim else "FALHOU",
                 cache="n/a", lora="n/a", bitfit="n/a",
                 dim=sum(int(c.get("dim", 0)) for c in comp))
        if ruim:
            f.setdefault("erros", []).append(("componentes", f"depende de {ruim}"))
    linhas = []
    for nome in [m for m in ordem if m in modelos or m == "fusecpath"]:
        r = resultados[nome]
        base_ok = all(r.get(k) in ("ok", "n/a") for k in ("carregar", "forward", "cache")) and r.get("pesos") == "ok"
        peft_ok = all(r.get(k) in ("ok", "n/a") for k in ("lora", "bitfit"))
        status = "OK" if base_ok and peft_ok else ("OK p/ frozen (PEFT com erro)" if base_ok else "FALHOU")
        erros = "; ".join(f"{e}: {m}" for e, m in r.get("erros", []))
        linhas.append({"modelo": nome, "status": status, "pesos": r.get("pesos"), "carregar": r.get("carregar"),
                       "forward": r.get("forward"), "cache": r.get("cache"), "lora": r.get("lora"),
                       "bitfit": r.get("bitfit"), "dim": r.get("dim"), "janela": r.get("janela"),
                       "janelas/ROI": r.get("janelas_por_roi"), "vram_gb": r.get("vram_pico_gb"),
                       "t_carregar_s": r.get("tempos", {}).get("carregar"), "t_total_s": r.get("t_total_s"),
                       "avisos": "; ".join(r.get("avisos", [])), "erro": erros[:400],
                       "dica": _dica(erros)})
    df = pd.DataFrame(linhas)
    out = Path(cfg.drive_results_dir) / "teste_rapido_modelos.csv"
    df.to_csv(out, index=False)
    display(df)
    print(f"\n[teste] resumo salvo em {out} | tempo total {(time.time() - t_total) / 60:.1f} min")
    for nome in [m for m in ordem if resultados[m].get("tracebacks")]:
        for etapa, tb in resultados[nome]["tracebacks"].items():
            print("\n" + "-" * 90 + f"\n[{nome}] etapa '{etapa}' — final do traceback:\n" + "\n".join(tb.strip().splitlines()[-14:]))
    ok = df[df.status == "OK"].modelo.tolist()
    parcial = df[df.status.str.startswith("OK p/")].modelo.tolist()
    falha = df[df.status == "FALHOU"]
    print("\n" + "=" * 90 + "\n= RESUMO\n" + "=" * 90)
    print(f"Prontos para a bateria completa: {ok}")
    if parcial:
        print(f"Frozen ok, mas LoRA/BitFit com erro (não entram no PEFT): {parcial}")
    for _, l in falha.iterrows():
        print(f"  ✗ {l.modelo}: {l.erro[:160]}" + (f"\n      → {l.dica}" if l.dica else ""))
    for _, l in df[df.avisos != ""].iterrows():
        print(f"  ! {l.modelo}: {l.avisos}")
    return df


# ------------------------------- EXECUÇÃO -------------------------------

# TODOS os modelos. Ordem: mais leves primeiro; fusecpath por último (reaproveita os embeddings
# de UNI, CONCH, Virchow e GigaPath). Os baselines (ResNet-50 ImageNet e cor) passam pelas
# mesmas sondas e classificadores, para comparação justa.
MODELOS_PARA_TESTAR = [
    "color", "imagenet_r50",
    "ctranspath", "chief", "lunit", "plip", "hibou_b", "beph", "pathoduet",
    "conch", "uni", "titan", "virchow", "gigapath", "athena", "fusecpath",
    "path_foundation",   # TensorFlow por último: depois de importado, o TF não devolve a RAM
    # "prism",  # exige transformers==4.42.4 (conflita com o TITAN): rode em sessão separada,
    #           # com TRANSFORMERS_VERSAO = "4.42.4" (CÉLULA 2) e só "prism" nesta lista.
]
assert list(CFG.seeds) == [0, 1], f"esperadas 2 sementes, encontrei {CFG.seeds}"

FASES_FROZEN = ["0_smoke", "1_frozen_lbfgs", "2_classificadores", "4_magnificacao"]   # magnificação ATIVADA

# Magnificação: a objetiva da captura não está nos arquivos. Se souber, informe (ex.: 20 ou 40)
# e a fase 4 testa também a escala que iguala o µm/pixel do pré-treino de cada modelo.
MAGNIFICACAO_CAPTURA = None
MAG_ZOOMS = [0.5, 0.75, 1.0, 1.5, 2.0]   # zoom sobre o pixel da captura (1,0 = sem reamostrar)
PEFT_NA_MELHOR_MAGNIFICACAO = True       # LoRA/BitFit na escala com maior F1 de validação na fase 4
CFG = dataclasses.replace(CFG, magnificacao_captura=MAGNIFICACAO_CAPTURA, mag_zooms=list(MAG_ZOOMS))
GRUPOS_PEFT = [["3a_smoke_lora", "3_lora"], ["3a_smoke_bitfit", "3_bitfit"]]   # independentes entre si

RODAR_POS = True        # TTA + limiar de decisão para F1 na melhor config frozen de cada modelo
RODAR_PEFT = True
TOP_K_PEFT = 4          # quantos modelos (melhores no frozen pela validação) recebem LoRA/BitFit
MODELOS_PEFT = None     # ou uma lista fixa, ex.: ["uni", "virchow"]; None = escolha automática
RODAR_ENSEMBLE = True   # média de probabilidades entre modelos, membros escolhidos pela validação

RODAR_SANIDADE = True

# ---- MODO DE TESTE: passa o mínimo por cada modelo e mostra todos os erros de uma vez.
# Rode primeiro com True, corrija o que aparecer e só então mude para False (bateria completa).
SO_TESTE_RAPIDO = False
MODELOS_TESTE = None    # None = todos de MODELOS_PARA_TESTAR (+ prism); ou uma lista, ex.: ["chief", "plip"]
TESTE_COM_PEFT = True   # inclui o passo de LoRA e de BitFit (só nos modelos que suportam)

if __name__ == "__main__":
    import traceback
    if RODAR_SANIDADE:
        run_sanity_checks(CFG)
    if SO_TESTE_RAPIDO:
        _lista = MODELOS_TESTE or (MODELOS_PARA_TESTAR + [m for m in ("prism",) if m not in MODELOS_PARA_TESTAR])
        TESTE = teste_rapido_modelos(_lista, CFG, com_peft=TESTE_COM_PEFT)
        print("\n>>> Teste rápido concluído. Corrija os itens acima, rode de novo só os que falharam "
              "(MODELOS_TESTE = [...]) e, com tudo OK, mude SO_TESTE_RAPIDO = False para a bateria completa.")
    else:
        verificar_pesos_locais(MODELOS_PARA_TESTAR)
        # 1) frozen (o que já existe em qualquer pasta de resultados é importado, não refeito)
        STATUS = run_all_foundations(MODELOS_PARA_TESTAR, FASES_FROZEN, CFG)
        COMPARACAO = compare_foundations(MODELOS_PARA_TESTAR, CFG)
        # 2) pós-processamento (TTA + limiar para F1) na melhor config frozen de cada modelo
        if RODAR_POS:
            for bb in MODELOS_PARA_TESTAR:
                try:
                    pos_processamento(bb, CFG)
                except Exception:   # noqa: BLE001
                    traceback.print_exc()
        # 3) LoRA / BitFit nos melhores pela validação
        if RODAR_PEFT:
            alvo = MODELOS_PEFT or selecionar_para_peft(COMPARACAO, TOP_K_PEFT)
            if alvo:
                STATUS_PEFT = run_all_foundations(alvo, GRUPOS_PEFT, CFG, status_name="status_peft.csv")
        # 4) ensemble entre modelos
        if RODAR_ENSEMBLE:
            ensemble_entre_modelos(MODELOS_PARA_TESTAR, CFG)
        # 5) comparação final (seleção pela validação) e teste pareado
        COMPARACAO = compare_foundations(MODELOS_PARA_TESTAR + ["ensemble"], CFG)
        PAREADA = comparacao_pareada(COMPARACAO, CFG)

      [LP-FT] cabeça inicial = sonda L-BFGS (C=3.16228, F1 val 0.8218)

= AUDITORIA DO DATASET (datas repetidas / dois dias de coleta)
tamanhos das ROIs: {(450, 250): 456}
arquivos idênticos: nenhum
imagens de origem por classe: {'healthy': 32, 'mild': 21, 'moderate': 27, 'severe': 28} | ROIs por classe: {'healthy': 114, 'mild': 114, 'moderate': 114, 'severe': 114}
números repetidos ENTRE classes: 27 (ex.: 005:healthy/mild, 006:healthy/mild, 011:healthy/mild, 017:healthy/moderate, 018:healthy/mild, 026:healthy/mild) -> grupos separados (classe faz parte do grupo)
mesmo número DENTRO da classe (com e sem sufixo): ['mild/045', 'moderate/040'] -> grupos separados
grupos com sufixo '-N': 13 ({'healthy': 9, 'mild': 1, 'moderate': 3})
dia de coleta: DESCONHECIDO para todas as imagens (sem subpastas e sem dias_coleta.csv).
  O agrupamento continua correto (classe + nome); para medir o efeito do dia, preencha
  a coluna 'dia' de /content/drive/MyDrive/resultados_histopatologia/dias_coleta_mode

,semente,train_rois,train_%,train_imagens,train_por_classe,val_rois,val_%,val_imagens,val_por_classe,test_rois,test_%,test_imagens,test_por_classe
0,0,316,69.3,75,"[75, 79, 86, 76]",90,19.7,22,"[27, 20, 17, 26]",50,11.0,11,"[12, 15, 11, 12]"
1,1,308,67.5,75,"[76, 85, 74, 73]",96,21.1,22,"[30, 16, 22, 28]",52,11.4,11,"[8, 13, 18, 13]"


[batch] GPU Tesla T4 (14.56 GB) -> fator 1x


,backbone,extract,partial_batch_size,partial_effective_target,partial_accum_steps
0,titan,16,4,32,8
1,gigapath,8,4,32,8
2,chief,128,32,32,1
3,ctranspath,128,32,32,1
4,lunit,64,16,32,2
5,plip,128,32,32,1
6,virchow,16,8,32,4
7,uni,64,32,32,1
8,conch,32,8,32,4
9,beph,64,32,32,1


[sanidade] todas as checagens passaram
[pesos locais] pasta: /content/drive/MyDrive/pesos_foundations
   OK      chief      -> CHIEF_CTransPath.pth
   OK      beph       -> BEPH_backbone.pth
   OK      pathoduet  -> checkpoint_HE.pth

= [1/17] MODELO: color | fases ['0_smoke', '1_frozen_lbfgs', '2_classificadores', '4_magnificacao']
[RAM] antes de color: RAM livre 10.8/12.7 GB (processo usa 1.0 GB)
[fase 0_smoke] pulada: color já tem resultados deste tipo salvos (pipeline já validado)

##########################################################################################
# FASE 1_frozen_lbfgs | color | run 'frozen_color'
##########################################################################################
[dataset] cópia local já completa (456 imagens) em /content/dataset
[dataset] 456 ROIs de 108 imagens de origem | por classe: {'healthy': 114, 'mild': 114, 'moderate': 114, 'severe': 114} | ROI 450x250

===== Split 70%/20%/10% por semente (por imagem de origem) =====  ->  /co

...

,Modelo,Escopo,Regime,Loss / classificador,LR,Escala,Config. batch,Sementes,Configs comparadas,F1 val,Critério interno,F1 teste,QWK teste,AUROC teste,Bal. acc teste,Zoom (x captura)
179,ensemble,todos,ensemble_guloso,lunit+plip,media_prob,1.0000,auto,2,1,0.7998,f1_macro,0.6162 ± 0.1420,0.7038 ± 0.0777,0.8538 ± 0.0539,0.6193 ± 0.1365,NaN
119,uni,todos,bitfit,corn,protocolo,0.5580,auto,2,46,0.7924,f1_macro,0.7792 ± 0.0366,0.8922 ± 0.0145,0.9136 ± 0.0578,0.7756 ± 0.0287,0.50
51,lunit,todos,frozen,ce,C_busca,1.6741,auto,2,44,0.7830,f1_macro,0.5819 ± 0.1398,0.6853 ± 0.0137,0.8419 ± 0.0536,0.6034 ± 0.1167,1.50
39,chief,todos,pos_tta_vies,corn,frozen_aug,1.0000,auto,2,32,0.7797,f1_macro,0.5648 ± 0.0818,0.6037 ± 0.0489,0.8621 ± 0.0820,0.5899 ± 0.0859,0.90
151,gigapath,todos,lora,corn,protocolo,0.5580,auto,2,32,0.7736,f1_macro,0.7198 ± 0.2052,0.8276 ± 0.0902,0.9228 ± 0.0576,0.7303 ± 0.1789,0.50
177,path_foundation,todos,frozen,corn,C_busca,1.0000,auto,2,30,0.7719,f1_macro,0.7211 ± 0.1026,0.8382 ± 0.0559,0.9117 ± 0.0543,0.7175 ± 0.0788,0.90
63,plip,todos,pos_vies,ce,frozen,1.1161,auto,2,44,0.7602,f1_macro,0.6320 ± 0.0961,0.7386 ± 0.0135,0.8484 ± 0.0468,0.6354 ± 0.0865,1.00
73,hibou_b,todos,frozen_clf,lda,hp_busca,0.8371,auto,2,30,0.7588,f1_macro,0.6850 ± 0.0521,0.8183 ± 0.1118,0.8987 ± 0.0336,0.7184 ± 0.0084,0.75
161,athena,todos,frozen,corn,C_busca,0.5580,auto,2,30,0.7562,f1_macro,0.6089 ± 0.0084,0.8026 ± 0.0488,0.8723 ± 0.0487,0.6216 ± 0.0134,0.50
139,virchow,todos,frozen,corn,C_busca,0.5580,auto,2,30,0.7441,f1_macro,0.6807 ± 0.0519,0.7722 ± 0.0011,0.8873 ± 0.0453,0.6836 ± 0.0304,0.50



===== Melhor configuração por regime (sonda L-BFGS, classificadores, LoRA, BitFit) =====  ->  /content/drive/MyDrive/resultados_histopatologia/comparacao_por_regime.csv


,Modelo,Escopo,Regime,Loss / classificador,LR,Escala,Config. batch,Sementes,Configs comparadas,F1 val,Critério interno,F1 teste,QWK teste,AUROC teste,Bal. acc teste,Zoom (x captura)
0,color,frozen,frozen,corn,C_busca,1.0000,auto,2,3,0.4679,qwk,0.5078 ± 0.0677,0.4966 ± 0.2876,0.7544 ± 0.0402,0.4995 ± 0.0639,NaN
1,color,frozen_clf,frozen_clf,knn,hp_busca,1.0000,auto,2,5,0.5710,qwk,0.4235 ± 0.0250,0.4043 ± 0.1107,0.6854 ± 0.0109,0.4514 ± 0.0034,NaN
2,color,pos_base,pos_base,knn,frozen_clf,1.0000,auto,2,1,0.5710,f1_macro,0.4888 ± 0.0282,0.4617 ± 0.1193,0.7298 ± 0.0174,0.5042 ± 0.0433,NaN
3,color,pos_vies,pos_vies,knn,frozen_clf,1.0000,auto,2,1,0.5236,f1_macro,0.3873 ± 0.0297,0.3391 ± 0.1787,0.7298 ± 0.0174,0.4103 ± 0.0111,NaN
6,imagenet_r50,frozen,frozen,corn,C_busca,1.1161,auto,2,8,0.6915,f1_macro,0.6938 ± 0.0480,0.8030 ± 0.0551,0.8788 ± 0.0056,0.6949 ± 0.0416,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,path_foundation,pos_base,pos_base,corn,frozen,1.0000,auto,2,1,0.7719,f1_macro,0.7563 ± 0.1045,0.8590 ± 0.0593,0.9356 ± 0.0569,0.7561 ± 0.0862,0.9
173,path_foundation,pos_tta,pos_tta,corn,frozen,1.0000,auto,2,1,0.7408,f1_macro,0.7754 ± 0.1570,0.8618 ± 0.1015,0.9244 ± 0.0624,0.7826 ± 0.1473,0.9
174,path_foundation,pos_tta_vies,pos_tta_vies,corn,frozen,1.0000,auto,2,1,0.7077,f1_macro,0.7369 ± 0.0775,0.8214 ± 0.0339,0.9244 ± 0.0624,0.7315 ± 0.0515,0.9
175,path_foundation,pos_vies,pos_vies,corn,frozen,1.0000,auto,2,1,0.7697,f1_macro,0.7400 ± 0.0442,0.8267 ± 0.0028,0.9356 ± 0.0569,0.7306 ± 0.0180,0.9



===== Bootstrap pareado por imagem: F1 do melhor modelo (pela validação) x demais =====  ->  /content/drive/MyDrive/resultados_histopatologia/comparacao_pareada_f1.csv


,Referência (melhor na validação),Comparado,ΔF1 teste (ref − comparado),IC 95%,P(Δ ≤ 0) bootstrap,Diferença clara?
0,ensemble,uni,-0.1572,"[-0.2601, -0.0554]",0.999,não
1,ensemble,lunit,0.0220,"[-0.0532, 0.1018]",0.270,não
2,ensemble,chief,0.0205,"[-0.0803, 0.1307]",0.319,não
3,ensemble,gigapath,-0.1119,"[-0.2226, -0.0131]",0.984,não
4,ensemble,path_foundation,-0.1023,"[-0.2231, 0.0223]",0.941,não
5,ensemble,plip,-0.0081,"[-0.0809, 0.0587]",0.585,não
6,ensemble,hibou_b,-0.1011,"[-0.2180, 0.0485]",0.912,não
7,ensemble,athena,-0.0192,"[-0.1588, 0.1198]",0.609,não
8,ensemble,virchow,-0.0744,"[-0.1948, 0.0663]",0.867,não
9,ensemble,ctranspath,-0.0588,"[-0.1782, 0.0892]",0.786,não
